# SCOPE-CTC V4 — Verified Prefix-Beam Acceleration + Matched Multi-Dataset Evaluation

**Research method:** Frozen SVTRv2-T + multi-crop/multi-scale posterior segment proposals, local CTC sequence-likelihood verification, and compatible reconstruction. **Deployment mechanism:** A matched original-backbone output is preserved when no crop evidence justifies heavy reconstruction. For one-native-crop inputs, first run the primary crop-view probe and detect a **relative transcript coverage gap**, not a short raw transcript alone. Reuse those same logits if SCOPE is activated; accelerate two mathematical CTC kernels with Numba (output parity checked).

**One run:** Full 4,789-image extended LTB, Common-6, Union14M-6 plus General deterministic 2,000, OST Weak/Heavy; same frozen official SVTRv2-T checkpoint and normalization. Every dataset: original vs proposed, accuracy, CER, Levenshtein S/D/I, paired rescue/regression, latency including probes, activation and audit. LTB: 26–35 / 36–55 / ≥56, A-Avg, E7 activated-row historical parity.

**Scientific boundary:** The coverage rule was chosen after seeing prior LTB/Common/Union14M/OST outcomes. This notebook is a **development-cohort confirmatory re-run**, *not an independent unseen test*. Historical retrospective estimates are **not GPU results**. New forward and runtime metrics must come from running this notebook. No per-image regression guarantee. Do not claim universal STR, independent generalization, or publication-level novelty from these reused benchmarks.

**Default:** `MAX_IMAGES=0` (entire LTB), `MAX_PER_OTHER_DATASET=0` (entire Common/Union/OST subsets); General is an explicitly labeled deterministic 2,000-image sample because full General is 400,000. Set `GENERAL_MODE='full'` for full General (very expensive). Checkpoints saved in separate Drive folder, optional switch below; no personal Drive data is read. Same pinned model/CTC score, no training or GT in inference.


**V4 update (quality parameters locked):** Replaces only the *implementation* of the E7 prefix beam search with a Numba kernel, guarded by random/edge tests and the first 128 actual calls automatically compared against the original Python implementation. Exact decoding thresholds, crop geometry, coverage activation, original SVTRv2-T fallback and evaluation labels stay unchanged. Historical E7 parity and paired metrics detect any change in image-level results. No claimed GPU speedup until this notebook is run.

**Current limitations:** Quality/latency remain dataset dependent; coverage gating is still a GT-blind prediction-length-disagreement heuristic, not a pixel-space text coverage estimator. All cited datasets have already been used for method development; no independent generalization or novelty proof is claimed. V4 is an inference efficiency patch, not a new OCR contribution.


In [ ]:
from pathlib import Path
import os,json
ENABLE_DRIVE_CHECKPOINTS = True
MULTIDATA_PERSIST_ROOT = Path('/content/SCOPE_V4_SPEED_JIT_RUN')
if ENABLE_DRIVE_CHECKPOINTS and Path('/content').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive',force_remount=False)
        MULTIDATA_PERSIST_ROOT = Path('/content/drive/MyDrive/SCOPE_V4_SPEED_JIT_RUN')
        print('Persistent checkpoint on Google Drive:', MULTIDATA_PERSIST_ROOT)
    except Exception as ex:
        print('Drive unavailable, continuing with /content checkpoint (session-local):',repr(ex))
MULTIDATA_PERSIST_ROOT.mkdir(parents=True,exist_ok=True)
print('Results checkpoint root:', MULTIDATA_PERSIST_ROOT)

In [ ]:
from pathlib import Path
import os,sys,hashlib,json,io,gzip,base64,subprocess,importlib.util
WORK = Path('/content' if Path('/content').exists() else '/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
OUT = MULTIDATA_PERSIST_ROOT/'LTB'
OUT.mkdir(parents=True,exist_ok=True)
REPO = WORK/'OpenOCR'
OPENOCR_REV = '1ccfc6ee6161f7133b192e16af3a9265273997ba'
CKPT_ID = '1xJ10gOQphZjEgawXsgsadkXaZa0BIDN7'
CFG_ID = '1a-NRSh9aoYpdtwNgctF7a5CbEHcTA71a'
HF_DATA = 'topdu/OpenOCR-Data'
CKPT_OVERRIDE = None
CFG_OVERRIDE = None
LTB_OVERRIDE = None
MAX_IMAGES = 0
IMAGE_BATCH = 2
STRICT_BASELINE_PARITY = True
SAVE_DEBUG_LOGITS = False
os.chdir(WORK)
CHECK_LEGACY_N = 48
WRITE_EVERY = 25
RUN_ORIGINAL_SVTR = True
SKIP_ONE_CROP_EXTRA = True
STRICT_BASELINE_PARITY = True
STRICT_E7_PARITY = True
HELDOUT_CSV = None
print('SCOPE V4 PREFIX-BEAM JIT matched LTB:',WORK,'Output:',OUT,'MAX_IMAGES:',MAX_IMAGES,'Original baseline:',RUN_ORIGINAL_SVTR)

In [ ]:
# 1. Setup: pinned OpenOCR and minimum requirements
import subprocess, importlib.util, re, numpy as np, torch

def call(args):
    subprocess.run(args, check=True)
if not (REPO/'.git').is_dir():
    call(['git','clone','-q','https://github.com/Topdu/OpenOCR.git',str(REPO)])
call(['git','-C',str(REPO),'checkout','--force',OPENOCR_REV])
packages = [('rapidfuzz','rapidfuzz'),('gdown','gdown'),('huggingface_hub','huggingface_hub'),
            ('pandas','pandas'),('PIL','pillow'),('cv2','opencv-python-headless'),('yaml','pyyaml'),
            ('imgaug','imgaug'),('lmdb','lmdb'),('tabulate','tabulate'),('numba','numba')]
need = [pkg for mod,pkg in packages if importlib.util.find_spec(mod) is None]
if need:call([sys.executable,'-m','pip','install','-q',*need])
# OpenOCR pinned legacy imgaug may reference removed NumPy 2 aliases. Keep same compatibility patch
# used in the previously successful GAWS notebook.
spec=importlib.util.find_spec('imgaug')
if spec and spec.origin:
    substitutions=[(r'np\.sctypes\["float"\]','[np.float16,np.float32,np.float64]'),
                   (r'np\.sctypes\["int"\]','[np.int8,np.int16,np.int32,np.int64]'),
                   (r'np\.sctypes\["uint"\]','[np.uint8,np.uint16,np.uint32,np.uint64]'),
                   (r'np\.complex(?![_0-9A-Za-z])','np.complex128'),
                   (r'np\.bool(?![_0-9A-Za-z])','np.bool_'),
                   (r'np\.float(?![_0-9A-Za-z])','np.float64'),
                   (r'np\.int(?![_0-9A-Za-z])','np.int64')]
    for f in Path(spec.origin).parent.rglob('*.py'):
        try:
            raw=f.read_text(); upd=raw
            for pat,rep in substitutions: upd=re.sub(pat,rep,upd)
            if raw!=upd: f.write_text(upd)
        except (PermissionError, UnicodeDecodeError): pass
p=REPO/'openrec/preprocess/cam_label_encode.py'
if p.is_file():
    old=p.read_text();new=old.replace('pygame.locals.SRCALPHA','pygame.SRCALPHA')
    if old!=new:p.write_text(new)
sys.path.insert(0,str(REPO))
try:
    from tools.engine.config import Config
    from openrec.postprocess import build_post_process
    from openrec.modeling import build_model
except Exception as exc:
    print('OpenOCR dependency fallback:',repr(exc))
    call([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')])
    # Newly installed imgaug might require patch; re-execute compatibility replacements above if needed.
    spec=importlib.util.find_spec('imgaug')
    if spec and spec.origin:
        for f in Path(spec.origin).parent.rglob('*.py'):
            try:
                raw=f.read_text(); upd=raw
                for pat,rep in substitutions:upd=re.sub(pat,rep,upd)
                if upd!=raw:f.write_text(upd)
            except (PermissionError,UnicodeDecodeError):pass
    from tools.engine.config import Config
    from openrec.postprocess import build_post_process
    from openrec.modeling import build_model
import gdown, pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image
assert torch.cuda.is_available(), 'Enable GPU (Kaggle or Colab); CPU inference on 232 tall LTB images is impractical.'
DEVICE='cuda'
print('OpenOCR pinned:',OPENOCR_REV,' GPU:',torch.cuda.get_device_name(0))

## Pinned original source modules (unchanged)

The next modules are copied byte-for-byte from the successful E5 inference environment, then the E7 decoder/adapter are added separately. Nothing is finetuned. Source hashes are reported at the end.

In [ ]:
%%writefile longtext_final.py
"""Long-text STR inference/benchmark: exact LTB Stitch v2 geometry and overlap algorithm.

Model: official SVTRv2-T OpenOCR CTC branch (strict checkpoint/config parity checks).
Methods: full12, full64, stitch_v2, geom_ctc. All share one checkpoint.
Original Stitch v2 crops the ORIGINAL PIL image, then resizes each crop with MSR.
No training, no GT-guided inference, no inference-parameter tuning on LTB.

NOTE: LTB protocol normalizes ASCII alphanumerics to lowercase and strips punctuation.
This is NOT a general-purpose punctuation-preserving OCR output pipeline.
"""
from __future__ import annotations

import math
import os
import re
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from rapidfuzz.distance import Levenshtein


# ======================= PINNED LEGACY PROCEDURE ==========================
# The two functions below are copied verbatim (algorithm) from user LTB.ipynb,
# cell G2. Do not silently 'improve' coefficients without separate validation.

def make_chunks(img, chunk_ar, overlap):
    w, h = img.size; cw = int(round(h * chunk_ar))
    if w <= cw * 1.2: return [(img, 0, w)]
    stride = max(1, int(cw * (1 - overlap))); xs = list(range(0, w - cw + 1, stride))
    if xs[-1] + cw < w: xs.append(w - cw)
    return [(img.crop((x, 0, x + cw, h)), x, x + cw) for x in xs]


def stitch_v2(parts, spans, tol=0.34):
    out, miss = parts[0], 0
    for j in range(1, len(parts)):
        b = parts[j]; wb = max(spans[j][1] - spans[j][0], 1); exp = max(0, spans[j - 1][1] - spans[j][0]) / wb * len(b)
        if exp < 0.5 or not b or not out: out += b; continue
        Lmax = min(len(out), len(b), int(round(exp * 1.6)) + 2); best = None
        for L in range(1, Lmax + 1):
            d = Levenshtein.distance(out[-L:], b[:L]); sc = L - 2 * d - 0.5 * abs(L - exp)
            if best is None or sc > best[0]: best = (sc, L, d)
        sc, L, d = best
        if sc <= 0 or d > tol * L: miss += 1; L = min(int(round(exp)), len(b)); out += b[L:]
        else: h = L // 2; out = out[:len(out) - L + h] + b[h:]
    return out, miss

# Legacy values selected among eight configurations in original LTB study.
# They are EMPIRICAL LOCKED CONFIG, not theoretic constants or a new grid search.
LEGACY_CHUNK_AR = 6.0
LEGACY_OVERLAP = 0.5


def normalize_ltb(s: str) -> str:
    return re.sub(r'[^0-9a-z]', '', str(s).lower())


def target_size_ltb(w: int, h: int, max_ratio: int = 12):
    """Original LTB.ipynb 'msr_round' target_size, copied geometry exactly."""
    if w <= 0 or h <= 0:
        raise ValueError('Invalid image width/height')
    base = [(64, 64), (96, 48), (112, 40), (128, 32)]
    r = min(max(1, round(w / h)), max_ratio)
    return base[r - 1] if r <= 4 else (32 * r, 32)


def preprocess_ltb(img: Image.Image, max_ratio: int = 12) -> torch.Tensor:
    """Original PIL bicubic -> RGB numpy -> [-1,1]."""
    w, h = img.size
    W, H = target_size_ltb(w, h, max_ratio)
    bicubic = Image.Resampling.BICUBIC if hasattr(Image, 'Resampling') else Image.BICUBIC
    a = np.asarray(img.convert('RGB').resize((W, H), bicubic), dtype=np.float32) / 255.
    return torch.from_numpy(((a - 0.5) / 0.5).transpose(2, 0, 1).copy())


# ======================= OFFICIAL SVTRv2-T ================================
# Identical tested RCTC extraction and strict load logic from GAWS notebook.

def rctc_logits(dec, x):
    B, C, H, W = x.shape
    x = dec.w_atten_block(x.permute(0, 2, 3, 1).reshape(-1, W, C)).reshape(B, H, W, C).permute(0, 3, 1, 2)
    x_kv = dec.fc_kv(x.flatten(2).transpose(1, 2)).reshape(B, H * W, 2, C).permute(2, 0, 3, 1)
    x_k, x_v = x_kv.unbind(0)
    tok = dec.char_token.tile([B, 1, 1])
    attn = (tok @ x_k).reshape([-1, 1, H, W])
    attn = F.softmax(attn, 2).permute(0, 3, 1, 2)
    x_v = x_v.reshape(B, C, H, W)
    feats = (attn @ x_v.permute(0, 3, 2, 1)).squeeze(2)
    return dec.fc(feats)


class CTCNet(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.transform = getattr(model, 'transform', None)
        self.encoder = model.encoder
        d = model.decoder
        self.rctc = d if hasattr(d, 'w_atten_block') else d.ctc_decoder

    def forward(self, x):
        if self.transform is not None:
            x = self.transform(x)
        return rctc_logits(self.rctc, self.encoder(x))


def load_official_svtr(repo, cfg_path, ckpt_path, device='cuda'):
    """Build official architecture, strict checkpoint check, verify extracted logits parity."""
    repo, cfg_path, ckpt_path = map(Path, (repo, cfg_path, ckpt_path))
    if not repo.is_dir() or not cfg_path.is_file() or not ckpt_path.is_file():
        raise FileNotFoundError((str(repo), str(cfg_path), str(ckpt_path)))
    sys.path.insert(0, str(repo))
    from tools.engine.config import Config
    from openrec.modeling import build_model
    from openrec.postprocess import build_post_process

    old_dir = Path.cwd()
    try:
        os.chdir(repo)  # official YAML uses repo-relative character dictionary paths
        cfg = Config(str(cfg_path)).cfg
        post = build_post_process(cfg['PostProcess'], cfg['Global'])
        channels = post.get_character_num()
        ccfg = {
            'name': 'CTCLabelDecode',
            'character_dict_path': cfg['PostProcess'].get('character_dict_path') or cfg['Global'].get('character_dict_path'),
            'use_space_char': cfg['PostProcess'].get('use_space_char', cfg['Global'].get('use_space_char', False)),
        }
        chars = list(build_post_process(ccfg, cfg['Global']).character)
        cfg['Architecture']['Decoder']['out_channels'] = channels
        model = build_model(cfg['Architecture'])
    finally:
        os.chdir(old_dir)

    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    if not isinstance(sd, dict):
        raise RuntimeError('Invalid checkpoint format')
    sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
    ref = model.state_dict()
    missing = set(ref) - set(sd)
    extra = set(sd) - set(ref)
    wrong = [k for k in set(ref) & set(sd) if ref[k].shape != sd[k].shape]
    if missing or extra or wrong:
        raise RuntimeError(f'Checkpoint/config mismatch: missing={list(missing)[:4]} extra={list(extra)[:4]} shape={wrong[:4]}')
    model.load_state_dict(sd, strict=True)
    if not chars or str(chars[0]).lower() not in {'blank', '[blank]', '<blank>', 'blank_token'}:
        raise RuntimeError(f'CTC character index 0 must be blank: {chars[:5]}')
    model = model.to(device).eval()
    net = CTCNet(model).to(device).eval()
    if net.rctc.fc.out_features != len(chars):
        raise RuntimeError('CTC output class count differs from dictionary')
    # Check CTCNet exactly matches official RCTC pre-softmax logit.
    probe = torch.randn(1, 3, 32, 192, device=device)
    captured = []
    hook = net.rctc.fc.register_forward_hook(lambda m, arg, output: captured.append(output.detach()))
    try:
        with torch.inference_mode():
            model(probe)
            logits = net(probe)
    finally:
        hook.remove()
    if len(captured) != 2 or not torch.allclose(captured[0], logits, atol=3e-4, rtol=3e-4):
        raise RuntimeError('Official-vs-extracted RCTC logits parity failed')
    del model, probe, captured
    return net, chars


def greedy_ctc(logits: np.ndarray, chars: list[str]) -> str:
    """CTC collapse, blank=0. No blank bias, beam width or confidence threshold."""
    pred = np.asarray(logits).argmax(-1).tolist()
    out, prev = [], -1
    for c in pred:
        if c != prev and c != 0:
            out.append(chars[c])
        prev = c
    return ''.join(out)


def geometry_ctc_same_crops(logits: list[np.ndarray], spans: list[tuple[int, int]], chars: list[str]) -> str:
    """Deterministic center-ownership stitching of CTC timesteps on ORIGINAL crop coordinates.

    SAME crops, backbone, preprocessing, and forward passes as Stitch v2.
    This is a baseline, not a novel or guaranteed accurate CTC alignment algorithm.
    """
    if not logits or len(logits) != len(spans):
        raise ValueError('Need one CTC logit tensor per crop span')
    if len(logits) == 1:
        return greedy_ctc(logits[0], chars)
    centers = [(a + b) / 2 for a, b in spans]
    cuts = [-float('inf')] + [(centers[i] + centers[i+1]) / 2 for i in range(len(centers)-1)] + [float('inf')]
    pieces = []
    for j, (lg, (x0, x1)) in enumerate(zip(logits, spans)):
        t = len(lg)
        if t == 0:
            raise RuntimeError('CTC produced zero timesteps for a crop')
        x = x0 + (np.arange(t) + 0.5) * ((x1 - x0) / t)
        keep = (x >= cuts[j]) & (x < cuts[j+1])
        if not keep.any():
            raise RuntimeError(f'No owned CTC frames for window {j}')
        pieces.append(np.asarray(lg)[keep])
    return greedy_ctc(np.concatenate(pieces, axis=0), chars)


class LongTextRecognizer:
    METHODS = ('full12', 'full64', 'stitch_v2', 'geom_ctc')

    def __init__(self, net: nn.Module, chars: list[str], device: str='cuda', batch_size: int=16):
        self.net = net.to(device).eval()
        self.chars = list(chars)
        self.device = str(device)
        self.batch_size = max(1, int(batch_size))  # memory-only knob, never tuned on accuracy
        if self.chars[0].lower() not in ('blank', '[blank]', '<blank>', 'blank_token'):
            raise ValueError('CTC blank must be class 0')

    @torch.inference_mode()
    def _forward_grouped(self, pils: list[Image.Image], max_ratio: int) -> list[np.ndarray]:
        tens = [preprocess_ltb(im, max_ratio) for im in pils]
        groups = defaultdict(list)
        for i, t in enumerate(tens):
            groups[tuple(t.shape)].append(i)
        ret: list[np.ndarray | None] = [None] * len(tens)
        for shape, ids in groups.items():
            _, h, w = shape
            # Bound temporary tensor volume as image width grows; only affects memory/throughput.
            bs = max(1, min(self.batch_size, int(self.batch_size * 192 / max(w, 1))))
            for start in range(0, len(ids), bs):
                ii = ids[start:start+bs]
                x = torch.stack([tens[i] for i in ii]).to(self.device)
                logits = self.net(x).float().detach().cpu().numpy()
                if logits.ndim != 3 or logits.shape[-1] != len(self.chars):
                    raise RuntimeError(f'Unexpected CTC shape: {logits.shape}')
                for i, lg in zip(ii, logits):
                    ret[i] = lg
        if any(x is None for x in ret):
            raise RuntimeError('Missing CTC results')
        return ret  # type: ignore[return-value]

    def predict_many(self, imgs: list[Image.Image], method: str='stitch_v2', return_info: bool=False):
        if method not in self.METHODS:
            raise ValueError(f'Method must be one of {self.METHODS}')
        if not imgs:
            return []
        imgs = [im.convert('RGB') for im in imgs]
        if method.startswith('full'):
            ratio = 12 if method == 'full12' else 64
            outs = self._forward_grouped(imgs, ratio)
            result = [{'prediction': normalize_ltb(greedy_ctc(lg, self.chars)),
                       'num_windows': 1, 'stitch_misses': 0} for lg in outs]
        else:
            crops, meta = [], []
            for im in imgs:
                left = len(crops)
                parts = make_chunks(im, LEGACY_CHUNK_AR, LEGACY_OVERLAP)
                crops.extend(p for p, _, _ in parts)
                spans = [(a, b) for _, a, b in parts]
                meta.append((left, len(parts), spans))
            lgs = self._forward_grouped(crops, max_ratio=12)
            result = []
            for left, count, spans in meta:
                crop_lgs = lgs[left:left+count]
                raw = [greedy_ctc(x, self.chars) for x in crop_lgs]
                norm_parts = [normalize_ltb(s) for s in raw]
                if method == 'stitch_v2':
                    text, misses = stitch_v2(norm_parts, spans)
                else:
                    text, misses = normalize_ltb(geometry_ctc_same_crops(crop_lgs, spans, self.chars)), 0
                result.append({'prediction': text, 'num_windows': count,
                               'stitch_misses': misses, 'crop_predictions': norm_parts,
                               'crop_spans': spans})
        return result if return_info else [r['prediction'] for r in result]

    def predict(self, img: Image.Image, method: str='stitch_v2', return_info: bool=False):
        return self.predict_many([img], method, return_info)[0]


def score_predictions(df):
    """df columns: id, gt, pred, optional bucket. Output exact-line acc & CER/S/D/I by bucket."""
    import pandas as pd
    rows = []
    for row in df.itertuples():
        gt = normalize_ltb(row.gt)
        pr = normalize_ltb(row.pred)
        ops = Levenshtein.editops(gt, pr)
        s = sum(o.tag == 'replace' for o in ops)
        d = sum(o.tag == 'delete' for o in ops)
        ins = sum(o.tag == 'insert' for o in ops)
        bucket = '26-35' if len(gt) <= 35 else ('36-55' if len(gt) <= 55 else '56+')
        rows.append({'id': row.id, 'gt': gt, 'pred': pr, 'bucket': bucket,
                     'correct': int(gt == pr), 'L': len(gt), 'S': s, 'D': d, 'I': ins})
    out = pd.DataFrame(rows)
    return out


def metric_table(df):
    """A-Avg only if ALL three LTB buckets are present."""
    import pandas as pd
    rows=[]
    for b in ['26-35', '36-55', '56+', 'ALL']:
        sub = df if b == 'ALL' else df[df.bucket == b]
        if len(sub) == 0:
            continue
        L = int(sub.L.sum())
        rows.append({'bucket': b, 'n': len(sub), 'accuracy': 100*sub.correct.mean(),
                     'CER': 100*(sub.S.sum()+sub.D.sum()+sub.I.sum())/max(L,1),
                     'S%': 100*sub.S.sum()/max(L,1),
                     'D%': 100*sub.D.sum()/max(L,1),
                     'I%': 100*sub.I.sum()/max(L,1)})
    present = set(df.bucket)
    if {'26-35','36-55','56+'} <= present:
        rows.append({'bucket': 'A-Avg', 'n':len(df),
                     'accuracy': float(np.mean([x['accuracy'] for x in rows if x['bucket'] in ('26-35','36-55','56+')]))})
    return pd.DataFrame(rows)


def run_regression_tests():
    # Three tests copied from original LTB; plus frame ownership and CTC blank semantics.
    sp = [(0, 100), (50, 150)]
    assert stitch_v2(['abcdefgh', 'efghijkl'], sp)[0] == 'abcdefghijkl'
    assert stitch_v2(['abcdefgh', 'xxxxyyyy'], [(0, 100), (100, 200)])[0] == 'abcdefghxxxxyyyy'
    im = Image.new('RGB',(170,20),'white')
    cs = make_chunks(im, LEGACY_CHUNK_AR, LEGACY_OVERLAP)
    assert [(a,b) for _,a,b in cs] == [(0,120),(50,170)]
    assert len(make_chunks(Image.new('RGB',(100,20)),LEGACY_CHUNK_AR,LEGACY_OVERLAP)) == 1
    # blank=0; two letters same token separated by blank must survive collapse.
    chars = ['blank','a','b']
    logits = np.array([[0,5,0],[0,5,0],[5,0,0],[0,5,0]],dtype=np.float32)
    assert greedy_ctc(logits,chars) == 'aa'
    one = np.array([[6,0,0],[0,8,0],[6,0,0],[0,0,8]],dtype=np.float32)
    two = np.array([[0,8,0],[6,0,0],[0,0,8],[6,0,0]],dtype=np.float32)
    assert geometry_ctc_same_crops([one,two],[(0,100),(50,150)],chars) == 'ab'
    sizes = [target_size_ltb(100,20), target_size_ltb(300,20), target_size_ltb(1500,20,12)]
    assert sizes == [(160,32),(384,32),(384,32)],sizes
    return True


if __name__ == '__main__':
    run_regression_tests()
    print('PASS: deterministic geometry/stitch/CTC/multi-window tests')

In [ ]:
%%writefile longtext_v3.py
"""Research candidate: Stitch v3 evidence-aware post-processing for SVTRv2 CTC.

NOT a validated improvement. Same *original-image* crops, same model and same
inference passes as Stitch v2. We change only the text-merge stage.

Frozen original Stitch v2 is imported from longtext_final.py. Three ablations:
    v3_conf: preserve v2 overlap selection; on *aligned single-character substitutions*
             choose the higher CTC event confidence rather than v2's fixed midpoint.
    v3_geom: preserve v2 successful matches; if v2 rejects a match, select by
             original-image geometry at the midpoint between the crop centres.
    v3_both: both rules together.

No new tuned hyperparameters. Confidence is a diagnostic proxy (not calibrated),
frame-to-pixel locations are approximate, and empirical gains are NOT guaranteed.
This is a coursework research prototype, not a new SOTA or publication claim.
"""
from __future__ import annotations

from dataclasses import dataclass
from collections import defaultdict
from typing import Sequence
import time

import numpy as np
from PIL import Image
from rapidfuzz.distance import Levenshtein

from longtext_final import (
    LongTextRecognizer, make_chunks, stitch_v2, normalize_ltb,
    LEGACY_CHUNK_AR, LEGACY_OVERLAP, greedy_ctc, geometry_ctc_same_crops,
)


@dataclass(frozen=True)
class CharEvidence:
    char: str
    confidence: float
    x: float


def ctc_events(logits: np.ndarray, chars: Sequence[str], x0: int, x1: int) -> list[CharEvidence]:
    """Convert greedy CTC character runs to confidence and approximate global x.

    For each nonblank run, its symbol probability is measured at the most
    confident timestep for that symbol; its position is that timestep's centre.
    This is not forced alignment and the position may differ from glyph centre.
    """
    v = np.asarray(logits, dtype=np.float64)
    if v.ndim != 2 or v.shape[1] != len(chars) or v.shape[0] < 1:
        raise ValueError(f'Expected nonempty [T,C] logits, got {v.shape}')
    z = v - np.max(v, axis=1, keepdims=True)
    expz = np.exp(z)
    p = expz / np.sum(expz, axis=1, keepdims=True)
    ids = np.argmax(v, axis=1)
    events = []
    t = 0
    while t < len(ids):
        j = t + 1
        while j < len(ids) and ids[j] == ids[t]:
            j += 1
        cls = int(ids[t])
        if cls != 0:
            label = normalize_ltb(str(chars[cls]))
            if len(label) > 1:
                raise ValueError(f'CTC label {chars[cls]!r} normalizes to multiple characters')
            if label:
                peak = t + int(np.argmax(p[t:j, cls]))
                x = float(x0 + (peak + 0.5) * ((x1 - x0) / len(ids)))
                events.append(CharEvidence(label, float(p[peak, cls]), x))
        t = j
    plain = normalize_ltb(greedy_ctc(v, list(chars)))
    if ''.join(e.char for e in events) != plain:
        raise AssertionError('Character-level evidence disagrees with greedy CTC output')
    return events


def _v2_match_info(out: str, b: str, prev_span, next_span, tol=0.34):
    """Exact legacy v2 scoring and acceptance; no change to existing thresholds."""
    wb = max(next_span[1] - next_span[0], 1)
    exp = max(0, prev_span[1] - next_span[0]) / wb * len(b)
    if exp < 0.5 or not b or not out:
        return {'mode': 'concat', 'exp': exp}
    lmax = min(len(out), len(b), int(round(exp * 1.6)) + 2)
    best = None
    for L in range(1, lmax + 1):
        d = Levenshtein.distance(out[-L:], b[:L])
        sc = L - 2*d - 0.5*abs(L-exp)
        if best is None or sc > best[0]:
            best = (sc, L, d)
    if best is None:
        return {'mode':'concat','exp':exp}
    sc,L,d = best
    return {
        'mode': ('reject' if sc<=0 or d>tol*L else 'accept'),
        'L': L, 'd': d, 'sc': sc, 'exp': exp,
    }


def stitch_evidence(
    chunks: Sequence[Sequence[CharEvidence]],
    spans: Sequence[tuple[int, int]],
    *,
    use_confidence: bool=True,
    geometric_fallback: bool=True,
    return_info: bool=False,
):
    """Stitch v3 candidate. Only alters v2 on disagreement or rejected overlap.

    No extra inference, no new learned weights, no threshold fitted to test.
    Comparison against raw v2 is essential; only adopt if robustly better.
    """
    if not chunks or len(chunks) != len(spans):
        raise ValueError('Need nonempty chunks and one span per chunk')
    spans = [tuple(map(int,s)) for s in spans]
    out = list(chunks[0])
    misses = 0
    changes = 0
    geom_misses = 0
    ambiguous = 0
    for j in range(1, len(chunks)):
        nxt = list(chunks[j])
        a = ''.join(e.char for e in out)
        b = ''.join(e.char for e in nxt)
        match = _v2_match_info(a,b,spans[j-1],spans[j])
        mode = match['mode']
        if mode == 'concat':
            out += nxt
            continue
        L = match['L']
        if mode == 'reject':
            misses += 1
            if geometric_fallback:
                # Midpoint between crop centres; lies inside their overlap for
                # equal-size windows. No predicted labels or GT used to route.
                prev_center = (spans[j-1][0] + spans[j-1][1]) / 2
                next_center = (spans[j][0] + spans[j][1]) / 2
                cut_x = (prev_center + next_center) / 2
                out = [e for e in out if e.x < cut_x] + [e for e in nxt if e.x >= cut_x]
                geom_misses += 1
            else:
                strip = min(int(round(match['exp'])),len(nxt))
                out += nxt[strip:]
            continue
        # This was an accepted legacy V2 match. Preserve it unless competing
        # CTC events are positionally aligned 1:1 with a substitution.
        h = L // 2
        left = out[-L:]
        right = nxt[:L]
        if use_confidence and match['d']:
            ops = Levenshtein.editops(''.join(x.char for x in left),
                                      ''.join(x.char for x in right))
            # Only same-index substitutions: no artificial re-indexing of
            # deletion/insertion or repeated-token alignments.
            safe = all(op.tag=='replace' and op.src_pos==op.dest_pos for op in ops)
            if safe:
                consensus = []
                for i in range(L):
                    aa, bb = left[i],right[i]
                    default = aa if i < h else bb
                    if aa.char == bb.char:
                        consensus.append(default)
                    elif aa.confidence > bb.confidence:
                        consensus.append(aa)
                        changes += int(default.char != aa.char)
                    elif bb.confidence > aa.confidence:
                        consensus.append(bb)
                        changes += int(default.char != bb.char)
                    else:
                        consensus.append(default)
                out = out[:-L] + consensus + nxt[L:]
                continue
            ambiguous += 1
        out = out[:-L+h] + nxt[h:]
    text = ''.join(e.char for e in out)
    if return_info:
        return {'prediction':text, 'stitch_misses':misses,
                'confidence_char_flips':changes, 'geometry_fallbacks':geom_misses,
                'unalignable_overlap':ambiguous}
    return text


class EvidenceLongTextRecognizer(LongTextRecognizer):
    """Reuses original crop and GPU logits for all v2/v3/geometry comparisons."""
    CROP_METHODS = ('stitch_v2','v3_conf','v3_geom','v3_both','geom_ctc')
    METHODS = LongTextRecognizer.METHODS + tuple(x for x in CROP_METHODS if x not in LongTextRecognizer.METHODS)

    def predict_all_cropped(self, imgs: list[Image.Image], return_info=False):
        imgs = [im.convert('RGB') for im in imgs]
        all_crops=[]; meta=[]
        for im in imgs:
            i = len(all_crops)
            parts=make_chunks(im,LEGACY_CHUNK_AR,LEGACY_OVERLAP)
            spans=[(a,b) for _,a,b in parts]
            all_crops.extend(p for p,_,_ in parts)
            meta.append((i,len(parts),spans))
        t0=time.perf_counter()
        all_lgs=self._forward_grouped(all_crops, max_ratio=12) if all_crops else []
        forward_seconds=time.perf_counter()-t0
        res={m:[] for m in self.CROP_METHODS}
        cpu_seconds=defaultdict(float)
        evidence_seconds=0.0
        for start,n,spans in meta:
            lgs=all_lgs[start:start+n]
            t1=time.perf_counter()
            evs=[ctc_events(lg,self.chars,a,b) for lg,(a,b) in zip(lgs,spans)]
            parts=[''.join(e.char for e in ev) for ev in evs]
            evidence_seconds+=time.perf_counter()-t1
            t1=time.perf_counter()
            basic,miss=stitch_v2(parts,spans)
            result={'stitch_v2':{'prediction':basic,'stitch_misses':miss}}
            cpu_seconds['stitch_v2']+=time.perf_counter()-t1
            for method,cc,gf in [('v3_conf',True,False),('v3_geom',False,True),('v3_both',True,True)]:
                t1=time.perf_counter()
                result[method]=stitch_evidence(evs,spans,use_confidence=cc,
                                              geometric_fallback=gf,return_info=True)
                cpu_seconds[method]+=time.perf_counter()-t1
            t1=time.perf_counter()
            result['geom_ctc']={'prediction':normalize_ltb(geometry_ctc_same_crops(lgs,spans,self.chars)),
                                'stitch_misses':0}
            cpu_seconds['geom_ctc']+=time.perf_counter()-t1
            for method in self.CROP_METHODS:
                d=dict(result[method]);d['num_windows']=n
                if return_info:
                    d['crop_predictions']=parts
                    d['crop_spans']=spans
                res[method].append(d)
            # Core parity: method with neither intervention is exactly v2.
            if stitch_evidence(evs,spans,use_confidence=False,geometric_fallback=False) != basic:
                raise AssertionError('Stitch v3 core does not reproduce legacy v2')
        return res, {'forward_seconds':forward_seconds,'evidence_seconds':evidence_seconds,
                     'cpu_seconds':dict(cpu_seconds),
                     'crops':len(all_crops), 'images':len(imgs)}

    def predict_many(self,imgs,method='stitch_v2',return_info=False):
        if method in self.CROP_METHODS:
            res,_ = self.predict_all_cropped(imgs,return_info)
            return res[method] if return_info else [x['prediction'] for x in res[method]]
        return super().predict_many(imgs,method,return_info)


def run_regression_tests_v3():
    from longtext_final import run_regression_tests
    assert run_regression_tests()
    chars=['blank','a','b','c','d']
    def logits(seq):
        ans=[]
        for idx in seq:
            row=np.full(len(chars), -8.,dtype=float)
            row[idx]=5.
            ans.append(row)
        return np.asarray(ans)
    lg=logits([1,1,0,1,2,0,2])
    e=ctc_events(lg,chars,10,80)
    assert ''.join(x.char for x in e)=='aabb' and all(10<=x.x<=80 for x in e)
    # Exact agreement is always preserved, no matter which ablation is on.
    p=['abcdef','cdefgh','efghij']
    spans=[(0,100),(50,150),(100,200)]
    evs=[[CharEvidence(c,.9,x0+(i+.5)*100/len(s)) for i,c in enumerate(s)]
          for s,(x0,x1) in zip(p,spans)]
    base=stitch_v2(p,spans)[0]
    for conf in (False,True):
        for geom in (False,True):
            ans=stitch_evidence(evs,spans,use_confidence=conf,geometric_fallback=geom)
            assert ans == base,(conf,geom,ans,base)
    # Exhaustively verify parity with original v2 for all random characters
    # when both interventions are disabled.
    rng=np.random.default_rng(14)
    alphabet='abcde'
    for i in range(200):
        k=int(rng.integers(1,7))
        ss=[''.join(rng.choice(list(alphabet),size=int(rng.integers(0,18)))) for j in range(k)]
        sp=[(j*50,j*50+100) for j in range(k)]
        evs=[[CharEvidence(c,.7,sp[j][0]+(z+.5)*100/max(len(s),1)) for z,c in enumerate(s)]
             for j,s in enumerate(ss)]
        a=stitch_v2(ss,sp)[0]
        b=stitch_evidence(evs,sp,use_confidence=False,geometric_fallback=False)
        assert a==b,(ss,sp,a,b)
    # Repeated letters remain separate events when separated by blank.
    assert ctc_events(logits([1,0,1]),chars,0,30)[0].char == 'a'
    assert ''.join(x.char for x in ctc_events(logits([1,0,1]),chars,0,30))=='aa'
    # Detect a real confidence substitution correction in an accepted overlap.
    a=list('abcxef');b=list('cdefgh');sp=[(0,100),(50,150)]
    ae=[CharEvidence(c, .02 if c=='x' else .8, (j+.5)*100/len(a)) for j,c in enumerate(a)]
    be=[CharEvidence(c, .99 if c=='d' else .8, 50+(j+.5)*100/len(b)) for j,c in enumerate(b)]
    base=stitch_v2([''.join(a),''.join(b)],sp)[0]
    newer=stitch_evidence([ae,be],sp,use_confidence=True,geometric_fallback=False)
    assert base!=newer and 'd' in newer,(base,newer)
    return {'passed':True,'toy_correction':{'v2':base,'v3_conf':newer},'random_parity_cases':200}


if __name__ == '__main__':
    print(run_regression_tests_v3())

In [ ]:
%%writefile orientation_probe.py
"""GT-blind orientation probe. Frozen original crop-before-resize + v3 geometry.

Never call this module with label text. All selector criteria use predictions,
CTC event geometry and confidence only. Scores are exploratory, not calibrated.
"""
from __future__ import annotations
from collections import defaultdict
from difflib import SequenceMatcher
import math,time
import numpy as np
import torch
from PIL import Image
import longtext_final as LT
import longtext_v3 as V3

METHODS=('original','cw_v3','ccw_v3','cw_v2','ccw_v2','cw_geom_ctc','ccw_geom_ctc','select_overlap','select_charconf')

def rotated(im,which):
    if which == 'original':return im.convert('RGB')
    if which == 'cw':return im.convert('RGB').transpose(Image.Transpose.ROTATE_270)
    if which == 'ccw':return im.convert('RGB').transpose(Image.Transpose.ROTATE_90)
    raise ValueError(which)

def overlap_agreement(event_lists,spans):
    match=0.;denom=0
    for i in range(len(spans)-1):
        left,right=event_lists[i],event_lists[i+1]
        a=max(spans[i][0],spans[i+1][0]);b=min(spans[i][1],spans[i+1][1])
        if b<=a:continue
        s1=''.join(e.char for e in left if a<=e.x<=b)
        s2=''.join(e.char for e in right if a<=e.x<=b)
        # Weight by shared recognized event count; empty/empty gives no false credit
        c=len(s1)+len(s2)
        if c:
            match+=c*SequenceMatcher(None,s1,s2,autojunk=False).ratio()
            denom+=c
    return float(match/denom) if denom else 0.0

def event_confidence(events):
    vals=[e.confidence for sub in events for e in sub]
    return float(np.mean(vals)) if vals else 0.0

def choose_orientation(a,b,kind='overlap'):
    # Positive direction names are literal rotations; no ground-truth, no label length
    if kind=='overlap':
        key_a=(a['agreement'],a['confidence'],len(a['pred']))
        key_b=(b['agreement'],b['confidence'],len(b['pred']))
    elif kind=='charconf':
        key_a=(a['confidence'],a['agreement'],len(a['pred']))
        key_b=(b['confidence'],b['agreement'],len(b['pred']))
    else:raise ValueError(kind)
    return ('cw',a) if key_a>=key_b else ('ccw',b)

class OrientationProbe:
    def __init__(self,net,chars,device='cuda',batch_size=2):
        self.core=LT.LongTextRecognizer(net,chars,device=device,batch_size=batch_size)
        self.chars=list(chars)

    def infer_batch(self,images):
        images=[im.convert('RGB') for im in images]
        # Original is 1-crop for this cohort. Orientation candidates use exact
        # make_chunks(raw pixels) THEN individual resized crop.
        allparts=[];meta=[]
        for n,im in enumerate(images):
            for angle in ('original','cw','ccw'):
                candidate=rotated(im,angle)
                parts=LT.make_chunks(candidate,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
                start=len(allparts)
                allparts.extend(p for p,_,_ in parts)
                meta.append((n,angle,start,len(parts),[(x0,x1) for _,x0,x1 in parts]))
        t0=time.perf_counter()
        logits=self.core._forward_grouped(allparts,max_ratio=12)
        gpu_s=time.perf_counter()-t0
        record=[{} for _ in images]
        for n,angle,start,count,spans in meta:
            cur=logits[start:start+count]
            events=[V3.ctc_events(lg,self.chars,a,b) for lg,(a,b) in zip(cur,spans)]
            v3=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)
            parts=[''.join(e.char for e in ev) for ev in events]
            v2,_=LT.stitch_v2(parts,spans)
            geom=LT.normalize_ltb(LT.geometry_ctc_same_crops(cur,spans,self.chars))
            result={'pred':v3,'v2':v2,'geom_ctc':geom,'n_crops':count,
                    'agreement':overlap_agreement(events,spans),
                    'confidence':event_confidence(events),'T_sum':sum(len(z) for z in cur)}
            record[n][angle]=result
        outputs=[]
        for r in record:
            cw=r['cw'];ccw=r['ccw']
            a=choose_orientation(cw,ccw,'overlap')
            b=choose_orientation(cw,ccw,'charconf')
            d={'original':r['original']['pred'],'cw_v3':cw['pred'],'ccw_v3':ccw['pred'],
               'cw_v2':cw['v2'],'ccw_v2':ccw['v2'],
               'cw_geom_ctc':cw['geom_ctc'],'ccw_geom_ctc':ccw['geom_ctc'],
               'select_overlap':a[1]['pred'],'select_charconf':b[1]['pred'],
               'overlap_choice':a[0],'charconf_choice':b[0],
               'cw_agreement':cw['agreement'],'ccw_agreement':ccw['agreement'],
               'cw_confidence':cw['confidence'],'ccw_confidence':ccw['confidence'],
               'cw_n_crops':cw['n_crops'],'ccw_n_crops':ccw['n_crops'],
               'original_T':r['original']['T_sum']}
            outputs.append(d)
        return outputs,{'num_tensors':len(allparts),'gpu_forward_seconds':gpu_s}

def run_cpu_tests():
    from PIL import Image
    class Dummy(torch.nn.Module):
        def forward(self,x):
            b,_,h,w=x.shape
            v=torch.zeros(b,w//4,3,device=x.device);v[:,:,0]=2.;v[:,(w//8):((w//8)+2),1]=3.
            return v
    im=Image.new('RGB',(18,210),'white')
    assert rotated(im,'cw').size==(210,18) and rotated(im,'ccw').size==(210,18)
    model=OrientationProbe(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    x,perf=model.infer_batch([im,Image.new('RGB',(31,220),'white')])
    assert len(x)==2 and all(set(METHODS).issubset(z.keys()) for z in x)
    assert all(z['overlap_choice'] in ('cw','ccw') for z in x)
    assert all(isinstance(z[k],str) for z in x for k in METHODS)
    assert any(z['cw_n_crops']>=2 for z in x)
    assert perf['num_tensors']>=10
    # Selector cannot access GT because it accepts only orientation dictionaries
    assert choose_orientation({'pred':'a','agreement':0.8,'confidence':0.3},
                              {'pred':'b','agreement':0.5,'confidence':0.9},'overlap')[0]=='cw'
    return {'passed':True,'methods':list(METHODS),'dummy_tensors':perf['num_tensors']}

In [ ]:
%%writefile density_probe.py
"""Input-resolution/forced-crop probe for high-density LTB lines.

All transformations GT-blind and predetermined. No model training or test tuning.
"""
from __future__ import annotations
from collections import defaultdict
from PIL import Image
import numpy as np
import torch
import time
from typing import Sequence
import longtext_final as LT
import longtext_v3 as V3

METHODS = ('baseline','width_x2','width_x3','force_2','force_3','force_3_width_x3')
MAX_INPUT_W = 768


def resized_tensor(img: Image.Image, factor: int) -> torch.Tensor:
    """Original MSR when factor=1; direct resize raw pixels (no double interpolation).

    factor>1 enlarges *horizontal* input dimension while keeping MSR height,
    a diagnostic OOD intervention on the frozen checkpoint.
    """
    if factor == 1:
        return LT.preprocess_ltb(img,max_ratio=12)
    w, h = LT.target_size_ltb(*img.size,max_ratio=12)
    w2 = int(min(MAX_INPUT_W, 4*round(w*factor/4)))
    if w2 <= w: raise ValueError('Expected an expanded width')
    mode = Image.Resampling.BICUBIC if hasattr(Image,'Resampling') else Image.BICUBIC
    a = np.asarray(img.convert('RGB').resize((w2,h),mode),dtype=np.float32)/255.
    return torch.from_numpy(((a-0.5)/0.5).transpose(2,0,1).copy())


def forced_crops(img: Image.Image, n: int):
    """Overlapping 2- or 3-view raw-image tiling, no GT or recognition output."""
    if n not in (2,3): raise ValueError('Only 2 and 3 fixed raw-image views')
    w,h=img.size
    if w < 8: return [(img,0,w)]  # no-op for truly narrow images; log number of crops
    cw=min(w-1,max(4,int(round(w*(0.65 if n==2 else 0.52)))))
    starts = [0,w-cw] if n==2 else [0,int(round((w-cw)/2)),w-cw]
    if len(set(starts)) != n: raise ValueError('Degenerate forced-crop geometry')
    assert starts[0]==0 and starts[-1]+cw == w
    return [(img.crop((x,0,x+cw,h)),x,x+cw) for x in starts]


class Probe:
    def __init__(self,net,chars,device='cuda',batch_size=8):
        self.net=net.to(device).eval();self.chars=list(chars)
        self.device=device;self.batch_size=batch_size
        if not self.chars or str(self.chars[0]).lower() not in ('blank','[blank]','<blank>','blank_token'):
            raise ValueError('Expected blank class zero')

    @torch.inference_mode()
    def _forward_tensors(self,tensors: Sequence[torch.Tensor]) -> list[np.ndarray]:
        groups=defaultdict(list)
        for i,t in enumerate(tensors):groups[tuple(t.shape)].append(i)
        ret=[None]*len(tensors)
        for shape,ids in groups.items():
            _,h,w=shape
            volume=max(1,w*h)
            bs=max(1,min(self.batch_size,int(self.batch_size*192*32/volume)))
            for start in range(0,len(ids),bs):
                ii=ids[start:start+bs]
                xx=torch.stack([tensors[i] for i in ii],dim=0).to(self.device)
                raw=self.net(xx).detach().float().cpu().numpy()
                assert raw.ndim==3 and raw.shape[2]==len(self.chars),(raw.shape,len(self.chars))
                for i,x in zip(ii,raw):ret[i]=x
        assert all(x is not None for x in ret)
        return ret

    def batch(self, imgs: Sequence[Image.Image]):
        imgs=[im.convert('RGB') for im in imgs]
        inputs=[];mapping=[]
        for i,im in enumerate(imgs):
            for m in METHODS:
                if m=='baseline': parts=[(im,0,im.width)]; factor=1
                elif m=='width_x2':parts=[(im,0,im.width)]; factor=2
                elif m=='width_x3':parts=[(im,0,im.width)]; factor=3
                elif m=='force_2':parts=forced_crops(im,2);factor=1
                elif m=='force_3':parts=forced_crops(im,3);factor=1
                elif m=='force_3_width_x3':parts=forced_crops(im,3);factor=3
                else:raise AssertionError(m)
                start=len(inputs)
                inputs.extend(resized_tensor(p,factor) for p,_,_ in parts)
                spans=[(a,b) for _,a,b in parts]
                mapping.append((i,m,start,len(parts),spans))
        t0=time.perf_counter();logits=self._forward_tensors(inputs)
        if str(self.device).startswith('cuda'):torch.cuda.synchronize()
        inference_s=time.perf_counter()-t0
        outputs={m:[None]*len(imgs) for m in METHODS}
        for i,m,start,n,spans in mapping:
            cur=logits[start:start+n]
            if n==1:
                pred=LT.normalize_ltb(LT.greedy_ctc(cur[0],self.chars))
            else:
                events=[V3.ctc_events(x,self.chars,a,b) for x,(a,b) in zip(cur,spans)]
                pred=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)
            outputs[m][i]={'prediction':pred,'num_crops':n,'ctc_Ts':[len(x) for x in cur],
                           'spans':spans}
        return outputs, {'forward_s':inference_s,'tensors':len(inputs)}


def run_cpu_tests():
    from PIL import Image
    import torch
    class Dummy(torch.nn.Module):
        def forward(self,x):
            b,_,h,w=x.shape
            t=w//4
            out=torch.zeros(b,t,3,device=x.device)
            out[...,0]=1.0
            out[...,1]=-0.1
            out[:,t//3,1]=2
            return out
    img=Image.new('RGB',(120,30),'white')
    assert [int(x.shape[-1]) for x in [resized_tensor(img,1),resized_tensor(img,2),resized_tensor(img,3)]]==[128,256,384]
    assert len(forced_crops(img,2))==2 and len(forced_crops(img,3))==3
    assert len(forced_crops(img,3)[0][0].getdata())>0
    p=Probe(Dummy(),['blank','a','b'],'cpu',batch_size=4)
    result,perf=p.batch([img,Image.new('RGB',(210,30),'white')])
    assert set(result)==set(METHODS) and perf['tensors']==2*11
    assert all(len(v)==2 and isinstance(v[0]['prediction'],str) for v in result.values())
    assert result['baseline'][0]['ctc_Ts']==[32]
    assert result['width_x3'][0]['ctc_Ts']==[96]
    assert result['force_3'][0]['num_crops']==3
    return {'passed':True,'methods':list(METHODS),'total_tensors_for_2_imgs':perf['tensors']}

In [ ]:
%%writefile unified_router.py
"""A fully source-image/CTC-driven router; historical IDs/labels are never input."""
from __future__ import annotations
from PIL import Image
import time
import torch
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP

class UnifiedRecognizer:
    def __init__(self,net,chars,device='cuda',batch_size=2):
        self.base=V3.EvidenceLongTextRecognizer(net,chars,device=device,batch_size=batch_size)
        self.orientation=OP.OrientationProbe(net,chars,device=device,batch_size=batch_size)
        self.dense=DP.Probe(net,chars,device=device,batch_size=batch_size)
        self.chars=list(chars)

    @staticmethod
    def classify(w,h,n):
        # Original raw-pixel geometry ONLY. ID, filename, GT and reference prediction unused.
        if int(w)<=int(h):
            assert int(n)==1, ('Tall expected a single original chunk',w,h,n)
            return 'orientation'
        if int(n)==1:
            return 'dense'
        return 'v3'

    def dense3_x3(self,images):
        # Reproduces density_probe.Probe.batch(...)["force_3_width_x3"] EXACTLY,
        # without paying for the five unused ablation branches.
        tensors=[];meta=[]
        for im in images:
            parts=DP.forced_crops(im,3)
            start=len(tensors)
            tensors.extend(DP.resized_tensor(p,3) for p,_,_ in parts)
            spans=[(a,b) for _,a,b in parts]
            meta.append((start,len(parts),spans))
        logits=self.dense._forward_tensors(tensors)
        ret=[]
        for start,n,spans in meta:
            cur=logits[start:start+n]
            if n==1:
                pred=LT.normalize_ltb(LT.greedy_ctc(cur[0],self.chars))
            else:
                events=[V3.ctc_events(x,self.chars,a,b) for x,(a,b) in zip(cur,spans)]
                pred=V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True)
            ret.append({'pred':pred,'dense_n_crops':n,'dense_ctc_Ts':[len(x) for x in cur]})
        return ret,{'dense_extra_tensors':len(tensors)}

    @torch.inference_mode()
    def infer_batch(self,images):
        images=[im.convert('RGB') for im in images]
        t0=time.perf_counter()
        # Independent matched baseline on EVERY image; the same official v3 Geometry.
        baseline=self.base.predict_many(images,method='v3_geom',return_info=True)
        routings=[self.classify(im.width,im.height,b['num_windows']) for im,b in zip(images,baseline)]
        rows=[{'route':route,'baseline_pred':b['prediction'],'baseline_num_windows':b['num_windows'],
               'pred':b['prediction'],'angle_choice':'', 'dense_n_crops':0,
               'orientation_agreement_delta':None,'orientation_conf_delta':None}
              for route,b in zip(routings,baseline)]
        ori_idx=[i for i,r in enumerate(routings) if r=='orientation']
        if ori_idx:
            ori,ori_perf=self.orientation.infer_batch([images[i] for i in ori_idx])
            for i,v in zip(ori_idx,ori):
                # Assert two independent original-orientation implementations agree.
                assert v['original']==rows[i]['baseline_pred'], ('Tall original parity',i)
                rows[i]['pred']=v['select_overlap']
                rows[i]['angle_choice']=v['overlap_choice']
                rows[i]['orientation_agreement_delta']=float(v['cw_agreement']-v['ccw_agreement'])
                rows[i]['orientation_conf_delta']=float(v['cw_confidence']-v['ccw_confidence'])
        else:ori_perf={'num_tensors':0}
        dense_idx=[i for i,r in enumerate(routings) if r=='dense']
        if dense_idx:
            preds,dense_perf=self.dense3_x3([images[i] for i in dense_idx])
            for i,v in zip(dense_idx,preds):
                rows[i]['pred']=v['pred']
                rows[i]['dense_n_crops']=int(v['dense_n_crops'])
                rows[i]['dense_ctc_Ts']=v['dense_ctc_Ts']
        else:dense_perf={'dense_extra_tensors':0}
        for r in rows:
            r.setdefault('dense_ctc_Ts',[])
        return rows,{'elapsed_seconds':time.perf_counter()-t0,
                     'baseline_images':len(images),'orientation_images':len(ori_idx),'dense_images':len(dense_idx),
                     'orientation_extra_tensors':ori_perf['num_tensors'],
                     'dense_extra_tensors':dense_perf['dense_extra_tensors']}

def run_cpu_tests():
    class Dummy(torch.nn.Module):
        def forward(self,x):
            b,_,h,w=x.shape
            t=w//4
            logits=torch.zeros((b,t,3),device=x.device)
            logits[:,:,0]=2.0
            logits[:,max(1,t//4),1]=4.0
            return logits
    net=Dummy()
    p=UnifiedRecognizer(net,['blank','a','b'],device='cpu',batch_size=2)
    imgs=[Image.new('RGB',(18,210),'white'),Image.new('RGB',(120,30),'white'),Image.new('RGB',(330,20),'white')]
    pred,perf=p.infer_batch(imgs)
    assert [x['route'] for x in pred]==['orientation','dense','v3'], pred
    assert pred[0]['angle_choice'] in ('cw','ccw')
    assert pred[1]['dense_n_crops']==3
    assert pred[2]['baseline_num_windows']>=2
    assert all(isinstance(x['pred'],str) for x in pred)
    # Dense-only wrapper must decode IDENTICALLY to the pre-existing full probe.
    all_dense,_=p.dense.batch([imgs[1]])
    only,_=p.dense3_x3([imgs[1]])
    assert only[0]['pred']==all_dense['force_3_width_x3'][0]['prediction']
    assert only[0]['dense_ctc_Ts']==all_dense['force_3_width_x3'][0]['ctc_Ts']
    return {'passed':True,'routes':[x['route'] for x in pred],'dense_parity':True}

In [ ]:
%%writefile alex_ctc.py
"""ALEX-CTC E1: Alignment-Latent cross-view character evidence (C0 prototype).

GT-blind, frozen SVTRv2-T. Reference-anchored CTC Viterbi alignment to every
visible crop, preserving baseline spatial order, then multi-crop posterior
inference over possible character substitutions. No thresholds on GT scores.
Not an exact joint decoder for insertions/deletions (future C1).
"""
from __future__ import annotations
import numpy as np
import longtext_v3 as V3
from longtext_final import normalize_ltb

ALPH='0123456789abcdefghijklmnopqrstuvwxyz'

def normalized_probs(logits,chars):
    z=np.asarray(logits,np.float64);z-=z.max(axis=1,keepdims=True)
    p=np.exp(z);p/=p.sum(axis=1,keepdims=True)
    q=np.zeros((len(p),37),np.float64)
    for i,c in enumerate(chars):
        norm=normalize_ltb(c) if i else ''
        if len(norm)==1 and norm in ALPH:q[:,1+ALPH.index(norm)]+=p[:,i]
        else:q[:,0]+=p[:,i]
    q=np.maximum(q,1e-30);q/=q.sum(axis=1,keepdims=True)
    return q


def reference_events(logits,spans,chars):
    evs=[V3.ctc_events(q,chars,*sp) for q,sp in zip(logits,spans)]
    out=list(evs[0])
    for j in range(1,len(evs)):
        nxt=list(evs[j]); a=''.join(x.char for x in out);b=''.join(e.char for e in nxt)
        match=V3._v2_match_info(a,b,spans[j-1],spans[j])
        if match['mode']=='concat':out+=nxt;continue
        if match['mode']=='reject':
            cut=((spans[j-1][0]+spans[j-1][1])/2+(spans[j][0]+spans[j][1])/2)/2
            out=[e for e in out if e.x<cut]+[e for e in nxt if e.x>=cut]
            continue
        L=match['L'];h=L//2
        out=out[:-L+h]+nxt[h:]
    s=normalize_ltb(''.join(e.char for e in out))
    actual=normalize_ltb(V3.stitch_evidence(evs,spans,use_confidence=False,geometric_fallback=True))
    assert s==actual,(s,actual)
    return out


def ctc_viterbi(q,txt):
    """CTC forced Viterbi path for normalized text, returns per-letter peak frame.

    q is 37-class posterior with punctuation mass mapped to blank. Unlike greedy,
    this marginal remains conditional on a hypothesized string, not ground truth.
    """
    T,C=q.shape
    if not txt:return [],0.
    lab=[1+ALPH.index(c) for c in txt]
    S=2*len(lab)+1
    minimum_frames=len(lab)+sum(lab[j]==lab[j-1] for j in range(1,len(lab)))
    if minimum_frames>T:return None,-np.inf
    st=np.zeros(S,np.int16);st[1::2]=lab
    L=np.log(np.maximum(q,1e-30))
    dp=np.full((T,S),-np.inf,np.float64)
    back=np.full((T,S),-1,np.int16)
    dp[0,0]=L[0,0]
    if S>1:dp[0,1]=L[0,st[1]]
    for t in range(1,T):
        for s in range(S):
            paths=[(dp[t-1,s],s)]
            if s>0:paths.append((dp[t-1,s-1],s-1))
            if s>1 and st[s]!=0 and st[s]!=st[s-2]:paths.append((dp[t-1,s-2],s-2))
            best=max(paths,key=lambda a:a[0]);dp[t,s]=best[0]+L[t,st[s]];back[t,s]=best[1]
    if S==1:end=0
    else:end=int(np.argmax([dp[T-1,S-2],dp[T-1,S-1]]))+S-2
    score=float(dp[T-1,end])
    if not np.isfinite(score):return None,-np.inf
    states=np.empty(T,dtype=int)
    s=end
    for t in range(T-1,-1,-1):
        states[t]=s
        if t>0:s=int(back[t,s]);assert s>=0
    frames=[]
    for k,c in enumerate(lab):
        ts=np.where(states==(2*k+1))[0]
        if not len(ts):return None,-np.inf
        best=int(ts[np.argmax(q[ts,c])]);frames.append(best)
    return frames,score


def decode(logits,spans,chars,return_info=False):
    ev=reference_events(logits,spans,chars)
    base=''.join(e.char for e in ev)
    if not ev:return ('',{'n_aligned_crops':0,'changed_chars':0}) if return_info else ''
    votes=[[] for _ in ev]
    feasible=0
    for k,(raw,(a,b)) in enumerate(zip(logits,spans)):
        ids=[j for j,e in enumerate(ev) if a<=e.x<=b]
        if not ids:continue
        txt=''.join(ev[j].char for j in ids)
        q=normalized_probs(raw,chars)
        frames,score=ctc_viterbi(q,txt)
        if frames is None:continue
        feasible+=1
        for j,t in zip(ids,frames):
            x=ev[j].x
            w=.5+.5*max(0,1-2*abs(x-(a+b)/2)/max(b-a,1))
            votes[j].append((w,q[t]))
    out=[];changed=0
    for j,e in enumerate(ev):
        if not votes[j]:out.append(e.char);continue
        v=votes[j];ww=np.array([s for s,_ in v],np.float64);ww/=ww.sum()
        # Bayesian geometric mean from multiple observation windows: all
        # crops vote once, so overlapping frame count cannot swamp output.
        score=sum(float(w)*np.log(np.maximum(d[1:],1e-30)) for w,(_,d) in zip(ww,v))
        c=ALPH[int(score.argmax())];out.append(c);changed+=int(c!=e.char)
    result=''.join(out)
    return (result,{'n_aligned_crops':feasible,'changed_chars':changed,'baseline_chars':len(ev)}) if return_info else result


def cpu_tests():
    chars=['blank','a','b','c']
    def logits(ids):
        x=np.full((len(ids),4),-8.,np.float32)
        for i,j in enumerate(ids):x[i,j]=8.
        return x
    a=logits([1,0,1,0,2]);assert decode([a],[(0,100)],chars)=='aab'
    q=normalized_probs(a,chars)
    frm,score=ctc_viterbi(q,'aab');assert len(frm)==3 and np.isfinite(score)
    assert decode([a,a],[(0,100),(0,100)],chars)=='aab'
    return {'passed':True,'repeat_ctc':True,'reference_parity':True}

In [ ]:
%%writefile alex_c1.py
"""ALEX-CTC C1 exploratory constrained edit-lattice over multiple CTC crop posteriors.

No GT or dataset IDs in inference; only observed crop posterior, crop geometry,
V3 reference event positions, and optional C0 same-length substitutions.
Edits: substitution, insertion, deletion. Candidate generation from witness crop
CTC greedy character events; optional edits are scored using exact CTC *forward*
sequence likelihood ratios on each covering crop. A length-changing path is
selected by budgeted DAG dynamic programming (not an unconstrained optimal
joint CTC decoder). Does not mix frame posteriors or use language priors.
"""
from __future__ import annotations
from dataclasses import dataclass, replace
from collections import defaultdict
from difflib import SequenceMatcher
from typing import Sequence
import numpy as np
import longtext_v3 as V3
import alex_ctc as AX

ALPH=AX.ALPH
NEG=-1e50

@dataclass(frozen=True)
class Proposal:
    op:str
    position:int
    char:str
    x:float


def ctc_forward_logprob(logq:np.ndarray, text:str)->float:
    """Exact forward sum over CTC paths for 37 normalized classes, log domain."""
    T,C=logq.shape
    if C!=37 or T<1:
        raise ValueError('Expected [T,37] nonempty normalized log posterior')
    if any(ch not in ALPH for ch in text):
        raise ValueError('Unnormalized transcription')
    labels=[1+ALPH.index(c) for c in text]
    repeats=sum(labels[j]==labels[j-1] for j in range(1,len(labels)))
    if len(labels)+repeats>T:return float('-inf')
    if not labels:return float(logq[:,0].sum())
    S=2*len(labels)+1
    states=np.zeros(S,dtype=np.int32)
    states[1::2]=labels
    can_jump=np.zeros(S,dtype=bool)
    can_jump[2:]=(states[2:]!=0)&(states[2:]!=states[:-2])
    prev=np.full(S,-np.inf,np.float64)
    prev[0]=logq[0,0];prev[1]=logq[0,labels[0]]
    for t in range(1,T):
        stay=prev
        step=np.empty(S,np.float64);step[0]=-np.inf;step[1:]=prev[:-1]
        jump=np.full(S,-np.inf,np.float64);jump[2:]=np.where(can_jump[2:],prev[:-2],-np.inf)
        prev=np.logaddexp(np.logaddexp(stay,step),jump)+logq[t,states]
    return float(np.logaddexp(prev[-1],prev[-2]))


def proposal_position(ref,op,j):
    if op!='ins':return float(ref[j].x)
    if j==0:return float(ref[0].x)
    if j==len(ref):return float(ref[-1].x)
    return float((ref[j-1].x+ref[j].x)/2.)


def generate_proposals(ref, crop_events, spans):
    """Generate one-symbol edits witnessed by original crop greedy events.

    Matching uses equal-string anchors (no hidden GT oracle). Restrict witness
    to the middle 70% of its crop to reduce truncation artefacts.
    """
    n=len(ref); proposals={}
    for events, (a,b) in zip(crop_events,spans):
        ids=[i for i,e in enumerate(ref) if a<=e.x<=b]
        if not ids or ids!=list(range(ids[0],ids[-1]+1)):continue
        original=''.join(ref[i].char for i in ids)
        observed=''.join(e.char for e in events)
        matcher=SequenceMatcher(None,original,observed,autojunk=False)
        for tag,u0,u1,v0,v1 in matcher.get_opcodes():
            op=None; pos=None; ch=''
            if tag=='replace' and u1-u0==1 and v1-v0==1:
                op='sub';pos=ids[u0];ch=events[v0].char
            elif tag=='insert' and v1-v0==1:
                op='ins';pos=ids[u0] if u0<len(ids) else ids[-1]+1
                ch=events[v0].char
            elif tag=='delete' and u1-u0==1:
                op='del';pos=ids[u0]
            if op is None:continue
            if ch and ch not in ALPH:continue
            x=proposal_position(ref,op,pos)
            # Avoid source-view boundaries, where partial characters are common.
            if not (a+0.15*(b-a)<=x<=b-0.15*(b-a)):continue
            # A witness symbol should also be spatially plausible relative to
            # the reference event (in CTC time/global crop coordinates).
            if op in ('sub','ins'):
                witness_x=events[v0].x
                if abs(witness_x-x)>max(0.18*(b-a),4):continue
            p=Proposal(op,pos,ch,x)
            proposals[(p.op,p.position,p.char)]=p
    return list(proposals.values())


def change_local(local,loc,op,ch):
    if op=='sub':
        if not 0<=loc<len(local):return None
        return local[:loc]+ch+local[loc+1:]
    if op=='del':
        if not 0<=loc<len(local):return None
        return local[:loc]+local[loc+1:]
    if op=='ins':
        if not 0<=loc<=len(local):return None
        return local[:loc]+ch+local[loc:]
    raise ValueError(op)


def evidence_scores(ref,logits,spans,chars,proposals):
    n=len(ref)
    prepared=[]
    for raw,(a,b) in zip(logits,spans):
        ids=[i for i,e in enumerate(ref) if a<=e.x<=b]
        if not ids or ids!=list(range(ids[0],ids[-1]+1)):continue
        q=AX.normalized_probs(raw,chars)
        logq=np.log(np.maximum(q,1e-35))
        txt=''.join(ref[i].char for i in ids)
        base=ctc_forward_logprob(logq,txt)
        prepared.append((int(a),int(b),ids,txt,logq,base))
    scores={}; score_details={}
    for p in proposals:
        diffs=[]; coverage=0
        for a,b,ids,txt,logq,base in prepared:
            if not np.isfinite(base):continue
            # Only trust center-covered views to avoid seam artefacts.
            if not (a+0.10*(b-a)<=p.x<=b-0.10*(b-a)):continue
            if p.op=='ins':
                if p.position<ids[0] or p.position>ids[-1]+1:continue
            else:
                if p.position not in ids:continue
            loc=p.position-ids[0]
            mutant=change_local(txt,loc,p.op,p.char)
            if mutant is None or mutant==txt:continue
            alt=ctc_forward_logprob(logq,mutant)
            if not np.isfinite(alt):continue
            coverage+=1
            # Equal weight per independently computed crop observation; avoid
            # repeated-frame double-counting inside one crop.
            diffs.append(float(alt-base))
        # No label/GT in criteria: insist on at least two usable views and
        # a consistent positive exact-CTC log-likelihood ratio.
        if coverage>=2:
            score=float(np.mean(diffs))
            if np.isfinite(score):
                scores[p]=score
                score_details[p]=(coverage,sum(d>0 for d in diffs),float(min(diffs)))
    return scores,score_details


def lattice_decode(reference:str, scores, max_edits=2, margin=1.8, types=('sub','ins','del')):
    """DAG over string positions / optional insertion slots, with edit budget.

    Keep edges have score 0. Proposed edits score (avg CTC log BF - margin).
    At most one insertion per gap, substitutions/deletions consume one glyph.
    """
    n=len(reference)
    ins=defaultdict(list); glyph=defaultdict(list)
    for p,score in scores.items():
        if p.op not in types or score<=margin or not np.isfinite(score):continue
        (ins if p.op=='ins' else glyph)[p.position].append((p,score-margin))
    # dp keyed (number_of_edits) => (score, transcript, accepted edits)
    states={0:(0.,'',[])}
    for i in range(n+1):
        # optional insertion before position i
        after={}
        for used,(v,t,ops) in states.items():
            if used not in after or v>after[used][0]:after[used]=(v,t,ops)
            if used<max_edits:
                for p,gain in ins.get(i,[]):
                    uv=used+1; cand=(v+gain,t+p.char,ops+[p])
                    if uv not in after or cand[0]>after[uv][0]:after[uv]=cand
        if i==n:states=after;break
        nxt={}
        for used,(v,t,ops) in after.items():
            nxt[used]=(v,t+reference[i],ops)
            if used<max_edits:
                for p,gain in glyph.get(i,[]):
                    uv=used+1; text=t+(p.char if p.op=='sub' else '')
                    cand=(v+gain,text,ops+[p])
                    if uv not in nxt or cand[0]>nxt[uv][0]:nxt[uv]=cand
        states=nxt
    best=max(states.values(),key=lambda z:z[0])
    return best[1],best[2],best[0]


def decode(logits,spans,chars,c0=None,return_info=False):
    """Produces C1_subonly and C1_edit using same proposal+scoring pool."""
    if c0 is None:c0=AX.decode(logits,spans,chars)
    ref0=AX.reference_events(logits,spans,chars)
    if len(c0)!=len(ref0):
        raise AssertionError('C0 and reference must share one-to-one char slots')
    ref=[replace(e,char=ch) for e,ch in zip(ref0,c0)]
    crop_events=[V3.ctc_events(raw,chars,*sp) for raw,sp in zip(logits,spans)]
    proposals=generate_proposals(ref,crop_events,spans)
    scores,details=evidence_scores(ref,logits,spans,chars,proposals)
    sub,sub_ops,_=lattice_decode(c0,scores,types=('sub',))
    edit,edit_ops,_=lattice_decode(c0,scores,types=('sub','ins','del'))
    info={'n_proposals':len(proposals),'n_scored':len(scores),
          'n_accepted_sub':len(sub_ops),'n_accepted_edit':len(edit_ops),
          'edit_insertions':sum(p.op=='ins' for p in edit_ops),
          'edit_deletions':sum(p.op=='del' for p in edit_ops),
          'edit_substitutions':sum(p.op=='sub' for p in edit_ops),
          'edit_path':';'.join(f'{p.op}@{p.position}:{p.char}' for p in edit_ops)}
    return (sub,edit,info) if return_info else edit


def cpu_tests():
    from itertools import product
    def brute(q,y):
        result=0.
        for path in product(range(q.shape[1]),repeat=q.shape[0]):
            collapsed=[]; prev=None
            for c in path:
                if c!=0 and c!=prev:collapsed.append(ALPH[c-1])
                prev=c
            if ''.join(collapsed)==y:
                v=1.
                for t,c in enumerate(path):v*=q[t,c]
                result+=v
        return result
    rng=np.random.default_rng(321)
    for L in ('','a','aa','aba','abb'):
        q=np.zeros((4,37),np.float64)
        pp=rng.dirichlet([1.8,1.8,1.8],size=4)
        q[:,:3]=pp
        a=ctc_forward_logprob(np.log(np.maximum(q,1e-35)),L)
        b=brute(q,L)
        assert np.isclose(np.exp(a),b,atol=1e-10), (L,a,b)
    # DAG unit tests demonstrate that C1 can change transcript length.
    dummy=lambda op,i,c:Proposal(op,i,c,float(i))
    for name,ref,p,answer in [('insertion','ab',dummy('ins',1,'c'),'acb'),
                              ('deletion','abc',dummy('del',1,''),'ac'),
                              ('substitution','abc',dummy('sub',1,'d'),'adc')]:
        out,ops,_=lattice_decode(ref,{p:4.},max_edits=1,margin=1.)
        assert out==answer,(name,out,answer)
    # Ref case can be preserved exactly when there is no positive witness.
    out,_,_=lattice_decode('abcdef',{},max_edits=2)
    assert out=='abcdef'
    return {'passed':True,'ctc_forward_vs_bruteforce':5,
            'edit_lattice':['insertion','deletion','substitution'],
            'zero_edit_fallback':True}

In [ ]:
%%writefile space_ctc.py
"""SPaCE-CTC E0: frozen CTC spatial-posterior alignment and prefix-beam pilot.

Hypothesis only. Geometry of CTC timesteps is approximate. No label or image ID
is consumed by inference functions. This module is intentionally self-contained
with no external library beyond NumPy and the pinned project's longtext_final.
"""
from __future__ import annotations

import math
from typing import Sequence
import numpy as np
from longtext_final import greedy_ctc, normalize_ltb


def logadd(*values):
    vals = [float(x) for x in values if x != -math.inf]
    if not vals:
        return -math.inf
    m = max(vals)
    return m + math.log(sum(math.exp(x-m) for x in vals))


def stable_probs(logits):
    a = np.asarray(logits, dtype=np.float64)
    if a.ndim != 2 or a.shape[0] == 0 or a.shape[1] < 2 or not np.isfinite(a).all():
        raise ValueError('Need finite nonempty CTC logits [T, C]')
    z = a - np.max(a, axis=-1, keepdims=True)
    exp = np.exp(z)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def owned_frames(pieces: Sequence[np.ndarray], spans: Sequence[tuple[int,int]]):
    if len(pieces)!=len(spans) or not pieces:
        raise ValueError('CTC evidence/spans mismatch')
    mids = [(a+b)/2. for a,b in spans]
    cuts = [-np.inf] + [(mids[i]+mids[i+1])/2 for i in range(len(mids)-1)] + [np.inf]
    anchors = []
    for i,(p,(x0,x1)) in enumerate(zip(pieces, spans)):
        t = p.shape[0]
        if x1<=x0 or t==0:
            raise ValueError('Invalid crop span or CTC length')
        coords = x0+(np.arange(t)+.5)*(x1-x0)/t
        ids = np.where((coords >= cuts[i]) & (coords < cuts[i+1]))[0]
        if not len(ids):
            raise RuntimeError('No frames owned by crop')
        anchors.extend((i,int(k),float(coords[k])) for k in ids)
    return anchors


def spatial_lattice(logits, spans, *, mode='aligned', alpha=.35, radius=2):
    """Anchor ownership + GT-blind geometric/CTC neighbor alignment.

    For each owned timestep, the owner observation retains at least 1-alpha of
    the log-probability weight. Neighbor contributions in overlapping spatial
    support are aligned softly within +/- radius frames around projected x.
    mode='coordinate' uses a strictly geometric nearest-frame ablation.
    mode='aligned' incorporates CTC-posterior similarity without GT.
    """
    if mode not in ('aligned','coordinate') or not (0<=alpha<=.49) or not (1<=radius<=6):
        raise ValueError('Unsupported spatial-lattice configuration')
    prob = [stable_probs(v) for v in logits]
    if len({v.shape[1] for v in prob})!=1:
        raise ValueError('CTC class counts differ between crops')
    anchors = owned_frames(prob, spans)
    out = []
    n_mixed=0
    for i,k,x in anchors:
        owner = prob[i][k]
        neighbors=[]
        for j, (q,(a,b)) in enumerate(zip(prob,spans)):
            if j==i or not (a<=x<=b):
                continue
            expected = (x-a)/(b-a)*len(q)-.5
            # Explicit one-frame-nearest support for coordinate-only control.
            if mode=='coordinate':
                neighbor = q[int(np.clip(round(expected),0,len(q)-1))]
            else:
                lo=max(0,int(np.floor(expected))-radius)
                hi=min(len(q)-1,int(np.ceil(expected))+radius)
                candidates=np.arange(lo,hi+1,dtype=int)
                # Bhattacharyya similarity in the actual distribution;
                # window around projected location prevents semantic shortcuts.
                sim = np.sqrt(q[candidates]*owner[None,:]).sum(axis=1)
                score=np.log(np.maximum(sim,1e-10)) - .5*((candidates-expected)/max(radius/2,1))**2
                weights=np.exp(score-np.max(score))
                weights /= weights.sum()
                neighbor=np.sum(q[candidates]*weights[:,None],axis=0)
            neighbors.append(neighbor)
        if neighbors:
            n_mixed+=1
            each=alpha/len(neighbors)
            logp=(1-alpha)*np.log(np.maximum(owner,1e-30))
            for p in neighbors:
                logp+=each*np.log(np.maximum(p,1e-30))
            logp-=logadd(*logp)
            out.append(logp)
        else:
            out.append(np.log(np.maximum(owner,1e-30)))
    ans=np.stack(out,axis=0)
    assert np.isfinite(ans).all()
    return ans, {'frames':len(anchors),'mixed_frames':n_mixed,'n_crops':len(prob)}


def ownership_log_probs(logits,spans):
    p=[stable_probs(v) for v in logits]
    return np.stack([np.log(np.maximum(p[i][k],1e-30)) for i,k,x in owned_frames(p,spans)])


def prefix_beam_search(log_probs,chars, beam_width=8, topk=6, nbest=8):
    """Reference CTC prefix beam search with blank=0, repeated-label logic.

    Inference is bounded via per-frame top-k emissions plus each current prefix
    last label (for exact repeat-path recurrence under pruned emissions).
    Beam pruning is approximate; therefore n-best membership is NOT an upper
    bound over the entire CTC distribution.
    """
    lp=np.asarray(log_probs,dtype=np.float64)
    if lp.ndim!=2 or lp.shape[1]!=len(chars) or lp.shape[0]<1:
        raise ValueError('Invalid prefix beam input')
    if beam_width<1 or topk<1:
        raise ValueError('Beam width and topk must be >=1')
    beam={(): (0., -math.inf)}
    for row in lp:
        chosen = set(np.argpartition(row, -min(len(row),topk))[-min(len(row),topk):].tolist())
        chosen.add(0)
        for pre in beam:
            if pre:
                chosen.add(pre[-1])
        next_b={}
        def add(key, add_pb=-math.inf,add_pnb=-math.inf):
            prev_pb,prev_pnb=next_b.get(key,(-math.inf,-math.inf))
            next_b[key]=(logadd(prev_pb,add_pb),logadd(prev_pnb,add_pnb))
        for pre,(pb,pnb) in beam.items():
            whole=logadd(pb,pnb)
            add(pre,add_pb=whole+row[0])
            for c in chosen:
                if c==0:continue
                if pre and pre[-1]==c:
                    add(pre,add_pnb=pnb+row[c])
                    add(pre+(c,),add_pnb=pb+row[c])
                else:
                    add(pre+(c,),add_pnb=whole+row[c])
        beam=dict(sorted(next_b.items(),key=lambda x:logadd(*x[1]),reverse=True)[:beam_width])
    ranked=sorted(beam.items(),key=lambda x:logadd(*x[1]),reverse=True)[:nbest]
    result=[]
    for seq,(pb,pnb) in ranked:
        pred=normalize_ltb(''.join(chars[c] for c in seq))
        result.append({'pred':pred,'logp':float(logadd(pb,pnb))})
    return result


def decode_all(logits, spans, chars, beam_width=8,topk=6):
    owner=ownership_log_probs(logits,spans)
    coord,mdc=spatial_lattice(logits,spans,mode='coordinate')
    aligned,mda=spatial_lattice(logits,spans,mode='aligned')
    def g(lp):
        return normalize_ltb(greedy_ctc(lp,chars))
    runs={
        'ownership_greedy':g(owner),
        'ownership_beam':prefix_beam_search(owner,chars,beam_width,topk),
        'coordinate_greedy':g(coord),
        'coordinate_beam':prefix_beam_search(coord,chars,beam_width,topk),
        'space_greedy':g(aligned),
        'space_beam':prefix_beam_search(aligned,chars,beam_width,topk),
        'metadata':mda,
    }
    return runs, {'owner':owner,'coordinate':coord,'aligned':aligned}


def cpu_tests():
    chars=['blank','a','b','c']
    def pth(ids):
        x=np.full((len(ids),len(chars)),-8.,np.float32)
        for i,c in enumerate(ids):x[i,c]=8.
        return x
    single=[pth([1,0,1,0,2])]
    out,latt=decode_all(single,[(0,100)],chars,5,4)
    assert out['ownership_greedy']=='aab',out
    assert out['ownership_beam'][0]['pred']=='aab',out
    assert out['space_beam'][0]['pred']=='aab',out
    repeated=[pth([1,1,1])]
    z=prefix_beam_search(ownership_log_probs(repeated,[(0,10)]),chars,4,4)
    assert z[0]['pred']=='a',z
    parts=[pth([1,0,2]),pth([2,0,3])]
    k,m=decode_all(parts,[(0,60),(30,90)],chars,5,4)
    assert k['metadata']['n_crops']==2 and k['metadata']['frames']>0
    assert m['aligned'].shape[1]==4
    assert np.max(np.abs(np.exp(m['aligned']).sum(axis=1)-1))<1e-7
    assert np.allclose(m['owner'],m['coordinate']) is False or k['metadata']['mixed_frames']>=0
    assert prefix_beam_search(np.array([[0,-100,-100,-100]],float),chars,4,4)[0]['pred']==''
    return {'passed':True,'repeated_character_test':True,'blank_test':True,'overlap_test':True}

In [ ]:
%%writefile alex_e5.py
"""ALEX E5: posterior-generated CTC candidate edits + matched C1 scoring.

EXPLORATORY. For each crop, CTC prefix beam search provides non-greedy
sequence hypotheses; Viterbi places their characters in crop coordinates.
Differences to C0 reference become (sub/ins/del) hypotheses and are scored by
exact CTC forward per crop using *the SAME scoring and DAG* as C1. This is an
ablation of candidate discovery, not an exact joint multi-crop CTC solver.
No labels, GT, IDs or historical predictions are accepted by decode().
"""
from __future__ import annotations
from collections import Counter
import numpy as np
import alex_ctc as AX
import alex_c1 as C1
import longtext_v3 as V3
import space_ctc as SPACE

ALPH=AX.ALPH
BEAM_CHARS=['']+list(ALPH)


def fast_forced_frames(q:np.ndarray, text:str):
    """Forced Viterbi CTC alignment: exact max-path with blank/repeats."""
    tmax=len(q)
    if not text:return [],0.
    labels=np.fromiter((1+ALPH.index(c) for c in text),dtype=np.int16)
    if len(labels)+sum(labels[j]==labels[j-1] for j in range(1,len(labels)))>tmax:
        return None, -np.inf
    states=np.zeros(2*len(labels)+1,dtype=np.int16)
    states[1::2]=labels
    size=len(states)
    allowed=np.zeros(size,dtype=bool)
    allowed[2:]=(states[2:]!=0)&(states[2:]!=states[:-2])
    logq=np.log(np.maximum(q,1e-30))
    prev=np.full(size,-np.inf,dtype=np.float64)
    prev[0]=logq[0,0]
    prev[1]=logq[0,labels[0]]
    trace=np.full((tmax,size),-1,np.int16)
    minus=np.full(size,-np.inf,dtype=np.float64)
    for t in range(1,tmax):
        step=minus.copy();step[1:]=prev[:-1]
        jump=minus.copy();jump[2:]=np.where(allowed[2:],prev[:-2],-np.inf)
        choices=np.stack((prev,step,jump),axis=0)
        which=np.argmax(choices,axis=0)
        trace[t]=np.arange(size)-which
        prev=choices[which,np.arange(size)]+logq[t,states]
    terminal=size-1 if prev[-1]>=prev[-2] else size-2
    best=float(prev[terminal])
    if not np.isfinite(best):return None, -np.inf
    state_path=np.zeros(tmax,np.int16)
    current=terminal
    for t in range(tmax-1,-1,-1):
        state_path[t]=current
        if t:current=int(trace[t,current])
    frames=[]
    for j,lab in enumerate(labels):
        ts=np.flatnonzero(state_path==2*j+1)
        if len(ts)==0:return None,-np.inf
        frames.append(int(ts[np.argmax(q[ts,lab])]))
    return frames,best


def beam_events(logq,span,beam_width=8,topk=5,nbest=5):
    """Return alternative character-event streams from actual 37-class posterior."""
    q=np.exp(logq)
    nbest_rows=SPACE.prefix_beam_search(logq,BEAM_CHARS,
            beam_width=beam_width,topk=topk,nbest=nbest)
    out=[];seen=set()
    a,b=span;T=len(q)
    for row in nbest_rows:
        text=row['pred']
        if not text or text in seen:continue
        seen.add(text)
        frames,score=fast_forced_frames(q,text)
        if frames is None:continue
        evs=[V3.CharEvidence(c,float(q[t,1+ALPH.index(c)]),
               float(a+(t+0.5)*(b-a)/T)) for c,t in zip(text,frames)]
        out.append((text,evs,float(row['logp'])))
    return out


def get_proposals(ref,logits,spans,chars,beam_width=8,topk=5,nbest=5):
    """Get greedy-only and unique non-greedy proposals on same raw evidence."""
    greedy_events=[V3.ctc_events(raw,chars,*sp) for raw,sp in zip(logits,spans)]
    greedy=C1.generate_proposals(ref,greedy_events,spans)
    greedy_keys={(p.op,p.position,p.char) for p in greedy}
    posterior={};counts=Counter(); n_uniq_alt=0;n_failed_align=0
    for raw,span,grev in zip(logits,spans,greedy_events):
        q=AX.normalized_probs(raw,chars)
        lp=np.log(np.maximum(q,1e-30))
        crop_greedy=''.join(ev.char for ev in grev)
        alternatives=beam_events(lp,span,beam_width,topk,nbest)
        for text,evs,logprob in alternatives:
            if text==crop_greedy:continue
            n_uniq_alt+=1
            for p in C1.generate_proposals(ref,[evs],[span]):
                k=(p.op,p.position,p.char)
                if k not in greedy_keys:
                    posterior[k]=p
                    counts[k]+=1
    info={'greedy_proposals':len(greedy),'posterior_only_proposals':len(posterior),
          'non_greedy_crop_hypotheses':n_uniq_alt,
          'posterior_repeated_witnesses':sum(v>=2 for v in counts.values())}
    return greedy, list(posterior.values()),info


def decode(logits,spans,chars,*,beam_width=8,frame_topk=5,nbest=5,
           max_edits=2,margin=1.8,return_info=False):
    c0=AX.decode(logits,spans,chars)
    ref0=AX.reference_events(logits,spans,chars)
    if len(ref0)!=len(c0):raise RuntimeError('C0 length mismatch')
    from dataclasses import replace
    ref=[replace(e,char=c) for e,c in zip(ref0,c0)]
    gp,pp,stats=get_proposals(ref,logits,spans,chars,beam_width,frame_topk,nbest)
    g_scores,_=C1.evidence_scores(ref,logits,spans,chars,gp)
    p_scores,_=C1.evidence_scores(ref,logits,spans,chars,pp)
    base,b_ops,_=C1.lattice_decode(c0,g_scores,max_edits=max_edits,margin=margin)
    joint={**g_scores,**p_scores}
    union,u_ops,_=C1.lattice_decode(c0,joint,max_edits=max_edits,margin=margin)
    posterior_only,p_ops,_=C1.lattice_decode(c0,p_scores,max_edits=max_edits,margin=margin)
    inf={**stats,'posterior_scored':len(p_scores),
        'posterior_positive_scores':sum(s>margin for s in p_scores.values()),
        'changed_by_posterior':int(union!=base),
        'union_edits':len(u_ops),
        'union_insertions':sum(p.op=='ins' for p in u_ops),
        'union_deletions':sum(p.op=='del' for p in u_ops),
        'union_substitutions':sum(p.op=='sub' for p in u_ops),
        'union_posterior_only_edits':sum((p.op,p.position,p.char) not in {(g.op,g.position,g.char) for g in gp} for p in u_ops)}
    return (base,posterior_only,union,inf) if return_info else union


def cpu_tests():
    import itertools
    from numpy.testing import assert_allclose
    rng=np.random.default_rng(9)
    for text in ('a','aa','ab','aba','abb'):
        q=np.full((6,37),1e-30,float)
        q[:,:3]=rng.dirichlet([2,3,4],size=6)
        q/=q.sum(axis=1,keepdims=True)
        fast,v=fast_forced_frames(q,text)
        slow,s=AX.ctc_viterbi(q,text)
        assert_allclose(v,s,rtol=0,atol=1e-9)
        assert fast==slow,(text,fast,slow)
    q=np.zeros((4,37),float)
    q[:,0]=.4;q[:,1]=.45;q[:,2]=.15
    out=beam_events(np.log(np.maximum(q,1e-30)),(0,100),beam_width=12,topk=3,nbest=8)
    assert len(out)>=2 and all(len(y)==len(ev) for y,ev,_ in out)
    return {'passed':True,'forced_viterbi_vs_reference':5,'non_greedy_beam_candidates':len(out)}

## E7 mechanism: posterior-derived variable-length span edits, grouped cross-scale evidence

GT-blind decoder API receives only the C1 reference, crop logits, crop geometry, and the frozen vocabulary. Crop pairs share a `group`, so multiple scales cannot manufacture additional independent votes. `e7_native` and `e7_cross` use identical candidate search and score functions. Neither uses a language model or GT.

In [ ]:
%%writefile alex_e7.py
"""ALEX E7 / SCOPE-CTC: scale-paired, segment-level counterfactual CTC decoding.

Research prototype. Uses frozen SVTRv2-T crop logits, no GT/IDs in inference.
The full CTC forward score is exact only for the nominated local substring and
crop-frame slice; the spatial segmentation and view dependence are approximations.
This is NOT a proof of literature novelty or generalization.
"""
from __future__ import annotations
from dataclasses import dataclass, replace
from difflib import SequenceMatcher
from collections import defaultdict
from itertools import combinations
import numpy as np
import longtext_v3 as V3
import alex_ctc as AX
import alex_c1 as C1
import alex_e5 as E5

@dataclass(frozen=True)
class SpanEdit:
    start: int
    end: int
    new: str
    x: float


def mapped_events(reference, source_events):
    """Interpolate character geometry for a C1-edited reference (GT-blind)."""
    source=''.join(e.char for e in source_events)
    if not source_events:
        return []
    coords=np.full(len(reference),np.nan)
    matcher=SequenceMatcher(None,source,reference,autojunk=False)
    for tag,a,b,c,d in matcher.get_opcodes():
        if d==c:continue
        if tag=='equal':
            coords[c:d]=[e.x for e in source_events[a:b]]
        else:
            lo=float(source_events[a-1].x) if a>0 else float(source_events[0].x)-3.
            hi=float(source_events[b].x) if b<len(source_events) else float(source_events[-1].x)+3.
            coords[c:d]=np.linspace(lo,hi,d-c+2)[1:-1]
    if np.isnan(coords).any():raise ValueError('Reference coordinate mapping incomplete')
    coords=np.maximum.accumulate(coords + np.arange(len(coords))*1e-5)
    return [V3.CharEvidence(ch,1.,float(x)) for ch,x in zip(reference,coords)]


def new_edit(proposal, text, ref_events, ids, observed_events, src_a,src_b):
    tag,u0,u1,v0,v1=proposal
    if tag=='equal':return None
    if tag not in ('insert','delete','replace'):return None
    if (u1-u0)>6 or (v1-v0)>6:return None
    if tag=='insert' and v0==v1:return None
    start=ids[u0] if u0<len(ids) else ids[-1]+1
    end=ids[u1-1]+1 if u1>u0 else start
    if end-start>6 or len(text[v0:v1])>6 or (end==start and v0==v1):return None
    new=text[v0:v1]
    if any(ch not in AX.ALPH for ch in new):return None
    if start==end:
        if not ref_events:return None
        if start==0:x=ref_events[0].x
        elif start==len(ref_events):x=ref_events[-1].x
        else:x=.5*(ref_events[start-1].x+ref_events[start].x)
    else:x=np.mean([e.x for e in ref_events[start:end]])
    if not (src_a+.12*(src_b-src_a) <= x <= src_b-.12*(src_b-src_a)):
        return None
    if v1>v0 and observed_events:
        wx=np.mean([e.x for e in observed_events[v0:v1]])
        if abs(wx-x)>max(5.,.19*(src_b-src_a)):
            return None
    return SpanEdit(int(start),int(end),new,float(x))


def generate_proposals(reference, ref_events, views, chars, beam_width=6, nbest=3):
    """GT-blind proposals from CTC greedy AND N-best on each view."""
    props={}; witnesses=defaultdict(set); beam_count=0
    for v in views:
        a,b=v['span']
        ids=[j for j,e in enumerate(ref_events) if a<=e.x<=b]
        if not ids or ids!=list(range(ids[0],ids[-1]+1)):continue
        # One virtual reference anchor on each side disambiguates an internal
        # insertion near overlap boundaries; only centrally witnessed edits
        # can survive the following spatial check.
        ids=list(range(max(0,ids[0]-1),min(len(reference),ids[-1]+2)))
        local=''.join(reference[j] for j in ids)
        raw=v['logits']
        greedy=V3.ctc_events(raw,chars,a,b)
        source=[(''.join(e.char for e in greedy),greedy)]
        q=AX.normalized_probs(raw,chars)
        lp=np.log(np.maximum(q,1e-35))
        beams=E5.beam_events(lp,(a,b),beam_width=beam_width,topk=5,nbest=nbest)
        beam_count+=len(beams)
        source.extend((text,ev) for text,ev,_ in beams)
        seen=set()
        for txt,events in source:
            if txt in seen:continue
            seen.add(txt)
            for opc in SequenceMatcher(None,local,txt,autojunk=False).get_opcodes():
                p=new_edit(opc,txt,ref_events,ids,events,a,b)
                if p is None:continue
                key=(p.start,p.end,p.new)
                props[key]=p;witnesses[key].add((v['group'],v['scale']))
    return list(props.values()),witnesses,beam_count


def make_view(raw,span,group,scale,chars,ref_events):
    q=AX.normalized_probs(raw,chars)
    a,b=span
    ids=[j for j,e in enumerate(ref_events) if a<=e.x<=b]
    q=np.asarray(q,dtype=np.float64)
    if not ids or ids!=list(range(ids[0],ids[-1]+1)):
        frames=None
    else:
        tx=''.join(ref_events[i].char for i in ids)
        frames,_=AX.ctc_viterbi(q,tx)
    return {'logits':raw,'span':tuple(span),'group':int(group),'scale':str(scale),
            'q':q,'logq':np.log(np.maximum(q,1e-35)),'indices':ids,'frames':frames}


def local_bf(view,p,reference,anchors=2):
    ids=view['indices']; frames=view['frames']
    if frames is None or not ids:return None
    a,b=view['span']
    if not (a+.10*(b-a) <= p.x <= b-.10*(b-a)):return None
    if p.start<ids[0] or p.end>ids[-1]+1:return None
    local_l=max(ids[0],p.start-anchors)
    local_r=min(ids[-1]+1,p.end+anchors)
    if local_r<=local_l:return None
    first=local_l-ids[0];last=local_r-1-ids[0]
    q=view['logq'];T=len(q)
    slack=max(3,int(round(T*.05)))
    projected=int(np.clip(round((p.x-a)/max(b-a,1)*T),0,T-1))
    # Do not crop away the very gap we are attempting to recover: a missing
    # span may have no reference-aligned token near its actual visual frames.
    tl=max(0,min(frames[first],projected)-slack)
    tr=min(T,max(frames[last],projected)+slack+1)
    if tr-tl<4:return None
    plain=reference[local_l:local_r]
    alt=reference[local_l:p.start]+p.new+reference[p.end:local_r]
    if alt==plain:return None
    qpiece=q[tl:tr]
    lp_plain=C1.ctc_forward_logprob(qpiece,plain)
    lp_alt=C1.ctc_forward_logprob(qpiece,alt)
    if not np.isfinite(lp_plain) or not np.isfinite(lp_alt):return None
    return float(lp_alt-lp_plain)


def robust_score(p,reference,views):
    """Median across scales in a crop, then across distinct crop groups."""
    groups=defaultdict(list)
    for v in views:
        d=local_bf(v,p,reference)
        if d is not None and np.isfinite(d):groups[v['group']].append(float(d))
    if len(groups)<2:return None
    scores=[float(np.median(g)) for g in groups.values()]
    support=sum(s>0 for s in scores)
    if support<2:return None
    # Group averaging is robust to repeated scale views within one image region.
    score=float(np.median(scores))
    cost=1.0+.6*max(p.end-p.start,len(p.new))
    return score-cost,{'group_count':len(groups),'support_groups':support,
                       'group_median':score,'cost':cost}


def compatible(a,b):
    # insertion slot occupies a virtual half-character interval.
    sa=2*a.start if a.start==a.end else 2*a.start+1
    ea=sa+1 if a.start==a.end else 2*a.end+1
    sb=2*b.start if b.start==b.end else 2*b.start+1
    eb=sb+1 if b.start==b.end else 2*b.end+1
    return ea<=sb or eb<=sa


def select_edits(reference,weighted,max_candidates=20,max_accepted=3,max_char_budget=8):
    # Fixed-size sparse segmental lattice search, no GT or image ID.
    ranked=sorted(((p,float(v)) for p,v in weighted.items() if v>0),
                  key=lambda z:(-z[1],z[0].start,z[0].end,z[0].new))[:max_candidates]
    best=[];best_gain=0.
    for k in range(1,min(max_accepted,len(ranked))+1):
        for subset in combinations(ranked,k):
            chars=sum(max(p.end-p.start,len(p.new)) for p,_ in subset)
            if chars>max_char_budget:continue
            ps=sorted((p for p,_ in subset),key=lambda a:(a.start,a.end,a.new))
            if any(not compatible(ps[i],ps[j]) for i in range(len(ps)) for j in range(i+1,len(ps))):
                continue
            gain=sum(score for _,score in subset)
            if gain>best_gain+1e-9:
                best_gain=gain;best=ps
    result=reference
    for p in sorted(best,key=lambda p:(p.start,p.end),reverse=True):
        result=result[:p.start]+p.new+result[p.end:]
    return result,best,best_gain


def decode(reference,base_logits,base_spans,extra_logits,extra_spans,chars,*,return_info=False):
    if len(base_logits)!=len(base_spans) or len(extra_logits)!=len(extra_spans):
        raise ValueError('Logits/spans mismatch')
    raw_ref=AX.reference_events(base_logits,base_spans,chars)
    events=mapped_events(reference,raw_ref)
    if not events or not reference:
        empty_fields={'candidates':0,'scored':0,'accepted':0,
                      'insertion_chars':0,'deleted_chars':0,
                      'beam_hypotheses':0,'gain':0.0,'edits':''}
        info={f'{scope}_{key}':value for scope in ('native','cross')
              for key,value in empty_fields.items()}
        return (reference,reference,info) if return_info else reference
    base=[make_view(q,sp,i,'base',chars,events) for i,(q,sp) in enumerate(zip(base_logits,base_spans))]
    extra=[]
    # Pair cross-scale evidence by exact original-image crop span, not timestep.
    group_map={tuple(sp):i for i,sp in enumerate(base_spans)}
    for q,sp in zip(extra_logits,extra_spans):
        if tuple(sp) not in group_map:
            raise ValueError('Extra view span not matched to base crop')
        extra.append(make_view(q,sp,group_map[tuple(sp)],'alt',chars,events))
    pred=[];metadata=[]
    for scope,views in [('native',base),('cross',base+extra)]:
        ps,wit,nb=generate_proposals(reference,events,views,chars)
        scores={};dets={}
        for p in ps:
            # A positive CTC Bayes factor alone may favor repeated letters even
            # when no observation actually proposed the character. Require
            # independent spatial *candidate witnesses*, not just likelihood.
            key=(p.start,p.end,p.new)
            if len({group for group,_scale in wit.get(key,set())})<2:
                continue
            s,detail=None,None
            out=robust_score(p,reference,views)
            if out is not None:s,detail=out
            if s is not None and s>0:
                scores[p]=s;dets[p]=detail
        text,ops,gain=select_edits(reference,scores)
        pred.append(text)
        metadata.append({'scope':scope,'candidates':len(ps),'scored':len(scores),
                         'accepted':len(ops),'insertion_chars':sum(len(p.new) for p in ops if p.start==p.end),
                         'deleted_chars':sum(p.end-p.start for p in ops if not p.new),
                         'beam_hypotheses':nb,'gain':float(gain),
                         'edits':'|'.join(f'{p.start}:{p.end}:{p.new}' for p in ops)})
    info={'native_'+k:v for k,v in metadata[0].items() if k!='scope'}
    info.update({'cross_'+k:v for k,v in metadata[1].items() if k!='scope'})
    return (pred[0],pred[1],info) if return_info else pred[1]


def cpu_tests():
    # GT-blind text event coordinate mapping when C1 changed string length.
    source=[V3.CharEvidence(c,1.,float(i*10+5)) for i,c in enumerate('abcdef')]
    mapped=mapped_events('abcxdef',source)
    assert ''.join(v.char for v in mapped)=='abcxdef'
    assert all(mapped[i].x<=mapped[i+1].x for i in range(len(mapped)-1))
    p=SpanEdit(1,1,'x',10.)
    assert select_edits('ab',{p:3.})[0]=='axb'
    p=SpanEdit(1,2,'',10.)
    assert select_edits('abc',{p:3.})[0]=='ac'
    p=SpanEdit(1,3,'xy',10.)
    assert select_edits('abcd',{p:3.})[0]=='axyd'
    # Exact local likelihood verification with two independent spatial crops.
    alphabet=['blank','a','b','c']
    def logits(v):
        z=np.full((len(v),4),-8.,np.float32)
        for i,k in enumerate(v):z[i,k]=8.
        return z
    q=logits([1,0,2,0,3,0,1,0,2,0,3])
    native,cross,info=decode('abcabc',[q,q],[(0,60),(0,60)],[],[],alphabet,return_info=True)
    assert native=='abcabc' and cross=='abcabc'
    assert info['cross_accepted']==0
    blank=np.full((12,4),-8.,np.float32)
    blank[:,0]=8.
    for reference in ('','ab'):
        native,cross,meta=decode(reference,[blank],[(0,80)],[blank],[(0,80)],alphabet,return_info=True)
        assert native==cross==reference
        assert meta['native_candidates']==meta['cross_candidates']==0
        assert meta['native_accepted']==meta['cross_accepted']==0
        assert meta['cross_edits']==''
    # Posterior-only character recovery: neither base view's greedy contains c,
    # paired higher-resolution views expose it in TWO overlapping crop groups.
    def paired_logits(group,extra=False):
        p=np.full((18,4),1e-5);p[:,0]=.99997
        for frame,lab in ((3,1),(13,2)):
            p[frame]=[.001,.001,.001,.001];p[frame,lab]=.997
        t=9 if group==0 else 6
        p[t]=[.80,.001,.001,.198] if not extra else [.01,.001,.001,.988]
        p/=p.sum(axis=1,keepdims=True)
        return np.log(p)
    psp=[(0,80),(20,100)]
    lows=[paired_logits(0),paired_logits(1)]
    highs=[paired_logits(0,True),paired_logits(1,True)]
    ref=AX.decode(lows,psp,alphabet)
    assert ref=='ab'
    a,b,debug=decode(ref,lows,psp,highs,psp,alphabet,return_info=True)
    assert a=='ab' and b=='acb' and debug['cross_accepted']==1,(a,b,debug)
    return {'passed':True,'span_insertion_deletion_replacement':3,'ref_coordinates':True,
            'no_unwarranted_edit':True,'posterior_only_cross_scale_missing_char':'ab -> acb'}


In [ ]:
%%writefile alex_e7_experiment.py
"""E7 matched baseline + cross-scale posterior segment reconstruction adapter.

Inputs are pixels and frozen model only. Labels/IDs/evaluation archives are NOT read.
"""
import time
from PIL import Image
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP
import unified_router as UNI
import alex_ctc as AX
import alex_c1 as C1
import alex_e7 as E7


def predict_image(image, recognizer):
    im=image.convert('RGB');chars=recognizer.chars
    tic=time.perf_counter()
    parts=LT.make_chunks(im,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
    native_spans=[(int(a),int(b)) for _,a,b in parts]
    native_logits=recognizer.base._forward_grouped([p for p,_,_ in parts],max_ratio=12)
    native_e=[V3.ctc_events(z,chars,*sp) for z,sp in zip(native_logits,native_spans)]
    v3=LT.normalize_ltb(V3.stitch_evidence(native_e,native_spans,use_confidence=False,geometric_fallback=True))
    route=UNI.UnifiedRecognizer.classify(im.width,im.height,len(parts))
    base_logits=native_logits;base_spans=native_spans
    base_parts=parts
    unified=v3
    if route=='orientation':
        scores={}
        for angle in ('cw','ccw'):
            rotated=OP.rotated(im,angle)
            p=LT.make_chunks(rotated,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
            spans=[(int(a),int(b)) for _,a,b in p]
            lg=recognizer.base._forward_grouped([c for c,_,_ in p],max_ratio=12)
            ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(lg,spans)]
            pred=LT.normalize_ltb(V3.stitch_evidence(ev,spans,use_confidence=False,geometric_fallback=True))
            scores[angle]={'pred':pred,'agreement':OP.overlap_agreement(ev,spans),
                           'confidence':OP.event_confidence(ev),'logits':lg,
                           'spans':spans,'parts':p}
        chosen,winner=OP.choose_orientation(scores['cw'],scores['ccw'],kind='overlap')
        base_logits=winner['logits'];base_spans=winner['spans'];base_parts=winner['parts']
        unified=winner['pred']
    elif route=='dense':
        base_parts=DP.forced_crops(im,3)
        base_spans=[(int(a),int(b)) for _,a,b in base_parts]
        base_logits=recognizer.dense._forward_tensors([DP.resized_tensor(p,3) for p,_,_ in base_parts])
        ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(base_logits,base_spans)]
        unified=LT.normalize_ltb(V3.stitch_evidence(ev,base_spans,use_confidence=False,geometric_fallback=True))
    c0=AX.decode(base_logits,base_spans,chars)
    c1_sub,c1,c1_meta=C1.decode(base_logits,base_spans,chars,c0=c0,return_info=True)
    # Matched second observation: same raw pixel crop spans, different horizontal MSR.
    # For Dense, the secondary view is x2 and active primary is x3.
    alt_logits=recognizer.dense._forward_tensors([DP.resized_tensor(p,2) for p,_,_ in base_parts])
    secondary_spans=[(int(a),int(b)) for _,a,b in base_parts]
    assert base_spans==secondary_spans
    local,cross,meta=E7.decode(c1,base_logits,base_spans,alt_logits,secondary_spans,chars,return_info=True)
    return {'v3_geom':v3,'unified_v1':unified,'c0_route':c0,'c1_route':c1,
            'e7_native':local,'e7_cross':cross,'route':route,
            'active_crops':len(base_spans),'extra_crops':len(alt_logits),
            'active_frames':sum(len(z) for z in base_logits),
            'extra_frames':sum(len(z) for z in alt_logits),
            'e7_native_candidates':meta['native_candidates'],
            'e7_native_accepted':meta['native_accepted'],
            'e7_cross_candidates':meta['cross_candidates'],
            'e7_cross_accepted':meta['cross_accepted'],
            'e7_cross_insertion_chars':meta['cross_insertion_chars'],
            'e7_cross_deleted_chars':meta['cross_deleted_chars'],
            'e7_cross_edits_path':meta['cross_edits'],
            'diagnostic_ms':round((time.perf_counter()-tic)*1000,3)}


def cpu_tests():
    import torch
    class Dummy(torch.nn.Module):
        def forward(self,xx):
            b,_,h,w=xx.shape
            t=max(4,w//4)
            y=torch.full((b,t,3),-6.,device=xx.device)
            y[:,:,0]=6.;y[:,max(1,t//3),1]=10.;y[:,max(2,t*2//3),2]=10.
            return y
    core=UNI.UnifiedRecognizer(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    for im in [Image.new('RGB',(18,210),'white'),Image.new('RGB',(120,30),'white'),Image.new('RGB',(330,20),'white')]:
        row=predict_image(im,core)
        assert isinstance(row['e7_cross'],str) and row['e7_cross']
        base,_=core.infer_batch([im])
        assert row['unified_v1']==LT.normalize_ltb(base[0]['pred']),row
        assert row['extra_crops']==row['active_crops']
    assert E7.cpu_tests()['passed']
    return {'passed':True,'routes':['orientation','dense','v3'],
            'baseline_parity':True,'extra_crop_alignment':True}

In [ ]:
import importlib,time,json,zipfile,io,base64,gzip,hashlib
import pandas as pd,numpy as np
sys.path.insert(0,str(WORK))
import longtext_final as LT,longtext_v3 as V3,orientation_probe as OP,density_probe as DP,unified_router as UNI
import alex_ctc as AX,alex_c1 as C1,space_ctc as SPACE,alex_e5 as E5
import alex_e7 as E7,alex_e7_experiment as EXP
for mod in (LT,V3,OP,DP,UNI,AX,C1,SPACE,E5,E7,EXP):importlib.reload(mod)
print('Base regression:',LT.run_regression_tests())
print('Geometry regression:',V3.run_regression_tests_v3()['passed'])
print('Density regression:',DP.run_cpu_tests())
print('Orientation regression:',OP.run_cpu_tests())
print('Unified regression:',UNI.run_cpu_tests())
print('CTC exact forward tests:',C1.cpu_tests())
print('CTC posterior beam tests:',E5.cpu_tests())
print('E7 segment decoder tests:',E7.cpu_tests())
print('E7 route+baseline parity tests:',EXP.cpu_tests())

## Logic-preserving fast SCOPE + per-stage timing
The two modules below use identical thresholds and edit selection to E7. The timing adapter also runs an **actual single-view SVTRv2-T** forward pass, rather than using Geometry v3 as a surrogate baseline. There are **no labels, historical predictions or test IDs** in the decoder.

A fixed-seed, GT-blind sample of 48 image IDs is evaluated through the original slower E7 decoder as an independent online parity check (the extra audit time is excluded from `scope_total_ms`). Full-dataset historical E7 output parity is checked **after inference**.


In [ ]:
%%writefile scope_e7_fast.py
"""SCOPE-CTC E7 exact-logic acceleration; no training and no GT.

Guarantee sought: bitwise identical final transcripts to E7.decode for the same
CTC crop logits. Deduplicate beam proposal generation and local CTC scoring.
No scale skipping or threshold modification in the parity-preserving mode.
"""
from __future__ import annotations
from collections import defaultdict
import numpy as np
import alex_e7 as LEG
import alex_ctc as AX
import alex_e5 as E5


def _key(p):
    return (p.start,p.end,p.new)


def _make_view(raw,span,group,scale,chars,ref_events):
    q=AX.normalized_probs(raw,chars)
    a,b=span
    ids=[j for j,e in enumerate(ref_events) if a<=e.x<=b]
    q=np.asarray(q,dtype=np.float64)
    if not ids or ids!=list(range(ids[0],ids[-1]+1)):
        frames=None
    else:
        tx=''.join(ref_events[i].char for i in ids)
        frames,_=E5.fast_forced_frames(q,tx)
    return {'logits':raw,'span':tuple(span),'group':int(group),'scale':str(scale),
            'q':q,'logq':np.log(np.maximum(q,1e-35)),'indices':ids,'frames':frames}


def _score(p,reference,views,bf_cache):
    groups=defaultdict(list)
    for v in views:
        k=(v['group'],v['scale'],p)
        if k not in bf_cache:
            bf_cache[k]=LEG.local_bf(v,p,reference)
        d=bf_cache[k]
        if d is not None and np.isfinite(d):
            groups[v['group']].append(float(d))
    if len(groups)<2:return None
    values=[float(np.median(g)) for g in groups.values()]
    support=sum(s>0 for s in values)
    if support<2:return None
    raw=float(np.median(values))
    return raw-(1.0+0.6*max(p.end-p.start,len(p.new)))


def _solve(scope,reference,ps,witnesses,views,nb,bf_cache):
    scores={}
    for p in ps:
        key=_key(p)
        if len({g for g,s in witnesses.get(key,set())})<2:continue
        score=_score(p,reference,views,bf_cache)
        if score is not None and score>0:scores[p]=score
    text,ops,gain=LEG.select_edits(reference,scores)
    info={'candidates':len(ps),'scored':len(scores),'accepted':len(ops),
          'insertion_chars':sum(len(p.new) for p in ops if p.start==p.end),
          'deleted_chars':sum(p.end-p.start for p in ops if not p.new),
          'beam_hypotheses':nb,'gain':float(gain),
          'edits':'|'.join(f'{p.start}:{p.end}:{p.new}' for p in ops)}
    return text,{scope+'_'+k:v for k,v in info.items()}


def decode(reference,base_logits,base_spans,extra_logits,extra_spans,chars,*,return_info=False):
    if len(base_logits)!=len(base_spans) or len(extra_logits)!=len(extra_spans):
        raise ValueError('Logits/spans mismatch')
    raw_ref=AX.reference_events(base_logits,base_spans,chars)
    events=LEG.mapped_events(reference,raw_ref)
    if not events or not reference:
        fields={'candidates':0,'scored':0,'accepted':0,'insertion_chars':0,
                'deleted_chars':0,'beam_hypotheses':0,'gain':0.,'edits':''}
        info={f'{s}_{k}':v for s in ('native','cross') for k,v in fields.items()}
        return (reference,reference,info) if return_info else reference
    base=[_make_view(q,sp,i,'base',chars,events) for i,(q,sp) in enumerate(zip(base_logits,base_spans))]
    group_map={tuple(sp):i for i,sp in enumerate(base_spans)}
    extra=[]
    for q,sp in zip(extra_logits,extra_spans):
        if tuple(sp) not in group_map:raise ValueError('Extra view span not matched to base crop')
        extra.append(_make_view(q,sp,group_map[tuple(sp)],'alt',chars,events))
    ps_base,wit_base,nb_base=LEG.generate_proposals(reference,events,base,chars)
    bf_cache={}
    native,info_native=_solve('native',reference,ps_base,wit_base,base,nb_base,bf_cache)
    if not extra:
        cross,info_cross=_solve('cross',reference,ps_base,wit_base,base,nb_base,bf_cache)
    else:
        ps_extra,wit_extra,nb_extra=LEG.generate_proposals(reference,events,extra,chars)
        all_candidates={_key(p):p for p in ps_base}
        for p in ps_extra:all_candidates[_key(p)]=p
        all_witnesses=defaultdict(set)
        for k,v in wit_base.items():all_witnesses[k].update(v)
        for k,v in wit_extra.items():all_witnesses[k].update(v)
        cross,info_cross=_solve('cross',reference,list(all_candidates.values()),
                                 all_witnesses,base+extra,nb_base+nb_extra,bf_cache)
    info={**info_native,**info_cross}
    return (native,cross,info) if return_info else cross


def cpu_tests():
    assert LEG.cpu_tests()['passed']
    chars=['blank','a','b','c']
    def logit(seq):
        arr=np.full((len(seq),4),-8.,np.float32)
        for i,j in enumerate(seq):arr[i,j]=8.
        return arr
    X=logit([1,0,2,0,3,0,1,0,2,0,3])
    for ref,base,spans,extra in [
        ('abcabc',[X,X],[(0,60),(0,60)],[]),
        ('', [np.tile(logit([0]),(9,1))],[(0,60)],[]),
    ]:
        old=LEG.decode(ref,base,spans,extra,spans if extra else [],chars,return_info=True)
        new=decode(ref,base,spans,extra,spans if extra else [],chars,return_info=True)
        assert old==new,(old,new)
    return {'passed':True,'no_math_or_threshold_changes':True}


In [ ]:
%%writefile scope_final_runtime.py
"""One-pass matched inference and CUDA-synchronised stage timing for SCOPE final.

Matched same SVTRv2-T checkpoint:
  SVTRv2-T raw full12 single-view CTC greedy (actual model pass),
  Geometry v3, Unified, ALEX C0/C1, SCOPE native and SCOPE cross-scale.
The SCOPE fast decoder reuses crop logits and has unchanged evidence rules.
"""
from __future__ import annotations
import time
from PIL import Image
import torch
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP
import unified_router as UNI
import alex_ctc as AX
import alex_c1 as C1
import alex_e7 as E7
import scope_e7_fast as FAST


def _sync(device):
    if 'cuda' in str(device) and torch.cuda.is_available():torch.cuda.synchronize()


def predict_image(image,recognizer,*,run_original=True,check_legacy=False,skip_structurally_impossible=True):
    """No GT, image ID, or archived prediction enters inference."""
    im=image.convert('RGB');chars=recognizer.chars;device=recognizer.base.device
    _sync(device);tic=time.perf_counter();st={}
    def timed(name,fn):
        _sync(device);a=time.perf_counter();value=fn();_sync(device)
        st[name]=st.get(name,0.)+1000*(time.perf_counter()-a)
        return value
    parts=LT.make_chunks(im,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
    native_spans=[(int(a),int(b)) for _,a,b in parts]
    native_logits=timed('main_forward_ms',lambda:recognizer.base._forward_grouped([p for p,_,_ in parts],max_ratio=12))
    native_e=[V3.ctc_events(z,chars,*sp) for z,sp in zip(native_logits,native_spans)]
    v3=LT.normalize_ltb(V3.stitch_evidence(native_e,native_spans,use_confidence=False,geometric_fallback=True))
    route=UNI.UnifiedRecognizer.classify(im.width,im.height,len(parts))
    base_logits=native_logits;base_spans=native_spans;base_parts=parts;unified=v3
    if route=='orientation':
        scores={}
        for angle in ('cw','ccw'):
            rotated=OP.rotated(im,angle)
            p=LT.make_chunks(rotated,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
            spans=[(int(a),int(b)) for _,a,b in p]
            lg=timed('orientation_forward_ms',lambda:recognizer.base._forward_grouped([c for c,_,_ in p],max_ratio=12))
            ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(lg,spans)]
            pred=LT.normalize_ltb(V3.stitch_evidence(ev,spans,use_confidence=False,geometric_fallback=True))
            scores[angle]={'pred':pred,'agreement':OP.overlap_agreement(ev,spans),
                           'confidence':OP.event_confidence(ev),'logits':lg,'spans':spans,'parts':p}
        _,winner=OP.choose_orientation(scores['cw'],scores['ccw'],kind='overlap')
        base_logits=winner['logits'];base_spans=winner['spans'];base_parts=winner['parts'];unified=winner['pred']
    elif route=='dense':
        base_parts=DP.forced_crops(im,3)
        base_spans=[(int(a),int(b)) for _,a,b in base_parts]
        base_logits=timed('dense_forward_ms',lambda:recognizer.dense._forward_tensors([DP.resized_tensor(p,3) for p,_,_ in base_parts]))
        ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(base_logits,base_spans)]
        unified=LT.normalize_ltb(V3.stitch_evidence(ev,base_spans,use_confidence=False,geometric_fallback=True))
    c0=timed('c0_ms',lambda:AX.decode(base_logits,base_spans,chars))
    c1_sub,c1,c1_info=timed('c1_ms',lambda:C1.decode(base_logits,base_spans,chars,c0=c0,return_info=True))
    _sync(device);c1_end=time.perf_counter()
    st['matched_c1_ms']=1000*(c1_end-tic)
    if skip_structurally_impossible and len(base_spans)<2:
        alt_logits=[];alt_spans=[]
    else:
        alt_logits=timed('paired_scale_forward_ms',lambda:recognizer.dense._forward_tensors([DP.resized_tensor(p,2) for p,_,_ in base_parts]))
        alt_spans=[(int(a),int(b)) for _,a,b in base_parts]
        assert base_spans==alt_spans
    native,cross,info=timed('scope_decoder_ms',lambda:FAST.decode(c1,base_logits,base_spans,alt_logits,alt_spans,chars,return_info=True))
    if check_legacy:
        legacy_alt=alt_logits
        legacy_spans=alt_spans
        if len(base_spans)<2 and skip_structurally_impossible:
            # Old SCOPE was forced to run an unnecessary second scale here.
            # Its output is provably unaltered: no two spatial crop groups.
            assert native==c1 and cross==c1
        else:
            a,b,old=timed('parity_legacy_overhead_ms',lambda:E7.decode(c1,base_logits,base_spans,legacy_alt,legacy_spans,chars,return_info=True))
            assert (a,b)==(native,cross),f'Fast decoder changed text: {(a,b)!r} vs {(native,cross)!r}'
            for k in old:
                if k in info and k not in ('native_gain','cross_gain'):
                    assert old[k]==info[k],(k,old[k],info[k])
    _sync(device);scope_end=time.perf_counter()
    st['scope_total_ms']=1000*(scope_end-tic)-st.get('parity_legacy_overhead_ms',0.)
    # The direct original-model baseline uses the ORIGINAL image, no crop,
    # one MSR target_size_ltb(..., max_ratio=12), and CTC greedy.
    if run_original:
        if len(native_logits)==1 and native_spans==[(0,im.width)]:
            raw_logits=native_logits[0]
            st['original_extra_forward_ms']=0.
            st['original_reused_from_native']=1
        else:
            raw_logits=timed('original_extra_forward_ms',lambda:recognizer.base._forward_grouped([im],max_ratio=12))[0]
            st['original_reused_from_native']=0
        original=LT.normalize_ltb(LT.greedy_ctc(raw_logits,chars))
    else:
        original='';st['original_extra_forward_ms']=float('nan')
        st['original_reused_from_native']=0
    st.setdefault('paired_scale_forward_ms',0.)
    st.setdefault('orientation_forward_ms',0.)
    st.setdefault('dense_forward_ms',0.)
    st['num_primary_forward_crops']=len(native_logits)+ (6 if route=='orientation' else 3 if route=='dense' else 0)
    return {
        'original_svtrv2_full12':original,
        'v3_geom':v3,'unified_v1':unified,'c0_route':c0,'c1_route':c1,
        'e7_native':native,'e7_cross':cross,'route':route,
        'active_crops':len(base_spans),'extra_crops':len(alt_logits),
        'active_frames':sum(len(q) for q in base_logits),
        'extra_frames':sum(len(q) for q in alt_logits),
        'e7_native_candidates':info['native_candidates'],
        'e7_native_accepted':info['native_accepted'],
        'e7_cross_candidates':info['cross_candidates'],
        'e7_cross_accepted':info['cross_accepted'],
        'e7_cross_edits_path':info['cross_edits'],
        'e7_cross_insertion_chars':info['cross_insertion_chars'],
        'e7_cross_deleted_chars':info['cross_deleted_chars'],
        'e7_cross_gain':info['cross_gain'],
        'diagnostic_ms':round(st['scope_total_ms'],3),
        **st
    }


def cpu_tests():
    class Dummy(torch.nn.Module):
        def forward(self,xx):
            b,_,h,w=xx.shape;t=max(4,w//4)
            y=torch.full((b,t,3),-6.,device=xx.device)
            y[:,:,0]=6.;y[:,max(1,t//3),1]=10.;y[:,max(2,t*2//3),2]=10.
            return y
    core=UNI.UnifiedRecognizer(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    old=__import__('alex_e7_experiment')
    for im in [Image.new('RGB',(18,210),'white'),Image.new('RGB',(120,30),'white'),Image.new('RGB',(330,20),'white')]:
        new=predict_image(im,core,check_legacy=True)
        base=old.predict_image(im,core)
        assert new['original_svtrv2_full12']==LT.normalize_ltb(core.base.predict(im,'full12'))
        for k in ('v3_geom','unified_v1','c0_route','c1_route','e7_native','e7_cross','route'):
            assert new[k]==base[k],(k,new[k],base[k])
    return {'passed':True,'three_routes':True,'original_full12':True,'fast_legacy_prediction_parity':True}


In [ ]:
import importlib
import scope_e7_fast as FAST,scope_final_runtime as FINAL
importlib.reload(FAST)
importlib.reload(FINAL)
print('SCOPE fast CPU tests:',FAST.cpu_tests())
print('Three-route direct SVTR + C1 + SCOPE parity:',FINAL.cpu_tests())
assert FAST.cpu_tests()['passed'] and FINAL.cpu_tests()['passed']


## Load frozen SVTRv2-T and Long Text Benchmark

Official OpenOCR revision, configuration SHA256, checkpoint SHA256, and full dataset manifest are checked exactly as in E5.

In [ ]:
# 3. Fetch official T assets (unless path overrides supplied)

def fetch_gdrive(fid,out,min_bytes):
    out=Path(out);out.parent.mkdir(exist_ok=True,parents=True)
    if out.is_file() and out.stat().st_size>=min_bytes:
        return out
    if out.exists():out.unlink()
    try:gdown.download(id=fid,output=str(out),quiet=False)
    except Exception as exc:raise RuntimeError(f'Could not download official asset {fid}: {exc}') from exc
    if not out.is_file() or out.stat().st_size<min_bytes:
        raise RuntimeError(f'Google Drive download failed or incomplete ({out}); set the matching local override.')
    return out
CKPT=Path(CKPT_OVERRIDE) if CKPT_OVERRIDE else fetch_gdrive(CKPT_ID,WORK/'longstr_svtr_t'/'best.pth',5_000_000)
CFG=Path(CFG_OVERRIDE) if CFG_OVERRIDE else fetch_gdrive(CFG_ID,WORK/'longstr_svtr_t'/'config.yml',1000)
assert CKPT.is_file() and CFG.is_file()
net,chars=LT.load_official_svtr(REPO,CFG,CKPT,DEVICE)
recognizer=UNI.UnifiedRecognizer(net,chars,device=DEVICE,batch_size=IMAGE_BATCH)

def sha256(file):
    h=hashlib.sha256()
    with open(file,'rb') as f:
        for buf in iter(lambda:f.read(1<<20),b''):h.update(buf)
    return h.hexdigest()
provenance={'OpenOCR_revision':OPENOCR_REV,'checkpoint_sha256':sha256(CKPT),'config_sha256':sha256(CFG),
            'classes':len(chars),'model':'SVTRv2-T','source_stitch':'LTB.ipynb G2', 'experiment':'ALEX-CTC E1 full-LTB frozen CTC posterior inference',
            'source_preproc':'LTB.ipynb cell 12',
            'legacy_window_ratio':LT.LEGACY_CHUNK_AR,'legacy_overlap':LT.LEGACY_OVERLAP,
            'test_protocol':'Exploratory reused LTB; all candidates predeclared, no candidate selected from test for independent claim'}
(OUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
print('PASS strict checkpoint + CTC logits parity; vocab size',len(chars))

if provenance['checkpoint_sha256']!='684f23bc3a2b1f724cadfd275e1a185456dd51b5e56c8830780d1998e90f77f6' or provenance['config_sha256']!='f1ab35a0f5e44d3c24c940cdf1a07404068ad9191bad9fe232ad4fcf38bbbe29':
    raise RuntimeError('Reference official checkpoint/config hash mismatch: no matched comparison')

In [ ]:
# 4. LTB auto-fetch with strict manifest validation; no data used for train.
import tarfile
if LTB_OVERRIDE:
    LTB_ROOT=Path(LTB_OVERRIDE)
else:
    archive=Path(hf_hub_download(repo_id=HF_DATA,filename='ltb.tar.xz',repo_type='dataset',revision='d59c13643f1d7f92831625fe161e2c19ec3d7f52'))
    LTB_ROOT=WORK/'longstr_ltb'
    marker=LTB_ROOT/'extract_complete.json'
    if not marker.exists():
        LTB_ROOT.mkdir(parents=True,exist_ok=True)
        with tarfile.open(archive,'r:xz') as tar:
            mm=tar.getmembers()
            for m in mm:
                target=(LTB_ROOT/m.name).resolve()
                if not (target == LTB_ROOT.resolve() or LTB_ROOT.resolve() in target.parents):
                    raise RuntimeError(f'Unsafe archive path {m.name}')
                if not (m.isfile() or m.isdir()):
                    raise RuntimeError(f'Unsupported archive member {m.name}')
            try:tar.extractall(LTB_ROOT,members=mm,filter='data')
            except TypeError:tar.extractall(LTB_ROOT,members=mm)
        marker.write_text(json.dumps({'archive':str(archive),'n':len(mm)}))
manifest=list(LTB_ROOT.rglob('ultra_long_25_list.txt'))
if len(manifest)!=1:raise RuntimeError(f'Expected one ultra_long_25_list.txt, got {manifest}')
labels=manifest[0].read_text(encoding='utf-8-sig').splitlines()
items=[]
for i,line in enumerate(labels):
    if not line.strip():continue
    fields=line.split('\t',1)
    if len(fields)!=2:raise RuntimeError(f'Manifest bad row: {line[:100]!r}')
    rel,gt=fields
    rel=rel.strip().strip('"');q=Path(rel)
    candidates=[manifest[0].parent/q,LTB_ROOT/q,manifest[0].parent.parent/q,
                manifest[0].parent/'img'/q.name,LTB_ROOT/'img'/q.name]
    img_path=next((v.resolve() for v in candidates if v.is_file()),None)
    if img_path is None: raise FileNotFoundError(rel)
    items.append({'id':len(items),'path':str(img_path),'gt':gt})
assert len(items)==4789, f'LTB expected 4789, received {len(items)}'
assert len({x['path'] for x in items})==len(items), 'LTB contains duplicate image paths: cannot run paired benchmark'
assert len({x['id'] for x in items})==len(items), 'LTB has duplicate sample IDs'
counts={b:sum(('26-35' if len(LT.normalize_ltb(x['gt']))<=35 else
               '36-55' if len(LT.normalize_ltb(x['gt']))<=55 else '56+')==b for x in items)
        for b in ('26-35','36-55','56+')}
assert counts=={'26-35':3376,'36-55':1147,'56+':266},counts

print('Full LTB ready:',len(items),'buckets:',counts)

In [ ]:
RUN_ITEMS=items if MAX_IMAGES==0 else items[:int(MAX_IMAGES)]
assert len(RUN_ITEMS)>0
if MAX_IMAGES:print('CAUTION: Subset pilot, NOT full LTB:',len(RUN_ITEMS))
else:print('FULL evaluation cohort:',len(RUN_ITEMS))

In [ ]:
%%writefile scope_final_runtime.py
"""One-pass matched inference and CUDA-synchronised stage timing for SCOPE final.

Matched same SVTRv2-T checkpoint:
  SVTRv2-T raw full12 single-view CTC greedy (actual model pass),
  Geometry v3, Unified, ALEX C0/C1, SCOPE native and SCOPE cross-scale.
The SCOPE fast decoder reuses crop logits and has unchanged evidence rules.
"""
from __future__ import annotations
import time
from PIL import Image
import torch
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP
import unified_router as UNI
import alex_ctc as AX
import alex_c1 as C1
import alex_e7 as E7
import scope_e7_fast as FAST


def _sync(device):
    if 'cuda' in str(device) and torch.cuda.is_available():torch.cuda.synchronize()


def predict_image(image,recognizer,*,run_original=True,check_legacy=False,skip_structurally_impossible=True,native_logits_override=None,native_forward_ms_override=0.):
    """No GT, image ID, or archived prediction enters inference."""
    im=image.convert('RGB');chars=recognizer.chars;device=recognizer.base.device
    _sync(device);tic=time.perf_counter();st={}
    def timed(name,fn):
        _sync(device);a=time.perf_counter();value=fn();_sync(device)
        st[name]=st.get(name,0.)+1000*(time.perf_counter()-a)
        return value
    parts=LT.make_chunks(im,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
    native_spans=[(int(a),int(b)) for _,a,b in parts]
    native_logits=(timed('main_forward_ms',lambda:recognizer.base._forward_grouped([p for p,_,_ in parts],max_ratio=12)) if native_logits_override is None else native_logits_override)
    if native_logits_override is not None:
        assert len(native_logits)==len(parts)==1, 'Only native single-crop cache supported'
        st['main_forward_ms']=float(native_forward_ms_override)
    native_e=[V3.ctc_events(z,chars,*sp) for z,sp in zip(native_logits,native_spans)]
    v3=LT.normalize_ltb(V3.stitch_evidence(native_e,native_spans,use_confidence=False,geometric_fallback=True))
    route=UNI.UnifiedRecognizer.classify(im.width,im.height,len(parts))
    base_logits=native_logits;base_spans=native_spans;base_parts=parts;unified=v3
    if route=='orientation':
        scores={}
        for angle in ('cw','ccw'):
            rotated=OP.rotated(im,angle)
            p=LT.make_chunks(rotated,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
            spans=[(int(a),int(b)) for _,a,b in p]
            lg=timed('orientation_forward_ms',lambda:recognizer.base._forward_grouped([c for c,_,_ in p],max_ratio=12))
            ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(lg,spans)]
            pred=LT.normalize_ltb(V3.stitch_evidence(ev,spans,use_confidence=False,geometric_fallback=True))
            scores[angle]={'pred':pred,'agreement':OP.overlap_agreement(ev,spans),
                           'confidence':OP.event_confidence(ev),'logits':lg,'spans':spans,'parts':p}
        _,winner=OP.choose_orientation(scores['cw'],scores['ccw'],kind='overlap')
        base_logits=winner['logits'];base_spans=winner['spans'];base_parts=winner['parts'];unified=winner['pred']
    elif route=='dense':
        base_parts=DP.forced_crops(im,3)
        base_spans=[(int(a),int(b)) for _,a,b in base_parts]
        base_logits=timed('dense_forward_ms',lambda:recognizer.dense._forward_tensors([DP.resized_tensor(p,3) for p,_,_ in base_parts]))
        ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(base_logits,base_spans)]
        unified=LT.normalize_ltb(V3.stitch_evidence(ev,base_spans,use_confidence=False,geometric_fallback=True))
    c0=timed('c0_ms',lambda:AX.decode(base_logits,base_spans,chars))
    c1_sub,c1,c1_info=timed('c1_ms',lambda:C1.decode(base_logits,base_spans,chars,c0=c0,return_info=True))
    _sync(device);c1_end=time.perf_counter()
    st['matched_c1_ms']=1000*(c1_end-tic)
    if skip_structurally_impossible and len(base_spans)<2:
        alt_logits=[];alt_spans=[]
    else:
        alt_logits=timed('paired_scale_forward_ms',lambda:recognizer.dense._forward_tensors([DP.resized_tensor(p,2) for p,_,_ in base_parts]))
        alt_spans=[(int(a),int(b)) for _,a,b in base_parts]
        assert base_spans==alt_spans
    native,cross,info=timed('scope_decoder_ms',lambda:FAST.decode(c1,base_logits,base_spans,alt_logits,alt_spans,chars,return_info=True))
    if check_legacy:
        legacy_alt=alt_logits
        legacy_spans=alt_spans
        if len(base_spans)<2 and skip_structurally_impossible:
            # Old SCOPE was forced to run an unnecessary second scale here.
            # Its output is provably unaltered: no two spatial crop groups.
            assert native==c1 and cross==c1
        else:
            a,b,old=timed('parity_legacy_overhead_ms',lambda:E7.decode(c1,base_logits,base_spans,legacy_alt,legacy_spans,chars,return_info=True))
            assert (a,b)==(native,cross),f'Fast decoder changed text: {(a,b)!r} vs {(native,cross)!r}'
            for k in old:
                if k in info and k not in ('native_gain','cross_gain'):
                    assert old[k]==info[k],(k,old[k],info[k])
    _sync(device);scope_end=time.perf_counter()
    st['scope_total_ms']=1000*(scope_end-tic)-st.get('parity_legacy_overhead_ms',0.)
    # The direct original-model baseline uses the ORIGINAL image, no crop,
    # one MSR target_size_ltb(..., max_ratio=12), and CTC greedy.
    if run_original:
        if len(native_logits)==1 and native_spans==[(0,im.width)]:
            raw_logits=native_logits[0]
            st['original_extra_forward_ms']=0.
            st['original_reused_from_native']=1
        else:
            raw_logits=timed('original_extra_forward_ms',lambda:recognizer.base._forward_grouped([im],max_ratio=12))[0]
            st['original_reused_from_native']=0
        original=LT.normalize_ltb(LT.greedy_ctc(raw_logits,chars))
    else:
        original='';st['original_extra_forward_ms']=float('nan')
        st['original_reused_from_native']=0
    st.setdefault('paired_scale_forward_ms',0.)
    st.setdefault('orientation_forward_ms',0.)
    st.setdefault('dense_forward_ms',0.)
    st['num_primary_forward_crops']=len(native_logits)+ (6 if route=='orientation' else 3 if route=='dense' else 0)
    return {
        'original_svtrv2_full12':original,
        'v3_geom':v3,'unified_v1':unified,'c0_route':c0,'c1_route':c1,
        'e7_native':native,'e7_cross':cross,'route':route,
        'active_crops':len(base_spans),'extra_crops':len(alt_logits),
        'active_frames':sum(len(q) for q in base_logits),
        'extra_frames':sum(len(q) for q in alt_logits),
        'e7_native_candidates':info['native_candidates'],
        'e7_native_accepted':info['native_accepted'],
        'e7_cross_candidates':info['cross_candidates'],
        'e7_cross_accepted':info['cross_accepted'],
        'e7_cross_edits_path':info['cross_edits'],
        'e7_cross_insertion_chars':info['cross_insertion_chars'],
        'e7_cross_deleted_chars':info['cross_deleted_chars'],
        'e7_cross_gain':info['cross_gain'],
        'diagnostic_ms':round(st['scope_total_ms'],3),
        **st
    }


def cpu_tests():
    class Dummy(torch.nn.Module):
        def forward(self,xx):
            b,_,h,w=xx.shape;t=max(4,w//4)
            y=torch.full((b,t,3),-6.,device=xx.device)
            y[:,:,0]=6.;y[:,max(1,t//3),1]=10.;y[:,max(2,t*2//3),2]=10.
            return y
    core=UNI.UnifiedRecognizer(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    old=__import__('alex_e7_experiment')
    for im in [Image.new('RGB',(18,210),'white'),Image.new('RGB',(120,30),'white'),Image.new('RGB',(330,20),'white')]:
        new=predict_image(im,core,check_legacy=True)
        base=old.predict_image(im,core)
        assert new['original_svtrv2_full12']==LT.normalize_ltb(core.base.predict(im,'full12'))
        for k in ('v3_geom','unified_v1','c0_route','c1_route','e7_native','e7_cross','route'):
            assert new[k]==base[k],(k,new[k],base[k])
    return {'passed':True,'three_routes':True,'original_full12':True,'fast_legacy_prediction_parity':True}


In [ ]:
%%writefile scope_guarded_runtime.py
"""SAFE-SCOPE v2: GT-blind structural protection + original SCOPE for eligible images.

This is a research-development routing policy. The 16-character guard was
selected AFTER exploring the existing LTB/external evaluation set and must be
considered tuned to that development evidence. A separate untouched test set
is required for confirmatory generalization or publication claims.

The original FULL SCOPE decoder, weight, CTC score and alignment are unchanged.
The change is ONLY early exit for structurally short single-crop images.
"""
from __future__ import annotations
import time
import numpy as np
import torch
from PIL import Image
import longtext_final as LT
import scope_final_runtime as FULL

POLICY_VERSION='SAFE_SCOPE_V2_STRUCTURAL_GUARD_v1'
DENSE_MIN_RAW_CHARS=16

def _sync(device):
    if 'cuda' in str(device) and torch.cuda.is_available():
        torch.cuda.synchronize()

def classify_geometry(image):
    im=image.convert('RGB')
    parts=LT.make_chunks(im,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
    n=len(parts)
    if n>1:
        return 'RUN_MULTI_CROP',n,None
    if im.width<=im.height:
        # CW/CCW share the SAME geometry. No GPU pass is needed to detect
        # whether rotation will produce at least two overlapping crop groups.
        shape_only=Image.new('RGB',(im.height,im.width))
        n_rot=len(LT.make_chunks(shape_only,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP))
        return ('RUN_ROTATED_MULTI_CROP' if n_rot>=2 else 'PRESERVE_SINGLE_ROTATED_CROP'),n,n_rot
    return 'NEED_RAW_CTC_LOAD',n,None

def predict_image(image,recognizer,*,run_original=True,check_legacy=False,skip_structurally_impossible=True):
    if not run_original:
        raise ValueError('Safe SCOPE requires matched original SVTRv2-T inference')
    im=image.convert('RGB')
    device=recognizer.base.device
    _sync(device);st=time.perf_counter()
    geom,n_native,n_rot=classify_geometry(im)
    native_override=None
    native_ms=0.
    raw=None
    if geom=='NEED_RAW_CTC_LOAD':
        _sync(device);t0=time.perf_counter()
        native_override=recognizer.base._forward_grouped([im],max_ratio=12)
        _sync(device);native_ms=1000*(time.perf_counter()-t0)
        assert len(native_override)==1
        raw=LT.normalize_ltb(LT.greedy_ctc(native_override[0],recognizer.chars))
        if len(raw)>=DENSE_MIN_RAW_CHARS:
            geom='RUN_DENSE_HIGH_TEXT_LOAD'
        else:
            geom='PRESERVE_SHORT_DENSE'
    active=geom.startswith('RUN_')
    if active:
        opts={'run_original':True,'check_legacy':check_legacy,
              'skip_structurally_impossible':skip_structurally_impossible}
        if native_override is not None:
            opts['native_logits_override']=native_override
            opts['native_forward_ms_override']=native_ms
        result=FULL.predict_image(im,recognizer,**opts)
        assert isinstance(result['original_svtrv2_full12'],str)
        if raw is not None:
            assert raw==result['original_svtrv2_full12'], 'Native logits cache invalidated raw baseline'
        result['safe_scope']=result['e7_cross']
        result['safe_scope_total_ms']=float(result['scope_total_ms'])+native_ms
        result['safe_scope_active']=1
        result['safe_scope_original_fallback']=0
        result['safe_scope_base_compared']=1
    else:
        if native_override is None:
            _sync(device);a=time.perf_counter()
            native_override=recognizer.base._forward_grouped([im],max_ratio=12)
            _sync(device);native_ms=1000*(time.perf_counter()-a)
            raw=LT.normalize_ltb(LT.greedy_ctc(native_override[0],recognizer.chars))
        assert isinstance(raw,str)
        result={
            'original_svtrv2_full12':raw,'safe_scope':raw,
            'v3_geom':raw,
            'route':'orientation' if im.width<=im.height else 'dense',
            'active_crops':1,'extra_crops':0,
            'active_frames':len(native_override[0]),'extra_frames':0,
            'main_forward_ms':native_ms,'dense_forward_ms':0.,
            'orientation_forward_ms':0.,'paired_scale_forward_ms':0.,
            'scope_decoder_ms':0.,'matched_c1_ms':float('nan'),
            'scope_total_ms':native_ms,'safe_scope_total_ms':native_ms,
            'original_extra_forward_ms':0.,'original_reused_from_native':1,
            'num_primary_forward_crops':1,'safe_scope_active':0,
            'safe_scope_original_fallback':1,'safe_scope_base_compared':1,
        }
        # Do NOT invent C0/C1/legacy E7 outputs for skipped images.
        # Only original_svtrv2_full12 and safe_scope are scored in this notebook.
    result['safe_guard_decision']=geom
    result['safe_guard_native_crop_count']=int(n_native)
    result['safe_guard_rotated_crop_count']=(-1 if n_rot is None else int(n_rot))
    result['safe_guard_raw_ctc_chars']=(-1 if raw is None else int(len(raw)))
    result['safe_guard_min_ctc_chars']=DENSE_MIN_RAW_CHARS
    result['safe_guard_policy_version']=POLICY_VERSION
    return result


def cpu_tests():
    from PIL import Image
    import unified_router as UNI
    class Dummy(torch.nn.Module):
        def forward(self,xx):
            b,_,h,w=xx.shape
            t=max(4,w//4)
            y=torch.full((b,t,3),-6.,device=xx.device)
            y[:,:,0]=6.
            y[:,max(1,t//3),1]=10.
            y[:,max(2,t*2//3),2]=10.
            return y
    core=UNI.UnifiedRecognizer(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    cases=[(Image.new('RGB',(120,30),'white'),'PRESERVE_SHORT_DENSE'),
           (Image.new('RGB',(20,120),'white'),'PRESERVE_SINGLE_ROTATED_CROP'),
           (Image.new('RGB',(330,20),'white'),'RUN_MULTI_CROP'),
           (Image.new('RGB',(18,210),'white'),'RUN_ROTATED_MULTI_CROP')]
    for im,want in cases:
        d,n,_=classify_geometry(im)
        assert d==('NEED_RAW_CTC_LOAD' if want=='PRESERVE_SHORT_DENSE' else want),(d,want,n)
        out=predict_image(im,core)
        assert out['safe_guard_decision']==want,(out,want)
        assert out['safe_scope']==out['original_svtrv2_full12'] if not d.startswith('RUN_') else isinstance(out['safe_scope'],str)
        if not d.startswith('RUN_'):
            assert out['extra_crops']==0
            assert out['safe_scope_active']==0
        else:assert out['safe_scope_active']==1
    return {'passed':True,'two_fallback_routes':True,'two_active_routes':True,'original_backbone_preserved':True}


In [ ]:
import importlib
import scope_final_runtime as FINAL, scope_guarded_runtime as GUARD
importlib.reload(FINAL)
importlib.reload(GUARD)
assert GUARD.cpu_tests()['passed']
print('SAFE GUARD CPU TESTS: PASS',GUARD.POLICY_VERSION)
print('DENSE RAW CHARACTER RULE:',GUARD.DENSE_MIN_RAW_CHARS)


## Final frozen method: evidence-based crop coverage probe + JIT CTC kernels

The native SVTRv2-T transcript is always computed as the control. Single-crop images are **not automatically classified as short when CTC predicts few characters**. For one-crop horizontal images with raw text under 16 characters, generate the **same three ×3 primary crop observations** as the full SCOPE candidate. Compare the stitched crop transcript length with the native transcript; activate SCOPE only if the additional observed coverage is at least `max(3, ceil(0.5*len(native)))`. For one-crop vertical images, probe both orientations with the original orientation selector. These are GT-blind, development-chosen evidence parameters; they are not a universal theoretical guarantee.

When activated, the primary probe logits are **reused without a second GPU forward**. When not activated, the **exact original SVTRv2-T transcript** is returned. Unlike SAFE v2, the only proof of "short" is no longer a short raw transcript. JIT kernels accelerate CTC forward and forced Viterbi alignment only after numerical parity tests; changing GT or any dataset-level signal is forbidden during inference.

Latency includes the probe even if it rejects SCOPE. This is the chosen accuracy/latency trade-off: some additional GPU work is deliberately spent on ordinary single-crop words to prevent catastrophic undercoverage of long words.

In [ ]:
%%writefile scope_final_runtime_v3.py
"""One-pass matched inference and CUDA-synchronised stage timing for SCOPE final.

Matched same SVTRv2-T checkpoint:
  SVTRv2-T raw full12 single-view CTC greedy (actual model pass),
  Geometry v3, Unified, ALEX C0/C1, SCOPE native and SCOPE cross-scale.
The SCOPE fast decoder reuses crop logits and has unchanged evidence rules.
"""
from __future__ import annotations
import time
from PIL import Image
import torch
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP
import unified_router as UNI
import alex_ctc as AX
import alex_c1 as C1
import alex_e7 as E7
import scope_e7_fast as FAST


def _sync(device):
    if 'cuda' in str(device) and torch.cuda.is_available():torch.cuda.synchronize()


def predict_image(image,recognizer,*,run_original=True,check_legacy=False,skip_structurally_impossible=True,native_logits_override=None,native_forward_ms_override=0.,dense_logits_override=None,dense_forward_ms_override=0.,orientation_logits_override=None,orientation_forward_ms_override=0.):
    """No GT, image ID, or archived prediction enters inference."""
    im=image.convert('RGB');chars=recognizer.chars;device=recognizer.base.device
    _sync(device);tic=time.perf_counter();st={}
    def timed(name,fn):
        _sync(device);a=time.perf_counter();value=fn();_sync(device)
        st[name]=st.get(name,0.)+1000*(time.perf_counter()-a)
        return value
    parts=LT.make_chunks(im,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
    native_spans=[(int(a),int(b)) for _,a,b in parts]
    native_logits=(timed('main_forward_ms',lambda:recognizer.base._forward_grouped([p for p,_,_ in parts],max_ratio=12)) if native_logits_override is None else native_logits_override)
    if native_logits_override is not None:
        assert len(native_logits)==len(parts)==1, 'Only native single-crop cache supported'
        st['main_forward_ms']=float(native_forward_ms_override)
    native_e=[V3.ctc_events(z,chars,*sp) for z,sp in zip(native_logits,native_spans)]
    v3=LT.normalize_ltb(V3.stitch_evidence(native_e,native_spans,use_confidence=False,geometric_fallback=True))
    route=UNI.UnifiedRecognizer.classify(im.width,im.height,len(parts))
    base_logits=native_logits;base_spans=native_spans;base_parts=parts;unified=v3
    if route=='orientation':
        scores={}
        for angle in ('cw','ccw'):
            rotated=OP.rotated(im,angle)
            p=LT.make_chunks(rotated,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
            spans=[(int(a),int(b)) for _,a,b in p]
            if orientation_logits_override is not None:
                lg=orientation_logits_override[angle]
                assert len(lg)==len(p),(angle,len(lg),len(p))
                st['orientation_forward_ms']=float(orientation_forward_ms_override)
            else:
                lg=timed('orientation_forward_ms',lambda:recognizer.base._forward_grouped([c for c,_,_ in p],max_ratio=12))
            ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(lg,spans)]
            pred=LT.normalize_ltb(V3.stitch_evidence(ev,spans,use_confidence=False,geometric_fallback=True))
            scores[angle]={'pred':pred,'agreement':OP.overlap_agreement(ev,spans),
                           'confidence':OP.event_confidence(ev),'logits':lg,'spans':spans,'parts':p}
        _,winner=OP.choose_orientation(scores['cw'],scores['ccw'],kind='overlap')
        base_logits=winner['logits'];base_spans=winner['spans'];base_parts=winner['parts'];unified=winner['pred']
    elif route=='dense':
        base_parts=DP.forced_crops(im,3)
        base_spans=[(int(a),int(b)) for _,a,b in base_parts]
        if dense_logits_override is not None:
            base_logits=dense_logits_override
            assert len(base_logits)==len(base_parts),(len(base_logits),len(base_parts))
            st['dense_forward_ms']=float(dense_forward_ms_override)
        else:
            base_logits=timed('dense_forward_ms',lambda:recognizer.dense._forward_tensors([DP.resized_tensor(p,3) for p,_,_ in base_parts]))
        ev=[V3.ctc_events(q,chars,*sp) for q,sp in zip(base_logits,base_spans)]
        unified=LT.normalize_ltb(V3.stitch_evidence(ev,base_spans,use_confidence=False,geometric_fallback=True))
    c0=timed('c0_ms',lambda:AX.decode(base_logits,base_spans,chars))
    c1_sub,c1,c1_info=timed('c1_ms',lambda:C1.decode(base_logits,base_spans,chars,c0=c0,return_info=True))
    _sync(device);c1_end=time.perf_counter()
    st['matched_c1_ms']=1000*(c1_end-tic)
    if skip_structurally_impossible and len(base_spans)<2:
        alt_logits=[];alt_spans=[]
    else:
        alt_logits=timed('paired_scale_forward_ms',lambda:recognizer.dense._forward_tensors([DP.resized_tensor(p,2) for p,_,_ in base_parts]))
        alt_spans=[(int(a),int(b)) for _,a,b in base_parts]
        assert base_spans==alt_spans
    native,cross,info=timed('scope_decoder_ms',lambda:FAST.decode(c1,base_logits,base_spans,alt_logits,alt_spans,chars,return_info=True))
    if check_legacy:
        legacy_alt=alt_logits
        legacy_spans=alt_spans
        if len(base_spans)<2 and skip_structurally_impossible:
            # Old SCOPE was forced to run an unnecessary second scale here.
            # Its output is provably unaltered: no two spatial crop groups.
            assert native==c1 and cross==c1
        else:
            a,b,old=timed('parity_legacy_overhead_ms',lambda:E7.decode(c1,base_logits,base_spans,legacy_alt,legacy_spans,chars,return_info=True))
            assert (a,b)==(native,cross),f'Fast decoder changed text: {(a,b)!r} vs {(native,cross)!r}'
            for k in old:
                if k in info and k not in ('native_gain','cross_gain'):
                    assert old[k]==info[k],(k,old[k],info[k])
    _sync(device);scope_end=time.perf_counter()
    st['scope_total_ms']=1000*(scope_end-tic)-st.get('parity_legacy_overhead_ms',0.)
    # The direct original-model baseline uses the ORIGINAL image, no crop,
    # one MSR target_size_ltb(..., max_ratio=12), and CTC greedy.
    if run_original:
        if len(native_logits)==1 and native_spans==[(0,im.width)]:
            raw_logits=native_logits[0]
            st['original_extra_forward_ms']=0.
            st['original_reused_from_native']=1
        else:
            raw_logits=timed('original_extra_forward_ms',lambda:recognizer.base._forward_grouped([im],max_ratio=12))[0]
            st['original_reused_from_native']=0
        original=LT.normalize_ltb(LT.greedy_ctc(raw_logits,chars))
    else:
        original='';st['original_extra_forward_ms']=float('nan')
        st['original_reused_from_native']=0
    st.setdefault('paired_scale_forward_ms',0.)
    st.setdefault('orientation_forward_ms',0.)
    st.setdefault('dense_forward_ms',0.)
    st['num_primary_forward_crops']=len(native_logits)+ (6 if route=='orientation' else 3 if route=='dense' else 0)
    return {
        'original_svtrv2_full12':original,
        'v3_geom':v3,'unified_v1':unified,'c0_route':c0,'c1_route':c1,
        'e7_native':native,'e7_cross':cross,'route':route,
        'active_crops':len(base_spans),'extra_crops':len(alt_logits),
        'active_frames':sum(len(q) for q in base_logits),
        'extra_frames':sum(len(q) for q in alt_logits),
        'e7_native_candidates':info['native_candidates'],
        'e7_native_accepted':info['native_accepted'],
        'e7_cross_candidates':info['cross_candidates'],
        'e7_cross_accepted':info['cross_accepted'],
        'e7_cross_edits_path':info['cross_edits'],
        'e7_cross_insertion_chars':info['cross_insertion_chars'],
        'e7_cross_deleted_chars':info['cross_deleted_chars'],
        'e7_cross_gain':info['cross_gain'],
        'diagnostic_ms':round(st['scope_total_ms'],3),
        **st
    }


def cpu_tests():
    class Dummy(torch.nn.Module):
        def forward(self,xx):
            b,_,h,w=xx.shape;t=max(4,w//4)
            y=torch.full((b,t,3),-6.,device=xx.device)
            y[:,:,0]=6.;y[:,max(1,t//3),1]=10.;y[:,max(2,t*2//3),2]=10.
            return y
    core=UNI.UnifiedRecognizer(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    old=__import__('alex_e7_experiment')
    for im in [Image.new('RGB',(18,210),'white'),Image.new('RGB',(120,30),'white'),Image.new('RGB',(330,20),'white')]:
        new=predict_image(im,core,check_legacy=True)
        base=old.predict_image(im,core)
        assert new['original_svtrv2_full12']==LT.normalize_ltb(core.base.predict(im,'full12'))
        for k in ('v3_geom','unified_v1','c0_route','c1_route','e7_native','e7_cross','route'):
            assert new[k]==base[k],(k,new[k],base[k])
    return {'passed':True,'three_routes':True,'original_full12':True,'fast_legacy_prediction_parity':True}


In [ ]:
%%writefile scope_ctc_jit.py
"""Optional exact-scoring CTC forward JIT, preserving C1's recurrence.

No GT, learned weights, threshold changes, or modification of the allowed CTC paths.
Numba disabled/fallback if absent or its results fail a parity self-test.
"""
import numpy as np
import alex_c1 as C1
import alex_ctc as AX

_ORIGINAL_FORWARD = C1.ctc_forward_logprob
_NUMBA_FORWARD = None
_JIT_READY = False
_JIT_STATUS = 'not_installed'


def _make_kernel():
    from numba import njit

    @njit(cache=True)
    def forward_kernel(logq, labels):
        T, C = logq.shape
        n = len(labels)
        if n == 0:
            result = 0.0
            for t in range(T):
                result += logq[t, 0]
            return result
        repeats=0
        for i in range(1,n):
            if labels[i]==labels[i-1]:
                repeats += 1
        if n+repeats>T:
            return -np.inf
        S=2*n+1
        states=np.zeros(S,np.int32)
        for i in range(n):
            states[2*i+1]=labels[i]
        previous=np.empty(S,np.float64)
        for s in range(S):
            previous[s]=-np.inf
        previous[0]=logq[0,0]
        previous[1]=logq[0,labels[0]]
        current=np.empty(S,np.float64)
        for t in range(1,T):
            for s in range(S):
                stay=previous[s]
                step=previous[s-1] if s>0 else -np.inf
                jump=previous[s-2] if (s>1 and states[s]!=0 and states[s]!=states[s-2]) else -np.inf
                current[s]=np.logaddexp(np.logaddexp(stay,step),jump)+logq[t,states[s]]
            sw=previous;previous=current;current=sw
        return np.logaddexp(previous[S-1],previous[S-2])
    return forward_kernel


def _as_labels(text):
    return np.fromiter((1+AX.ALPH.index(x) for x in text),dtype=np.int32,count=len(text))


def _fast(logq,text):
    arr=np.asarray(logq,dtype=np.float64)
    if arr.ndim!=2 or arr.shape[1]!=37 or arr.shape[0]<1:
        raise ValueError('Expected [T,37] nonempty normalized log posterior')
    return float(_NUMBA_FORWARD(arr,_as_labels(text)))


def activate(strict=True):
    """Perform deterministic edge/random parity tests before patching C1 globally."""
    global _NUMBA_FORWARD,_JIT_READY,_JIT_STATUS
    if _JIT_READY:
        return {'available':True,'status':_JIT_STATUS}
    try:
        _NUMBA_FORWARD = _make_kernel()
        g=np.random.default_rng(20261121)
        seqs=['','a','ab','aaa','abba','abcdef','000000','abc1a','z9z9z9z9']
        for T in [1,2,3,4,7,15,32,64,120]:
            for trial in range(2):
                data=g.uniform(.001,1,size=(T,37))
                data=data/data.sum(axis=1,keepdims=True)
                logq=np.log(data)
                for txt in seqs:
                    old=_ORIGINAL_FORWARD(logq,txt)
                    new=_fast(logq,txt)
                    if not (np.isneginf(old) and np.isneginf(new)):
                        if not np.isclose(old,new,rtol=1e-11,atol=1e-10):
                            raise AssertionError((T,txt,old,new))
        # tie case and stable high blank probabilities
        z=np.full((25,37),-12.,np.float64)
        z[:,0]=0.
        for txt in ('','a','aa','aba'):
            old=_ORIGINAL_FORWARD(z,txt);new=_fast(z,txt)
            if not np.isclose(old,new,rtol=1e-11,atol=1e-10):
                raise AssertionError((txt,old,new))
        C1.ctc_forward_logprob=_fast
        _JIT_READY=True
        _JIT_STATUS='jit_exact_recurrence_parity_pass'
        return {'available':True,'status':_JIT_STATUS}
    except Exception as exc:
        C1.ctc_forward_logprob=_ORIGINAL_FORWARD
        _JIT_READY=False
        _JIT_STATUS=f'fallback_original_{type(exc).__name__}:{exc}'
        if strict:
            raise
        return {'available':False,'status':_JIT_STATUS}


def cpu_benchmark(n=80,T=64):
    import time
    g=np.random.default_rng(734)
    q=g.uniform(.001,1,size=(T,37));q=np.log(q/q.sum(axis=1,keepdims=True))
    s='correctreconstruction'
    t0=time.perf_counter()
    for _ in range(n):_ORIGINAL_FORWARD(q,s)
    t1=time.perf_counter()
    for _ in range(n):C1.ctc_forward_logprob(q,s)
    t2=time.perf_counter()
    return {'original_ms':1000*(t1-t0)/n,'patched_ms':1000*(t2-t1)/n,
            'ratio':max(1e-9,(t1-t0))/max(1e-9,(t2-t1))}

_ORIGINAL_FORCED = None
_FORCED_KERNEL = None


def _make_forced_kernel():
    from numba import njit

    @njit(cache=True)
    def kernel(q, labels):
        T, C=q.shape
        n=len(labels)
        if n==0:
            return np.empty(0,np.int32), 0.0, True
        repeats=0
        for k in range(1,n):
            if labels[k]==labels[k-1]:repeats+=1
        if n+repeats>T:
            return np.empty(0,np.int32),-np.inf,False
        S=2*n+1
        states=np.zeros(S,np.int32)
        for k in range(n):states[2*k+1]=labels[k]
        lq=np.log(np.maximum(q,1e-30))
        prev=np.empty(S,np.float64)
        curr=np.empty(S,np.float64)
        for s in range(S):prev[s]=-np.inf
        prev[0]=lq[0,0];prev[1]=lq[0,labels[0]]
        trace=np.empty((T,S),np.int16)
        trace[:,:]=-1
        for t in range(1,T):
            for s in range(S):
                stay=prev[s]
                step=prev[s-1] if s>=1 else -np.inf
                jump=prev[s-2] if (s>=2 and states[s]!=0 and states[s]!=states[s-2]) else -np.inf
                if stay>=step and stay>=jump:
                    curr[s]=stay+lq[t,states[s]];trace[t,s]=s
                elif step>=jump:
                    curr[s]=step+lq[t,states[s]];trace[t,s]=s-1
                else:
                    curr[s]=jump+lq[t,states[s]];trace[t,s]=s-2
            tmp=prev;prev=curr;curr=tmp
        terminal=S-1 if prev[S-1]>=prev[S-2] else S-2
        score=prev[terminal]
        if not np.isfinite(score):return np.empty(0,np.int32),-np.inf,False
        state_path=np.empty(T,np.int32)
        state=terminal
        for t in range(T-1,-1,-1):
            state_path[t]=state
            if t:state=int(trace[t,state])
        frames=np.empty(n,np.int32)
        for k in range(n):
            best=-1
            bestval=-1.0
            target=2*k+1
            for t in range(T):
                if state_path[t]==target:
                    v=q[t,labels[k]]
                    if best<0 or v>bestval:
                        best=t;bestval=v
            if best<0:return np.empty(0,np.int32),-np.inf,False
            frames[k]=best
        return frames,float(score),True
    return kernel


def activate_forced(strict=True):
    """Enable JIT forced alignment only after matching E5 on deterministic cases."""
    global _ORIGINAL_FORCED,_FORCED_KERNEL
    import alex_e5 as E5
    if _ORIGINAL_FORCED is not None and E5.fast_forced_frames is not _ORIGINAL_FORCED:
        return {'available':True,'status':'already_active'}
    _ORIGINAL_FORCED=E5.fast_forced_frames
    try:
        _FORCED_KERNEL=_make_forced_kernel()
        def fast_forced_frames(q, text):
            if not text:return [],0.0
            labels=_as_labels(text)
            frames,score,ok=_FORCED_KERNEL(np.asarray(q,dtype=np.float64),labels)
            if not ok:return None,-np.inf
            return [int(x) for x in frames],float(score)
        rg=np.random.default_rng(20261121)
        seqs=['','a','ab','aaa','abba','abcdef','0000','z9z9z9z9']
        for T in [2,3,4,8,16,32,70]:
            for seed in range(2):
                q=rg.uniform(.001,1,size=(T,37));q/=q.sum(axis=1,keepdims=True)
                for txt in seqs:
                    old=_ORIGINAL_FORCED(q,txt)
                    new=fast_forced_frames(q,txt)
                    if old[0] != new[0] or not (
                      (np.isneginf(old[1]) and np.isneginf(new[1])) or
                      np.isclose(old[1],new[1],rtol=1e-11,atol=1e-10)):
                        raise AssertionError((T,txt,old,new))
        E5.fast_forced_frames=fast_forced_frames
        return {'available':True,'status':'forced_alignment_parity_pass'}
    except Exception as e:
        E5.fast_forced_frames=_ORIGINAL_FORCED
        if strict:raise
        return {'available':False,'status':f'forced_alignment_fallback:{e}'}


def forced_benchmark(n=80,T=70):
    import time
    import alex_e5 as E5
    g=np.random.default_rng(734)
    q=g.uniform(.001,1,size=(T,37));q/=q.sum(axis=1,keepdims=True)
    txt='correctreconstruction'
    a=time.perf_counter()
    for _ in range(n):_ORIGINAL_FORCED(q,txt)
    b=time.perf_counter()
    for _ in range(n):E5.fast_forced_frames(q,txt)
    c=time.perf_counter()
    return {'original_ms':1000*(b-a)/n,'patched_ms':1000*(c-b)/n,'ratio':(b-a)/max(1e-9,c-b)}


In [ ]:
%%writefile scope_coverage_runtime.py
"""SCOPE-CTC final candidate: native baseline protection plus counterfactual crop-coverage probe.

Frozen SVTRv2-T only; decision uses pixels, CTC logits and predicted strings.
No GT, dataset name, image ID, historical correctness or test-set membership.

Design: long-text recall requires detecting grossly under-decoded native views.
For a one-crop image, use SAME primary route observations as full SCOPE to
compare the amount of supported crop text to native CTC text; if evidence is
insufficient, return the actual original-model prediction. If activated, reuse
probe logits in full SCOPE, preventing duplicate heavy primary forward passes.

The evidence rule below was designed after reviewing LTB/external benchmark
failures: it is DEVELOPMENT-TUNED, not independent validation or a published
universal guarantee. Inference never receives GT and never uses dataset IDs.
"""
from __future__ import annotations
import time
import math
import numpy as np
import torch
from PIL import Image
import longtext_final as LT
import longtext_v3 as V3
import orientation_probe as OP
import density_probe as DP
import scope_final_runtime_v3 as FULL

POLICY_VERSION='SCOPE_FINAL_COVERAGE_PROBE_V3_LOCKED_20261121'
RAW_DIRECT_DENSE_TRIGGER=16
COVERAGE_EXTRA_MIN=3
COVERAGE_RELATIVE_GAIN=0.5


def _sync(device):
    if 'cuda' in str(device) and torch.cuda.is_available():torch.cuda.synchronize()


def _native_crop_count(image):
    return len(LT.make_chunks(image,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP))


def _evidence_threshold(raw):
    return max(COVERAGE_EXTRA_MIN,int(math.ceil(COVERAGE_RELATIVE_GAIN*len(raw))))


def _coverage_witness(raw,probe):
    """Raw vs multi-view text evidence; no truth oracle or prediction confidence label."""
    if not isinstance(raw,str) or not isinstance(probe,str):raise ValueError('Expected transcript strings')
    gain=len(probe)-len(raw)
    return bool(gain>=_evidence_threshold(raw)),int(gain)


def _pred_from_crops(logits,spans,chars):
    if not logits:return ''
    events=[V3.ctc_events(x,chars,a,b) for x,(a,b) in zip(logits,spans)]
    return LT.normalize_ltb(V3.stitch_evidence(events,spans,use_confidence=False,geometric_fallback=True))


def _orientation_probe(im,recognizer):
    """Reuse exactly the two orientation probe forwards/full candidate selector."""
    scores={};cache={};device=recognizer.base.device;elapsed=0.0
    for angle in ('cw','ccw'):
        rotated=OP.rotated(im,angle)
        parts=LT.make_chunks(rotated,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
        spans=[(int(a),int(b)) for _,a,b in parts]
        _sync(device);start=time.perf_counter()
        logits=recognizer.base._forward_grouped([c for c,_,_ in parts],max_ratio=12)
        _sync(device);elapsed+=1000*(time.perf_counter()-start)
        ev=[V3.ctc_events(z,recognizer.chars,*span) for z,span in zip(logits,spans)]
        text=LT.normalize_ltb(V3.stitch_evidence(ev,spans,use_confidence=False,geometric_fallback=True))
        scores[angle]={'pred':text,'agreement':OP.overlap_agreement(ev,spans),
                       'confidence':OP.event_confidence(ev),'logits':logits,'spans':spans,'parts':parts}
        cache[angle]=logits
    angle,winner=OP.choose_orientation(scores['cw'],scores['ccw'],kind='overlap')
    return winner['pred'],cache,elapsed


def _fill_fallback(raw,forward_ms,im,meta):
    return dict(original_svtrv2_full12=raw,safe_scope=raw,
          route=('orientation' if im.width<=im.height else 'dense'),
          v3_geom=raw,active_crops=1,extra_crops=0,
          active_frames=int(meta.get('raw_frames',0)),extra_frames=0,
          main_forward_ms=float(forward_ms),dense_forward_ms=0.,orientation_forward_ms=0.,
          paired_scale_forward_ms=0.,scope_decoder_ms=0.,matched_c1_ms=float('nan'),
          original_extra_forward_ms=0.,original_reused_from_native=1,
          num_primary_forward_crops=1,
          scope_total_ms=float(forward_ms)+float(meta.get('probe_ms',0.)),
          safe_scope_total_ms=float(forward_ms)+float(meta.get('probe_ms',0.)),
          safe_scope_active=0,safe_scope_original_fallback=1,safe_scope_base_compared=1)


def predict_image(image,recognizer,*,run_original=True,check_legacy=False,skip_structurally_impossible=True):
    if not run_original:raise ValueError('Matched SVTRv2-T original baseline mandatory')
    im=image.convert('RGB')
    device=recognizer.base.device
    start=time.perf_counter()
    parts=LT.make_chunks(im,LT.LEGACY_CHUNK_AR,LT.LEGACY_OVERLAP)
    if len(parts)>1:
        out=FULL.predict_image(im,recognizer,run_original=True,check_legacy=check_legacy,
           skip_structurally_impossible=skip_structurally_impossible)
        out['safe_scope']=out['e7_cross']
        out['safe_scope_active']=1
        out['safe_scope_original_fallback']=0
        out['safe_scope_base_compared']=1
        out['safe_scope_total_ms']=float(out['scope_total_ms'])
        out['coverage_decision']='MULTI_CROP_DIRECT'
        out['coverage_probe_ms']=0.
        out['coverage_probe_extra_chars']=0
        out['coverage_probe_candidate_chars']=-1
        out['coverage_raw_chars']=len(out['original_svtrv2_full12'])
        out['coverage_probe_crops']=0
        out['coverage_native_crops']=len(parts)
        out['coverage_policy_version']=POLICY_VERSION
        _sync(device)
        out['scope_core_total_ms']=float(out['scope_total_ms'])
        out['safe_scope_total_ms']=1000*(time.perf_counter()-start)
        out['scope_total_ms']=out['safe_scope_total_ms']
        return out

    # A single native crop: first obtain a matched original SVTRv2-T transcript.
    _sync(device);t0=time.perf_counter()
    raw_logits=recognizer.base._forward_grouped([im],max_ratio=12)
    _sync(device)
    raw_forward_ms=1000*(time.perf_counter()-t0)
    if len(raw_logits)!=1:raise AssertionError('Raw full12 must be a single view')
    raw=LT.normalize_ltb(LT.greedy_ctc(raw_logits[0],recognizer.chars))
    meta={'raw_frames':len(raw_logits[0]),'probe_ms':0.,'probe_crops':0,
          'candidate_chars':-1,'extra_chars':0,'raw_chars':len(raw)}
    dense_logits=None;orient_logits=None
    active=False
    reason=''

    if im.width>im.height and len(raw)>=RAW_DIRECT_DENSE_TRIGGER:
        active=True;reason='DENSE_RAW_TEXT_LOAD'
    elif im.width>im.height:
        # The single-crop case that v2 incorrectly declared short. Generate
        # the primary Dense observations and compare transcript COVERAGE to
        # original raw. If activated these logits are REUSED by SCOPE.
        crop_parts=DP.forced_crops(im,3)
        _sync(device);start_probe=time.perf_counter()
        dense_logits=recognizer.dense._forward_tensors(
              [DP.resized_tensor(p,3) for p,_,_ in crop_parts])
        _sync(device);meta['probe_ms']=1000*(time.perf_counter()-start_probe)
        spans=[(int(a),int(b)) for _,a,b in crop_parts]
        candidate=_pred_from_crops(dense_logits,spans,recognizer.chars)
        active,meta['extra_chars']=_coverage_witness(raw,candidate)
        meta['candidate_chars']=len(candidate)
        meta['probe_crops']=len(dense_logits)
        reason='DENSE_COVERAGE_SUPPORTED' if active else 'DENSE_PRESERVE_RAW'
    else:
        # This also covers vertically oriented one-crop images, which v2
        # previously early-exited without ever checking the rotated content.
        candidate,orient_logits,probe_ms=_orientation_probe(im,recognizer)
        meta['probe_ms']=float(probe_ms)
        meta['candidate_chars']=len(candidate)
        meta['probe_crops']=sum(map(len,orient_logits.values()))
        active,meta['extra_chars']=_coverage_witness(raw,candidate)
        reason='ORIENTATION_COVERAGE_SUPPORTED' if active else 'ORIENTATION_PRESERVE_RAW'

    if active:
        out=FULL.predict_image(im,recognizer,run_original=True,check_legacy=check_legacy,
            skip_structurally_impossible=skip_structurally_impossible,
            native_logits_override=raw_logits,native_forward_ms_override=raw_forward_ms,
            dense_logits_override=dense_logits,dense_forward_ms_override=meta['probe_ms'] if dense_logits is not None else 0.,
            orientation_logits_override=orient_logits,orientation_forward_ms_override=meta['probe_ms'] if orient_logits is not None else 0.)
        if out['original_svtrv2_full12']!=raw:
            raise AssertionError('Raw baseline changed when reusing primary logits')
        out['safe_scope']=out['e7_cross']
        out['safe_scope_total_ms']=float(out['scope_total_ms'])+raw_forward_ms+meta['probe_ms']
        out['safe_scope_active']=1
        out['safe_scope_original_fallback']=0
        out['safe_scope_base_compared']=1
    else:
        out=_fill_fallback(raw,raw_forward_ms,im,meta)

    out['coverage_decision']=reason
    out['coverage_probe_ms']=float(meta['probe_ms'])
    out['coverage_probe_extra_chars']=int(meta['extra_chars'])
    out['coverage_probe_candidate_chars']=int(meta['candidate_chars'])
    out['coverage_raw_chars']=len(raw)
    out['coverage_probe_crops']=int(meta['probe_crops'])
    out['coverage_native_crops']=len(parts)
    out['coverage_policy_version']=POLICY_VERSION
    _sync(device)
    out['scope_core_total_ms']=float(out['scope_total_ms'])
    out['safe_scope_total_ms']=1000*(time.perf_counter()-start)
    out['scope_total_ms']=out['safe_scope_total_ms']
    return out


def cpu_tests():
    import unified_router as UNI
    class Dummy(torch.nn.Module):
        def forward(self,x):
            batch,_,_,w=x.shape
            T=max(4,w//4)
            z=torch.full((batch,T,3),-6.,device=x.device)
            z[:,:,0]=5.
            z[:,max(1,T//3),1]=10.
            z[:,max(2,2*T//3),2]=10.
            return z
    model=UNI.UnifiedRecognizer(Dummy(),['blank','a','b'],device='cpu',batch_size=2)
    for im in [Image.new('RGB',(120,30),'white'),Image.new('RGB',(20,120),'white'),
               Image.new('RGB',(330,20),'white'),Image.new('RGB',(18,210),'white')]:
        x=predict_image(im,model)
        assert isinstance(x['safe_scope'],str) and isinstance(x['original_svtrv2_full12'],str)
        assert x['safe_scope_total_ms']>=0
        if x['safe_scope_active']==0:
            assert x['safe_scope']==x['original_svtrv2_full12']
        if x['safe_scope_active']==1 and x['route']=='dense':
            old=FULL.predict_image(im,model)
            assert x['safe_scope']==old['e7_cross']
    # The logic and normalization of the coverage comparator are deliberate.
    assert _coverage_witness('abc','abcdefgh')[0]
    assert not _coverage_witness('abc','abcde')[0]
    assert not _coverage_witness('abc','ab')[0]
    assert _coverage_witness('', 'abcd')[0]
    return {'passed':True,'test_routes':4,'fallback_matches_original':True}


In [ ]:
import importlib, inspect, json, time
import scope_ctc_jit as JIT
JIT_FORWARD=JIT.activate(strict=True)
JIT_FORCED=JIT.activate_forced(strict=True)
print('CTC forward JIT:',JIT_FORWARD,'Forced Viterbi JIT:',JIT_FORCED)
print('CTC forward CPU speed test:',JIT.cpu_benchmark(n=80))
print('Forced alignment CPU speed test:',JIT.forced_benchmark(n=80))
import scope_final_runtime_v3 as FULL_V3, scope_coverage_runtime as COVER
importlib.reload(FULL_V3)
importlib.reload(COVER)
assert COVER.cpu_tests()['passed']
assert FULL_V3.cpu_tests()['passed']
assert FAST.cpu_tests()['passed']
GUARD=COVER
assert 'gt' not in inspect.signature(GUARD.predict_image).parameters
assert 'dataset' not in inspect.signature(GUARD.predict_image).parameters
print('FINAL COVERAGE RUNTIME CPU TESTS: PASS; original fallback and all 3 routes tested')


## Verified V4 speed optimization: Numba CTC prefix beam search

This patch does **not** alter any OCR hypothesis/threshold. The historical Python decoder is retained as the reference implementation. A strict numerical/sequence parity test runs before enabling JIT; the first 128 actual GPU-derived beam calls are checked again. At score ties, the kernel falls back to historical Python decoding to preserve deterministic tie behavior. Numba is a backend optimization, **not an additional research contribution**.

The observed offline CPU speedup comes from saved logits; report Colab GPU end-to-end timings separately. The V4 run uses a fresh checkpoint directory; never reuse old predictions when claiming latency improvement.

In [ ]:
%%writefile scope_beam_jit_v4.py
"""Parity-guarded Numba replacement for Python CTC prefix beam-search hot loop.

Algorithm unchanged: blank+repeat CTC prefix recurrence, top-k pruning,
beam width and order, n-best. No labels, ground truth or dataset information.
Requires numerical and prediction parity before activation.
"""
from __future__ import annotations
import math
import numpy as np
import space_ctc as SPACE
from longtext_final import normalize_ltb

_ORIGINAL=SPACE.prefix_beam_search
_READY=False
_ORIGINAL_FALLBACKS=0
_PARITY_CHECKED=0
_AUDIT_REMAINING=0


def _kernel_factory():
    from numba import njit
    @njit(cache=True)
    def la(a,b):
        if a==-np.inf:return b
        if b==-np.inf:return a
        m=a if a>b else b
        return m+math.log(math.exp(a-m)+math.exp(b-m))
    @njit(cache=True)
    def _run(lp,topmask,beam_width,nbest):
        T,C=lp.shape
        B=beam_width
        # All surviving prefixes have length <= T.
        current=np.zeros((B,T+1),dtype=np.int16)
        clens=np.zeros(B,dtype=np.int32)
        cpb=np.full(B,-np.inf,dtype=np.float64)
        cpnb=np.full(B,-np.inf,dtype=np.float64)
        cpb[0]=0.
        nb=1
        K=B*(C+1)
        ambiguity=False
        cand=np.zeros((K,T+1),dtype=np.int16)
        lens=np.zeros(K,dtype=np.int32)
        pb=np.full(K,-np.inf,dtype=np.float64)
        pnb=np.full(K,-np.inf,dtype=np.float64)
        selected=np.zeros(C,dtype=np.bool_)
        for t in range(T):
            for c in range(C):selected[c]=topmask[t,c]
            for i in range(nb):
                if clens[i]>0:selected[current[i,clens[i]-1]]=True
            count=0
            for i in range(nb):
                plen=clens[i]
                pblank=cpb[i]
                pnonblank=cpnb[i]
                whole=la(pblank,pnonblank)
                # Blank: same prefix, blank accumulator.
                index=-1
                for j in range(count):
                    if lens[j]==plen:
                        equal=True
                        for k in range(plen):
                            if cand[j,k]!=current[i,k]:equal=False;break
                        if equal:index=j;break
                if index==-1:
                    index=count;count+=1
                    lens[index]=plen
                    for k in range(plen):cand[index,k]=current[i,k]
                    pb[index]=-np.inf;pnb[index]=-np.inf
                pb[index]=la(pb[index],whole+lp[t,0])
                last=current[i,plen-1] if plen>0 else -1
                for c in range(1,C):
                    if not selected[c]:continue
                    pc=lp[t,c]
                    if c==last:
                        # Repeating label without blank: retain same prefix.
                        index=-1
                        for j in range(count):
                            if lens[j]==plen:
                                equal=True
                                for k in range(plen):
                                    if cand[j,k]!=current[i,k]:equal=False;break
                                if equal:index=j;break
                        if index==-1:
                            index=count;count+=1
                            lens[index]=plen
                            for k in range(plen):cand[index,k]=current[i,k]
                            pb[index]=-np.inf;pnb[index]=-np.inf
                        pnb[index]=la(pnb[index],pnonblank+pc)
                        branch_score=pblank+pc
                    else:
                        branch_score=whole+pc
                    extended=plen+1
                    index=-1
                    for j in range(count):
                        if lens[j]==extended and cand[j,plen]==c:
                            equal=True
                            for k in range(plen):
                                if cand[j,k]!=current[i,k]:equal=False;break
                            if equal:index=j;break
                    if index==-1:
                        index=count;count+=1
                        lens[index]=extended
                        for k in range(plen):cand[index,k]=current[i,k]
                        cand[index,plen]=c
                        pb[index]=-np.inf;pnb[index]=-np.inf
                    pnb[index]=la(pnb[index],branch_score)
            if count==0:raise ValueError('no beam')
            # Conservative fallback on exact/near ties: Python's set/dict
            # iteration can break equal-score ties in a different order.
            totals=np.empty(count,dtype=np.float64)
            for j in range(count):totals[j]=la(pb[j],pnb[j])
            for j in range(count):
                if totals[j]==-np.inf:continue
                for k in range(j+1,count):
                    if abs(totals[j]-totals[k])<=1e-10:
                        ambiguity=True
            # Stable top B by total likelihood; no string semantics in sorting.
            keep=np.ones(count,dtype=np.bool_)
            new_n=min(count,B)
            for i in range(new_n):
                best=-1;maxscore=-np.inf
                for j in range(count):
                    if not keep[j]:continue
                    score=la(pb[j],pnb[j])
                    if best==-1 or score>maxscore:
                        best=j;maxscore=score
                keep[best]=False
                clens[i]=lens[best]
                for k in range(lens[best]):current[i,k]=cand[best,k]
                cpb[i]=pb[best];cpnb[i]=pnb[best]
            nb=new_n
        out=np.zeros((min(nb,nbest),T+1),dtype=np.int16)
        outlen=np.zeros(len(out),dtype=np.int32)
        outscore=np.zeros(len(out),dtype=np.float64)
        # current already sorted by prefix likelihood
        for i in range(len(out)):
            outlen[i]=clens[i]
            outscore[i]=la(cpb[i],cpnb[i])
            for k in range(clens[i]):out[i,k]=current[i,k]
        return out,outlen,outscore,ambiguity
    return _run

_KERNEL=None


def prefix_beam_search(log_probs,chars,beam_width=8,topk=6,nbest=8):
    global _ORIGINAL_FALLBACKS,_PARITY_CHECKED,_AUDIT_REMAINING
    lp=np.asarray(log_probs,dtype=np.float64)
    if lp.ndim!=2 or lp.shape[1]!=len(chars) or lp.shape[0]<1:
        raise ValueError('Invalid prefix beam input')
    if beam_width<1 or topk<1:
        raise ValueError('Beam width and topk must be >=1')
    topmask=np.zeros(lp.shape,dtype=np.bool_)
    k=min(topk,len(chars))
    topinds=np.argpartition(lp,-k,axis=1)[:,-k:]
    np.put_along_axis(topmask,topinds,True,axis=1)
    seq,lens,scores,ambiguous=_KERNEL(lp,topmask,beam_width,nbest)
    if ambiguous:
        _ORIGINAL_FALLBACKS += 1
        return _ORIGINAL(log_probs,chars,beam_width=beam_width,topk=topk,nbest=nbest)
    out=[]
    for i in range(len(lens)):
        pred=normalize_ltb(''.join(chars[int(j)] for j in seq[i,:lens[i]]))
        out.append({'pred':pred,'logp':float(scores[i])})
    if _AUDIT_REMAINING>0:
        old=_ORIGINAL(log_probs,chars,beam_width=beam_width,topk=topk,nbest=nbest)
        if len(old)!=len(out) or any(a['pred']!=b['pred'] or abs(a['logp']-b['logp'])>1e-8 for a,b in zip(old,out)):
            raise RuntimeError('PREFIX_BEAM_PARITY_FAILURE: stop inference, do not use scores')
        _PARITY_CHECKED+=1
        _AUDIT_REMAINING-=1
    return out


def activate(strict=True,audit_first=128):
    global _KERNEL,_READY,_AUDIT_REMAINING,_ORIGINAL_FALLBACKS,_PARITY_CHECKED
    if _READY:return {'status':'already_active','available':True}
    try:
        _KERNEL=_kernel_factory()
        # First compile and compare random, low blank, high blank, repeated letters and ties.
        rng=np.random.default_rng(20261121)
        for T in (1,2,3,4,9,16,48,80):
            for case in range(6):
                q=rng.uniform(1e-6,1,size=(T,37))
                if case==1:q[:,0]*=70
                if case==2:q[:,0]*=.05
                if case==3:q.fill(.5)
                if case==4:q[:,1]=q[:,2]=10
                if case==5:q[:,0]=q[:,1]=10
                q/=q.sum(axis=1,keepdims=True)
                for b,top,n in ((6,5,3),(3,3,3)):
                    old=_ORIGINAL(np.log(q),['']+list('0123456789abcdefghijklmnopqrstuvwxyz'),beam_width=b,topk=top,nbest=n)
                    new=prefix_beam_search(np.log(q),['']+list('0123456789abcdefghijklmnopqrstuvwxyz'),beam_width=b,topk=top,nbest=n)
                    if [x['pred'] for x in old]!=[x['pred'] for x in new] or not all(abs(x['logp']-y['logp'])<1e-9 for x,y in zip(old,new)):
                        raise AssertionError((T,case,b,top,old[:2],new[:2]))
        SPACE.prefix_beam_search=prefix_beam_search
        _AUDIT_REMAINING=int(audit_first)
        _PARITY_CHECKED=0
        _ORIGINAL_FALLBACKS=0
        _READY=True
        return {'status':'numba_prefix_beam_parity_pass','available':True}
    except Exception as exc:
        SPACE.prefix_beam_search=_ORIGINAL
        _READY=False
        if strict:raise
        return {'status':str(exc),'available':False}


def benchmark(n=50,T=48):
    global _AUDIT_REMAINING
    preserved_audit=_AUDIT_REMAINING
    _AUDIT_REMAINING=0
    rng=np.random.default_rng(201)
    q=rng.uniform(.0001,1,(T,37));q[:,0]*=5;q/=q.sum(axis=1,keepdims=True)
    args=(np.log(q),['']+list('0123456789abcdefghijklmnopqrstuvwxyz'))
    import time
    a=time.perf_counter()
    for i in range(n):_ORIGINAL(*args,beam_width=6,topk=5,nbest=3)
    b=time.perf_counter()
    for i in range(n):prefix_beam_search(*args,beam_width=6,topk=5,nbest=3)
    c=time.perf_counter()
    _AUDIT_REMAINING=preserved_audit
    return {'original_ms':(b-a)*1e3/n,'fast_ms':(c-b)*1e3/n,'ratio':(b-a)/(c-b),'auditing_paused_during_microbenchmark':True}


def audit_status():
    return {'audit_checked':int(_PARITY_CHECKED),'audit_remaining':int(_AUDIT_REMAINING),
            'tie_fallbacks':int(_ORIGINAL_FALLBACKS),'active':bool(_READY)}


In [ ]:
import importlib, json, time, inspect
import space_ctc as SPACE, alex_e5 as E5
import scope_beam_jit_v4 as BEAM_JIT
BEAM_JIT_STATUS = BEAM_JIT.activate(strict=True, audit_first=128)
assert BEAM_JIT_STATUS['available']
assert SPACE.prefix_beam_search is BEAM_JIT.prefix_beam_search
BEAM_CPU_BENCH = BEAM_JIT.benchmark(n=50)
print('V4 prefix-beam JIT:', BEAM_JIT_STATUS)
print('V4 microbenchmark (random synthetic CPU posterior, not GPU end-to-end):',BEAM_CPU_BENCH)
print('Automatic parity audit:',BEAM_JIT.audit_status())
assert E5.cpu_tests()['passed']
assert FAST.cpu_tests()['passed']
assert FULL_V3.cpu_tests()['passed']
assert COVER.cpu_tests()['passed']
print('V4 READY: 128 early beam predictions will be checked against original Python search')

## Independent GPU inference — full LTB, checkpoint by ID

No label is read inside `GUARD.predict_image`. No historical E7 row or GT is passed into inference. `safe_scope` uses the original frozen SVTRv2-T result on guarded images, otherwise the unmodified E7 Cross-Scale result. `raw baseline` is recorded on every image. The distinct output path prevents accidental use of the old E7 cache.

In [ ]:
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd, numpy as np, json, hashlib, time, os
LTB_JOURNAL=OUT/'scope_coverage_inference_GT_blind.jsonl'
MODEL_LOCK=hashlib.sha256(json.dumps({
  'policy':COVER.POLICY_VERSION,'native_min':COVER.RAW_DIRECT_DENSE_TRIGGER,
  'coverage_extra':COVER.COVERAGE_EXTRA_MIN,
  'coverage_relative':COVER.COVERAGE_RELATIVE_GAIN,
  'model':provenance['checkpoint_sha256'],
  'cover_sha':sha256(WORK/'scope_coverage_runtime.py'),
  'jit_sha':sha256(WORK/'scope_ctc_jit.py'),
  'beam_jit_sha':sha256(WORK/'scope_beam_jit_v4.py'),
  'beam_jit_active':BEAM_JIT_STATUS['available'],
  'full_sha':sha256(WORK/'scope_final_runtime_v3.py'),
  'jit_enabled':JIT_FORWARD['available'] and JIT_FORCED['available'],
  'data_revision':SUITE.DATASET_REV if 'SUITE' in globals() else 'd59c13643f1d7f92831625fe161e2c19ec3d7f52'
},sort_keys=True).encode()).hexdigest()
LOCK_PATH=OUT/'scope_coverage_run_lock.json'
lock={'model_lock':MODEL_LOCK,'n_images':len(items),'policy':COVER.POLICY_VERSION,'preprocessing':'full12','max_images':MAX_IMAGES}
if LOCK_PATH.is_file():
  assert json.loads(LOCK_PATH.read_text())==lock,'Unsafe resume: model/code/benchmark changed'
else:LOCK_PATH.write_text(json.dumps(lock,indent=2))
done={}
if LTB_JOURNAL.is_file():
  bad_tail=False
  for line in LTB_JOURNAL.read_text(encoding='utf-8').splitlines():
    if line.strip():
      try:r=json.loads(line)
      except json.JSONDecodeError:
        bad_tail=True;break
      done[int(r['id'])]=r
  if bad_tail:
    tmp=LTB_JOURNAL.with_suffix('.repaired')
    tmp.write_text(''.join(json.dumps(done[k],ensure_ascii=False,default=str)+'\n' for k in sorted(done)))
    tmp.replace(LTB_JOURNAL)
    print('Repaired incomplete LTB journal tail')
assert set(done).issubset({int(x['id']) for x in RUN_ITEMS})
pending=[x for x in RUN_ITEMS if int(x['id']) not in done]
print('FINAL SCOPE LTB:',len(done),'done,',len(pending),'remaining')
flush=[]
try:
  for item in tqdm(pending,desc='SCOPE FINAL coverage JIT LTB'):
    t=time.perf_counter()
    with Image.open(item['path']) as pic:im=pic.convert('RGB')
    pred=COVER.predict_image(im,recognizer,check_legacy=False)
    pred['image_io_and_total_ms']=1000*(time.perf_counter()-t)
    pred['id']=int(item['id'])
    done[int(item['id'])]=pred;flush.append(pred)
    if len(flush)>=25:
      with LTB_JOURNAL.open('a',encoding='utf-8') as f:
        for r in flush:f.write(json.dumps(r,ensure_ascii=False,default=str)+'\n')
        f.flush();os.fsync(f.fileno())
      flush=[]
finally:
  if flush:
    with LTB_JOURNAL.open('a',encoding='utf-8') as f:
      for r in flush:f.write(json.dumps(r,ensure_ascii=False,default=str)+'\n')
      f.flush();os.fsync(f.fileno())
RAW=pd.DataFrame([done[int(x['id'])] for x in RUN_ITEMS]).sort_values('id').reset_index(drop=True)
assert len(RAW)==len(RUN_ITEMS) and RAW['id'].is_unique
assert RAW['original_svtrv2_full12'].map(lambda s:isinstance(s,str)).all()
assert RAW['safe_scope'].map(lambda s:isinstance(s,str)).all()
RAW.to_csv(OUT/'predictions_GT_blind.csv',index=False)
print('SEALED GT-blind predictions:',len(RAW),'active:',int(RAW['safe_scope_active'].sum()),
      'probed:',int(RAW['coverage_probe_crops'].gt(0).sum()))


## Active-row historical E7 parity

Only activated rows should match E7 exactly. Protected rows intentionally fall back to original SVTRv2-T. This audit does not use labels to select predictions.

In [ ]:
import io,gzip,base64,hashlib,pandas as pd
E7_PARITY_BLOB_SHA256="8ddb4efec680f719a5055debdbab7ddb5a663b44b839605af76c6a5afb68751a"
E7_PARITY_BLOB_B64="..."
raw=base64.b64decode(E7_PARITY_BLOB_B64)
assert hashlib.sha256(raw).hexdigest()==E7_PARITY_BLOB_SHA256
OLD_E7=pd.read_csv(io.BytesIO(gzip.decompress(raw)),keep_default_na=False)
active_rows=RAW[RAW['safe_scope_active'].astype(int).eq(1)].copy()
parity=active_rows[['id','e7_native','e7_cross']].merge(OLD_E7,on='id',how='left',validate='one_to_one',indicator=True,suffixes=('_new','_e7'))
assert (parity['_merge']=='both').all()
for m in ('e7_native','e7_cross'):
    parity[m+'_exact_parity']=parity[m+'_new'].eq(parity[m+'_e7'])
E7_PARITY_COUNTS={m:int(parity[m+'_exact_parity'].sum()) for m in ('e7_native','e7_cross')}
parity[['id','e7_native_exact_parity','e7_cross_exact_parity']].to_csv(OUT/'e7_decoder_parity.csv',index=False)
if not all(n==len(active_rows) for n in E7_PARITY_COUNTS.values()):
    parity[~(parity['e7_native_exact_parity']&parity['e7_cross_exact_parity'])].to_csv(OUT/'e7_decoder_mismatches.csv',index=False)
print('Historical E7 exact-output parity:',E7_PARITY_COUNTS,'/',len(active_rows))
E7_PARITY_PASS=all(n==len(active_rows) for n in E7_PARITY_COUNTS.values())
if MAX_IMAGES==0:assert E7_PARITY_PASS,'Review E7 decoder mismatches before claiming same-output speedup'

assert RAW.loc[RAW['safe_scope_active'].astype(int).eq(0),'safe_scope'].equals(RAW.loc[RAW['safe_scope_active'].astype(int).eq(0),'original_svtrv2_full12'])
print('Baseline preservation on guarded rows: PASS')


## Strict LTB evaluation — both methods, all three length buckets and A-Avg

The reference is read *after* predictions are saved. Report exact-line, CER, S/D/I, paired rescues and regressions. No cumulative metrics.

In [ ]:
from rapidfuzz.distance import Levenshtein
from scipy.stats import binomtest
GT={int(it['id']):LT.normalize_ltb(it['gt']) for it in items}
S=RAW.copy()
S['gt']=S['id'].map(GT)
assert S['gt'].notna().all()
S['bucket']=S['gt'].map(lambda x:'26-35' if len(x)<=35 else '36-55' if len(x)<=55 else '56+')
if MAX_IMAGES==0:assert S['bucket'].value_counts().to_dict()=={'26-35':3376,'36-55':1147,'56+':266}
METHODS=['original_svtrv2_full12','safe_scope']
for m in METHODS:
    S[m+'_ok']=S[m].eq(S['gt'])
    ops=[Levenshtein.editops(a,b) for a,b in zip(S[m],S['gt'])]
    S[m+'_S']=[sum(k.tag=='replace' for k in op) for op in ops]
    S[m+'_D']=[sum(k.tag=='insert' for k in op) for op in ops]
    S[m+'_I']=[sum(k.tag=='delete' for k in op) for op in ops]
    S[m+'_edits']=S[m+'_S']+S[m+'_D']+S[m+'_I']
res=[];paired=[]
for m in METHODS:
  av=[]
  for b in ('26-35','36-55','56+'):
    d=S[S.bucket==b];a=float(d[m+'_ok'].mean());av.append(a)
    res.append(dict(method=m,bucket=b,n=len(d),correct=int(d[m+'_ok'].sum()),accuracy=a,
      CER=float(d[m+'_edits'].sum()/d['gt'].str.len().sum()),
      S=int(d[m+'_S'].sum()),D=int(d[m+'_D'].sum()),I=int(d[m+'_I'].sum())))
  res.append(dict(method=m,bucket='Overall',n=len(S),correct=int(S[m+'_ok'].sum()),
    accuracy=float(S[m+'_ok'].mean()),CER=float(S[m+'_edits'].sum()/S['gt'].str.len().sum()),
    S=int(S[m+'_S'].sum()),D=int(S[m+'_D'].sum()),I=int(S[m+'_I'].sum())))
  res.append(dict(method=m,bucket='A-Avg',n=len(S),correct=None,
    accuracy=float(np.mean(av)),CER=None,S=None,D=None,I=None))
for b in ('26-35','36-55','56+','Overall'):
  d=S if b=='Overall' else S[S.bucket==b]
  w=int((d['safe_scope_ok']&~d['original_svtrv2_full12_ok']).sum())
  l=int((~d['safe_scope_ok']&d['original_svtrv2_full12_ok']).sum())
  paired.append(dict(test='safe_scope',control='original_svtrv2_full12',bucket=b,
    n=len(d),rescued=w,regressed=l,net=w-l,
    mcnemar_exact_p=float(binomtest(w,w+l,.5).pvalue) if w+l else 1.))
Q=pd.DataFrame(res);PAIR=pd.DataFrame(paired)
S.to_csv(OUT/'predictions_scored_full.csv',index=False)
Q.to_csv(OUT/'quality_by_bucket.csv',index=False)
PAIR.to_csv(OUT/'paired_vs_original.csv',index=False)
lt=pd.DataFrame([{'family':'LTB','dataset':'LTB_extended_4789','stage':stage,'n':int(S[stage].notna().sum()),
 'mean_ms':float(S[stage].mean()),'median_ms':float(S[stage].median()),
 'p95_ms':float(S[stage].quantile(.95))} for stage in ['safe_scope_total_ms','main_forward_ms',
 'coverage_probe_ms','scope_decoder_ms','paired_scale_forward_ms']])
lt.to_csv(OUT/'latency_by_bucket_stage.csv',index=False)
print('FINAL SCOPE quality:');print(Q.to_string(index=False));print('Paired:');print(PAIR.to_string(index=False))


## Cross-dataset evaluation — same official datasets, new safe inference checkpoint

The dataset loader is retained. The evaluator compares **only original SVTRv2-T vs SAFE-SCOPE** because unprotected ALEX/Unified/E7 outputs are *not computed* on early-exit rows. This avoids presenting missing ablations as measured baselines.

In [ ]:
%%writefile scope_safe_suite.py
"""One-pass cross-dataset evaluation for the pinned SCOPE-CTC E7 FINAL.

This module ONLY handles data acquisition, locked evaluation, crash recovery and
reporting. It does not alter model weights, crop policy, beam, or edit thresholds.
"""
from __future__ import annotations
import csv
import hashlib
import io
import json
import math
import os
import random
import statistics
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from rapidfuzz.distance import Levenshtein
from scipy.stats import binomtest

DATASET_REPO = 'topdu/OpenOCR-Data'
DATASET_REV = 'd59c13643f1d7f92831625fe161e2c19ec3d7f52'

# Official OpenOCR test variants, NOT the larger IC13_1015 / IC15_2077 variants.
SPECS = [
    ('Common','IIIT5K','test/IIIT5k',3000),
    ('Common','SVT','test/SVT',647),
    ('Common','IC13-857','test/IC13_857',857),
    ('Common','IC15-1811','test/IC15_1811',1811),
    ('Common','SVTP','test/SVTP',645),
    ('Common','CUTE80','test/CUTE80',288),
    ('Union14M','Curve','u14m/curve',2426),
    ('Union14M','Multi-Oriented','u14m/multi_oriented',1369),
    ('Union14M','Artistic','u14m/artistic',900),
    ('Union14M','Contextless','u14m/contextless',779),
    ('Union14M','Salient','u14m/salient',1585),
    ('Union14M','Multi-Words','u14m/multi_words',829),
    ('OST','Weak','OST/weak',None),
    ('OST','Heavy','OST/heavy',None),
]
GENERAL = ('Union14M','General','u14m/general',400000)
METHODS = ['original_svtrv2_full12','safe_scope']
CANDIDATE_METHOD = 'safe_scope'


def checksum(obj):
    if isinstance(obj,(bytes,bytearray)): payload=obj
    else: payload=json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def fetch_lmdb(spec, data_home, *, revision=DATASET_REV):
    from huggingface_hub import hf_hub_download
    family,name,folder,expected=spec
    data_home=Path(data_home)
    local=data_home/folder/'data.mdb'
    if local.is_file() and local.stat().st_size>4096:
        return local.parent
    data_home.mkdir(parents=True, exist_ok=True)
    err=None
    for attempt in range(1,4):
        try:
            result=hf_hub_download(repo_id=DATASET_REPO,filename=folder+'/data.mdb',
                                   repo_type='dataset',revision=revision,
                                   local_dir=str(data_home),etag_timeout=60)
            result=Path(result)
            if result.stat().st_size<=4096:raise RuntimeError(f'Download suspiciously small: {result}')
            return result.parent
        except Exception as ex:
            err=ex
            print(f'DOWNLOAD RETRY {name} {attempt}/3: {type(ex).__name__}: {ex}',flush=True)
            if attempt<3:time.sleep(2*attempt)
    raise RuntimeError(f'Failed to download {name} LMDB: {err}') from err


class LmdbSamples:
    def __init__(self,folder,*,expected=None):
        import lmdb
        self.folder=Path(folder)
        self.env=lmdb.open(str(self.folder),readonly=True,lock=False,readahead=False,
                           meminit=False,max_readers=32)
        with self.env.begin(write=False) as tr:
            count=tr.get(b'num-samples')
            if count is None:raise RuntimeError(f'Missing num-samples: {folder}')
            self.n=int(count)
            if self.n<=0:raise RuntimeError(f'Empty LMDB: {folder}')
            for i in sorted(set((1, max(1,self.n//2),self.n))):
                if not tr.get(f'image-{i:09d}'.encode()):
                    raise RuntimeError(f'Missing image-{i:09d} in {folder}')
                if tr.get(f'label-{i:09d}'.encode()) is None:
                    raise RuntimeError(f'Missing label-{i:09d} in {folder}')
        if expected is not None and self.n!=expected:
            raise RuntimeError(f'Unexpected sample count {folder}: expected {expected}, got {self.n}')
    def get_image(self,idx):
        idx=int(idx)
        if not 1<=idx<=self.n:raise IndexError(idx)
        with self.env.begin(write=False) as tr:
            b=tr.get(f'image-{idx:09d}'.encode())
        if not b:raise RuntimeError(f'Missing/corrupt image idx={idx} in {self.folder}')
        with Image.open(io.BytesIO(b)) as pic:
            return pic.convert('RGB')
    def get_label(self,idx):
        idx=int(idx)
        with self.env.begin(write=False) as tr:
            b=tr.get(f'label-{idx:09d}'.encode())
        if b is None:raise RuntimeError(f'Missing label idx={idx} in {self.folder}')
        return b.decode('utf-8','replace')
    def close(self):
        self.env.close()


def sample_indices(n, limit, seed):
    if limit is None or int(limit)==0 or int(limit)>=n:return list(range(1,n+1))
    if int(limit)<0:raise ValueError('limit must be nonnegative')
    return sorted(random.Random(int(seed)).sample(range(1,n+1),int(limit)))


def bucket(text):
    length=len(text)
    if length<=25:return '01-25'
    if length<=35:return '26-35'
    if length<=55:return '36-55'
    return '56+'


def resume_journal(path, valid_indices):
    path=Path(path);result={};bad_tail=False
    if path.exists():
        with path.open('r',encoding='utf-8') as f:
            for no,line in enumerate(f,1):
                if not line.strip():continue
                try:r=json.loads(line)
                except json.JSONDecodeError:
                    bad_tail=True
                    print(f'WARNING: ignored unfinished checkpoint line {no} in {path.name}')
                    break
                idx=int(r['sample_index'])
                if idx not in valid_indices:raise RuntimeError(f'Resume ID outside selected cohort: {idx}')
                result[idx]=r
    if bad_tail:
        temp=path.with_suffix('.repaired')
        with temp.open('w',encoding='utf-8') as f:
            for idx in sorted(result):f.write(json.dumps(result[idx],ensure_ascii=False,default=str)+'\n')
        temp.replace(path)
    return result


def run_dataset(spec, *, model, out_root, data_home, model_fingerprint,
                general_mode='off', general_n=2000, seed=20261120, max_per_dataset=0,
                flush_every=25, strict_image_errors=False, infer_fn=None,
                dataset_opener=None):
    from tqdm.auto import tqdm
    if infer_fn is None:
        import scope_guarded_runtime as runtime
        infer_fn=runtime.predict_image
    family,name,folder,expected=spec
    if name=='General' and general_mode=='off':
        raise ValueError('General is disabled')
    ds_key=family+'__'+name.replace(' ','_')
    out=Path(out_root)/ds_key
    out.mkdir(parents=True,exist_ok=True)
    if dataset_opener is not None:
        ds=dataset_opener(spec)
    else:
        folder_path=fetch_lmdb(spec,data_home)
        ds=LmdbSamples(folder_path,expected=expected)
    try:
        if name=='General' and general_mode=='sample':
            selected=sample_indices(ds.n,general_n,seed)
            kind='DETERMINISTIC_SUBSET_NOT_FULL_BENCHMARK'
        else:
            selected=sample_indices(ds.n,max_per_dataset,seed)
            kind='FULL' if len(selected)==ds.n else 'DETERMINISTIC_SUBSET_NOT_FULL_BENCHMARK'
        spec_data={
            'family':family,'dataset':name,'folder':folder,'source_repo':DATASET_REPO,
            'source_revision':DATASET_REV,'available_samples':ds.n,'evaluated_samples':len(selected),
            'sample_kind':kind,'selection_seed':seed,'selection_sha256':checksum(selected),
            'model_fingerprint':model_fingerprint,
            'inference_module':'scope_coverage_runtime.predict_image',
            'control_methods':METHODS,'run_original':True,
            'ctc_processing':'frozen SVTRv2-T, coverage probe and reused primary CTC, JIT exact scoring',
        }
        spec_file=out/'run_spec.json'
        if spec_file.exists():
            old=json.loads(spec_file.read_text())
            if old!=spec_data:raise RuntimeError(f'Unsafe resume: changed dataset config for {ds_key}')
        else:spec_file.write_text(json.dumps(spec_data,indent=2,ensure_ascii=False))
        journal=out/'inference_GT_blind_journal.jsonl'
        done=resume_journal(journal,set(selected))
        pending=[idx for idx in selected if idx not in done]
        print(f'\n{ds_key}: {len(done)} completed, {len(pending)} remaining / {len(selected)} '+
              f'({kind}), available {ds.n}',flush=True)
        buf=[];tick=time.perf_counter()
        try:
            for idx in tqdm(pending,desc=ds_key):
                try:
                    io_start=time.perf_counter()
                    image=ds.get_image(idx)
                    io_ms=(time.perf_counter()-io_start)*1000
                    pred=infer_fn(image,model,run_original=True,check_legacy=False,
                                  skip_structurally_impossible=True)
                    del image
                    r={k:(v.item() if isinstance(v,np.generic) else v) for k,v in pred.items()}
                    r.update(sample_index=int(idx),dataset=name,family=family,
                             status='OK',image_io_ms=round(io_ms,3))
                except (KeyboardInterrupt,SystemExit):
                    raise
                except Exception as exc:
                    r={'sample_index':int(idx),'dataset':name,'family':family,
                       'status':'FAILED','error_type':type(exc).__name__,
                       'error_summary':str(exc)[:500]}
                    if strict_image_errors:raise
                done[idx]=r;buf.append(r)
                if len(buf)>=int(flush_every):
                    with journal.open('a',encoding='utf-8') as f:
                        for z in buf:f.write(json.dumps(z,ensure_ascii=False,default=str)+'\n')
                        f.flush();os.fsync(f.fileno())
                    buf=[]
                    if len(done)%250<flush_every:
                        print(f'{ds_key}: {len(done)}/{len(selected)} completed, elapsed {round(time.perf_counter()-tick,1)}s',flush=True)
        finally:
            if buf:
                with journal.open('a',encoding='utf-8') as f:
                    for z in buf:f.write(json.dumps(z,ensure_ascii=False,default=str)+'\n')
                    f.flush();os.fsync(f.fileno())
        if len(done)!=len(selected):
            raise RuntimeError(f'Incomplete run {ds_key}: {len(done)} != {len(selected)}')
        rows=pd.DataFrame([done[idx] for idx in selected])
        outpath=out/'predictions_GT_blind.csv'
        rows.to_csv(outpath,index=False)
        return ds_key,ds,rows,out
    except Exception:
        ds.close()
        raise


def score_dataset(ds,rows,out,normalize_fn,*,fail_if_incomplete=False):
    data=rows.copy()
    assert data['sample_index'].is_unique, 'Duplicate LMDB IDs'
    labels=[ds.get_label(idx) for idx in data['sample_index'].astype(int)]
    data['gt_original']=labels
    data['gt']=[normalize_fn(s) for s in labels]
    data['gt_length']=data['gt'].str.len()
    data['length_bucket']=data['gt'].map(bucket)
    data['gt_symbols_removed']=data['gt_original'].map(lambda s:sum(not('a'<=ch.lower()<='z' or '0'<=ch<='9') for ch in s))
    data['eligible']=data['status'].eq('OK') & data['gt_length'].gt(0)
    for m in METHODS:
        if m not in data:raise RuntimeError(f'Missing method column {m}')
        data[m]=data[m].fillna('')
    N=len(data);valid=data[data['eligible']].copy()
    if valid.empty:raise RuntimeError('No valid samples for scoring')
    missing=N-len(valid)
    print(f'SCORED {len(valid)}/{N}; excluded {missing} (inference failure/zero normalized GT)',flush=True)
    if fail_if_incomplete and missing:raise RuntimeError('Incomplete benchmark, see scored CSV')
    for m in METHODS:
        valid[m+'_ok']=valid[m].eq(valid['gt'])
        ops=[Levenshtein.editops(a,b) for a,b in zip(valid[m],valid['gt'])]
        # Order is prediction -> GT; insert is missing characters, delete is extra output.
        valid[m+'_S']=[sum(x.tag=='replace' for x in op) for op in ops]
        valid[m+'_D']=[sum(x.tag=='insert' for x in op) for op in ops]
        valid[m+'_I']=[sum(x.tag=='delete' for x in op) for op in ops]
        valid[m+'_edits']=(valid[m+'_S']+valid[m+'_D']+valid[m+'_I'])
    valid['SCOPE_vs_original']=valid['safe_scope_ok'].astype(int)-valid['original_svtrv2_full12_ok'].astype(int)
    for col in valid.columns:
        if col.endswith(('_ok','_S','_D','_I','_edits')) or col=='SCOPE_vs_original':
            data[col]=data['sample_index'].map(valid.set_index('sample_index')[col])
    data.to_csv(Path(out)/'predictions_scored.csv',index=False)
    return data,valid


def summarise_scored(scored_datasets, group_name='External'):
    """Collect per-dataset and length rows; per-dataset exact and CER use one cohort."""
    quality=[];paired=[];latency=[];by_length=[]
    for spec,df,all_count in scored_datasets:
        family,name,*_=spec
        if df.empty:continue
        d=df.copy()
        subsets=[(name,d)] + [(f'LEN:{b}',sub) for b,sub in d.groupby('length_bucket',sort=True)]
        for subsection,part in subsets:
            if subsection.startswith('LEN:'):
                suffix={'length_bucket':subsection[4:]}
            else:suffix={'length_bucket':'ALL'}
            for m in METHODS:
                total_gt=int(part['gt_length'].sum())
                row={'family':family,'dataset':name,'length_bucket':suffix['length_bucket'],
                     'method':m,'n_evaluated':len(part),'n_expected':all_count,
                     'correct':int(part[m+'_ok'].sum()),
                     'accuracy':float(part[m+'_ok'].mean()),
                     'CER':float(part[m+'_edits'].sum()/total_gt) if total_gt else float('nan'),
                     'S':int(part[m+'_S'].sum()),'D':int(part[m+'_D'].sum()),'I':int(part[m+'_I'].sum()),
                     'gt_chars':total_gt}
                (by_length if subsection.startswith('LEN:') else quality).append(row)
        for bname,subset in [('ALL',d)]+[(b,t) for b,t in d.groupby('length_bucket',sort=True)]:
            for baseline in ('original_svtrv2_full12',):
                w=int((subset['safe_scope_ok']&~subset[baseline+'_ok']).sum())
                l=int((~subset['safe_scope_ok']&subset[baseline+'_ok']).sum())
                paired.append({'family':family,'dataset':name,'length_bucket':bname,
                               'baseline':baseline,'method':'safe_scope',
                               'n_evaluated':len(subset),'rescued':w,'regressed':l,'net':w-l,
                               'mcnemar_exact_p':float(binomtest(w,w+l,.5).pvalue) if w+l else 1.})
        for stage in ('safe_scope_total_ms','scope_total_ms','coverage_probe_ms','matched_c1_ms','main_forward_ms','paired_scale_forward_ms',
                      'scope_decoder_ms','image_io_ms'):
            if stage not in d:continue
            values=pd.to_numeric(d[stage],errors='coerce').dropna()
            if not len(values):continue
            latency.append({'family':family,'dataset':name,'stage':stage,'n':len(values),
                            'mean_ms':float(values.mean()),'median_ms':float(values.median()),
                            'p95_ms':float(values.quantile(.95)),'total_s':float(values.sum()/1000)})
        # Reused raw original when one native crop, matching original FINAL latency accounting.
        if 'original_extra_forward_ms' in d and 'original_reused_from_native' in d:
            raw_ms=np.where(pd.to_numeric(d['original_reused_from_native'],errors='coerce').fillna(0).eq(1),
                            pd.to_numeric(d['main_forward_ms'],errors='coerce'),
                            pd.to_numeric(d['original_extra_forward_ms'],errors='coerce'))
            r=pd.Series(raw_ms).dropna()
            if len(r):
                latency.append({'family':family,'dataset':name,'stage':'raw_svtr_ms','n':len(r),
                                'mean_ms':float(r.mean()),'median_ms':float(r.median()),
                                'p95_ms':float(r.quantile(.95)),'total_s':float(r.sum()/1000)})
    quality=pd.DataFrame(quality);paired=pd.DataFrame(paired)
    latency=pd.DataFrame(latency);by_length=pd.DataFrame(by_length)
    return quality,paired,latency,by_length


def aggregate_families(scored_datasets):
    summary=[]
    families=sorted(set(spec[0] for spec,_,_ in scored_datasets))
    if any(spec[0]=='Union14M' for spec,_,_ in scored_datasets):
        families.append('Union14M_6_NON_GENERAL')
    for family in families+['ALL_EXTERNAL']:
        selected=[(spec,d) for spec,d,_ in scored_datasets if
                  spec[0]==family or
                  (family=='Union14M_6_NON_GENERAL' and spec[0]=='Union14M' and spec[1]!='General') or
                  family=='ALL_EXTERNAL']
        if not selected:continue
        ds_rows=[]
        for m in METHODS:
            n=sum(len(d) for _,d in selected)
            hits=sum(int(d[m+'_ok'].sum()) for _,d in selected)
            edits=sum(int(d[m+'_edits'].sum()) for _,d in selected)
            chars=sum(int(d['gt_length'].sum()) for _,d in selected)
            means=[float(d[m+'_ok'].mean()) for _,d in selected]
            summary.append({'family':family,'method':m,'datasets_included':len(selected),
                            'n_evaluated':n,'correct':hits,'micro_accuracy':hits/n if n else float('nan'),
                            'dataset_macro_accuracy':float(np.mean(means)),
                            'micro_CER':edits/chars if chars else float('nan'),
                            'aggregation_note':('FULL six non-General Union14M subsets' if family=='Union14M_6_NON_GENERAL' else
                                'Exploratory: if General is sampled this family includes a PARTIAL dataset; '
                                'macro weights named datasets equally, not a claimed official ranking')})
    return pd.DataFrame(summary)


def pack_results(zip_path,root,ltb_out,meta):
    zip_path=Path(zip_path);root=Path(root);ltb_out=Path(ltb_out)
    zip_path.parent.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
        z.writestr('MULTIDATASET_PROTOCOL.json',json.dumps(meta,indent=2,ensure_ascii=False))
        for p in sorted(root.rglob('*')):
            if p.is_file() and ('/.' not in str(p)):
                z.write(p,arcname='cross_dataset/'+str(p.relative_to(root)))
        for module_name in ['longtext_final.py','longtext_v3.py','density_probe.py','orientation_probe.py',
                            'unified_router.py','alex_ctc.py','alex_c1.py','alex_e5.py','space_ctc.py',
                            'alex_e7.py','scope_e7_fast.py','scope_final_runtime_v3.py',
                            'scope_coverage_runtime.py','scope_ctc_jit.py','scope_beam_jit_v4.py','scope_safe_suite.py']:
            source_file=Path(__file__).resolve().parent/module_name
            if source_file.exists():z.write(source_file,arcname='source_modules/'+module_name)
        for name in ['quality_by_bucket.csv','paired_vs_original.csv','latency_by_bucket_stage.csv',
                     'predictions_GT_blind.csv','predictions_scored_full.csv','ltb_coverage_decisions.csv',
                     'e7_decoder_parity.csv','protocol.json','provenance.json']:
            f=ltb_out/name
            if f.is_file():z.write(f,arcname='LTB_extended/'+name)
    return zip_path


def local_logic_smoke_test():
    """No network/GPU needed; validate scoring, groups, paired, and sparse output."""
    class Fake:
        def get_label(self,idx):
            return {1:'Ab 12',2:'ac',3:'zb'}[int(idx)]
    rows=[]
    for idx in (1,2,3):
        p='ab12' if idx==1 else 'ac' if idx==2 else 'zz'
        q='ab12' if idx==1 else 'ac' if idx==2 else 'zb'
        r={m:p for m in METHODS}
        r['safe_scope']=q
        r.update(sample_index=idx,status='OK',scope_total_ms=20.,matched_c1_ms=5.,
                 main_forward_ms=3.,original_extra_forward_ms=0.,original_reused_from_native=1,
                 image_io_ms=1.)
        rows.append(r)
    import tempfile
    with tempfile.TemporaryDirectory() as d:
        norm=lambda s:''.join(ch for ch in s.lower() if ('a'<=ch<='z' or '0'<=ch<='9'))
        scored,valid=score_dataset(Fake(),pd.DataFrame(rows),d,norm)
        assert len(valid)==3
        q,p,l,b=summarise_scored([(('Common','TEST','/fake',3),valid,3)])
        assert q.query("method == 'safe_scope'")['correct'].iloc[0]==3
        assert p.query("baseline == 'original_svtrv2_full12' and length_bucket == 'ALL'")['rescued'].iloc[0]==1
        assert l.query("stage == 'raw_svtr_ms'")['mean_ms'].iloc[0]==3.0
        assert b.query("length_bucket == '01-25'")['n_evaluated'].max()==3
        assert (scored['gt'].tolist()==['ab12','ac','zb'])
        assert sample_indices(50,8,17)==sample_indices(50,8,17)
    return True


In [ ]:
import scope_safe_suite as SUITE
import importlib
importlib.reload(SUITE)
assert SUITE.local_logic_smoke_test(), 'Matched scorer test failed'
GENERAL_MODE='sample'
GENERAL_SAMPLE_N=2000
GENERAL_SAMPLE_SEED=20261120
MAX_PER_OTHER_DATASET=0
EXTERNAL_OUT=MULTIDATA_PERSIST_ROOT/'SCOPE_FINAL/CrossDataset'
EXTERNAL_OUT.mkdir(exist_ok=True,parents=True)
EXTERNAL_DATA_HOME=Path('/content/SCOPE_OFFICIAL_OPENOCR_DATA')
EXTERNAL_WRITE_EVERY=25
plan=SUITE.SPECS+([SUITE.GENERAL] if GENERAL_MODE!='off' else [])
model_fingerprint=SUITE.checksum({
  'policy':COVER.POLICY_VERSION,
  'raw_threshold':COVER.RAW_DIRECT_DENSE_TRIGGER,
  'relative_coverage':COVER.COVERAGE_RELATIVE_GAIN,
  'extra_min':COVER.COVERAGE_EXTRA_MIN,
  'model':provenance['checkpoint_sha256'],
  'runtime_sha':sha256(WORK/'scope_coverage_runtime.py'),
  'v3_sha':sha256(WORK/'scope_final_runtime_v3.py'),
  'jit_sha':sha256(WORK/'scope_ctc_jit.py'),
  'beam_jit_sha':sha256(WORK/'scope_beam_jit_v4.py'),
  'beam_jit_status':BEAM_JIT_STATUS['status'],
  'suite_sha':sha256(WORK/'scope_safe_suite.py'),
  'jit_forward':JIT_FORWARD['available'],
  'jit_forced':JIT_FORCED['available']})
print('FINAL MULTIDATASET:',len(plan),'datasets; model lock:',model_fingerprint)


In [ ]:
import traceback
external_scored=[];external_status=[]
for spec in plan:
    ds=None
    try:
        key,ds,pred,out_dir=SUITE.run_dataset(
            spec,model=recognizer,out_root=EXTERNAL_OUT,
            data_home=EXTERNAL_DATA_HOME,model_fingerprint=model_fingerprint,
            general_mode=GENERAL_MODE,general_n=GENERAL_SAMPLE_N,
            seed=GENERAL_SAMPLE_SEED,max_per_dataset=MAX_PER_OTHER_DATASET,
            flush_every=EXTERNAL_WRITE_EVERY,
            infer_fn=COVER.predict_image)
        scored,valid=SUITE.score_dataset(ds,pred,out_dir,LT.normalize_ltb)
        external_scored.append((spec,valid,len(pred)))
        failure=int(pred['status'].ne('OK').sum())
        excluded=int(len(pred)-len(valid))
        status={'family':spec[0],'dataset':spec[1],
          'planned':len(pred),'scored':len(valid),
          'inference_failures':failure,'excluded_total':excluded,
          'sample_kind':json.loads((out_dir/'run_spec.json').read_text())['sample_kind'],
          'status':'PASS' if failure==0 and excluded==0 else 'PARTIAL_WITH_EXCLUSIONS' if failure==0 else 'FAILED_INFERENCE'}
        external_status.append(status)
        print('DATASET',spec[1],len(valid),'original',round(float(valid['original_svtrv2_full12_ok'].mean()),4),
          'SCOPE',round(float(valid['safe_scope_ok'].mean()),4),
          'active',int(valid['safe_scope_active'].sum()),
          'probe',int(valid['coverage_probe_crops'].gt(0).sum()),flush=True)
    except (KeyboardInterrupt,SystemExit):raise
    except Exception as exc:
        external_status.append({'family':spec[0],'dataset':spec[1],
          'status':'FAILED','exception':str(exc)})
        print('DATASET FAILED',spec[1],traceback.format_exc()[-2500:],flush=True)
    finally:
        if ds is not None:ds.close()
    pd.DataFrame(external_status).to_csv(EXTERNAL_OUT/'completion_status.csv',index=False)
print('All datasets processed:',len(external_scored),'/',len(plan))


In [ ]:
QX,PX,LX,BLX=SUITE.summarise_scored(external_scored)
FX=SUITE.aggregate_families(external_scored)
for fname,frame in [('quality_by_dataset.csv',QX),
   ('paired_by_dataset_length.csv',PX),('latency_by_dataset.csv',LX),
   ('quality_by_dataset_length.csv',BLX),('macro_micro_by_family.csv',FX)]:
    frame.to_csv(EXTERNAL_OUT/fname,index=False)
status_df=pd.DataFrame(external_status)
passed_names={spec[1] for spec,_,_ in external_scored}
expected_names={spec[1] for spec in plan}
assert len(external_status)==len(plan)
print('COMPLETION:');print(status_df.to_string(index=False))
print('SVTRv2 original vs SCOPE on EVERY dataset:')
print(QX[QX.method.isin(['original_svtrv2_full12','safe_scope'])].to_string(index=False))
print('Paired wins/losses:');print(PX[PX.length_bucket=='ALL'].to_string(index=False))
print('Macro/micro:');print(FX.to_string(index=False))
print('LTB 3 buckets + A-Avg:');print(Q.to_string(index=False))
# A full dataset is not necessarily fully evaluable: exclude zero normalized labels separately.
coverage=[]
for spec,d,_ in external_scored:
    for dec,sub in d.groupby('coverage_decision'):
        coverage.append({'family':spec[0],'dataset':spec[1],
             'decision':str(dec),'count':len(sub),
             'mean_total_ms':float(sub['safe_scope_total_ms'].mean()),
             'mean_probe_ms':float(sub['coverage_probe_ms'].mean()),
             'correct_original':int(sub['original_svtrv2_full12_ok'].sum()),
             'correct_scope':int(sub['safe_scope_ok'].sum())})
COVERAGE_DF=pd.DataFrame(coverage)
COVERAGE_DF.to_csv(EXTERNAL_OUT/'coverage_by_dataset_decision.csv',index=False)
ltb_coverage=S.groupby('coverage_decision',dropna=False).agg(
    n=('id','size'),scope_accuracy=('safe_scope_ok','mean'),
    mean_total_ms=('safe_scope_total_ms','mean'),
    mean_probe_ms=('coverage_probe_ms','mean')).reset_index()
ltb_coverage.to_csv(OUT/'ltb_coverage_decisions.csv',index=False)
print('LTB coverage audit:');print(ltb_coverage.to_string(index=False))


In [ ]:
meta={'experiment':'SCOPE_V4_PREFIX_BEAM_JIT_PARITY_LOCKED',
      'source_checkpoint_sha256':provenance['checkpoint_sha256'],
      'source_config_sha256':provenance['config_sha256'],
      'openocr_commit':OPENOCR_REV,
      'policy_version':COVER.POLICY_VERSION,
      'raw_dense_threshold':COVER.RAW_DIRECT_DENSE_TRIGGER,
      'coverage_min_extra':COVER.COVERAGE_EXTRA_MIN,
      'coverage_relative_gain':COVER.COVERAGE_RELATIVE_GAIN,
      'jit_forward':JIT_FORWARD,'jit_alignment':JIT_FORCED,
      'beam_jit':BEAM_JIT_STATUS,'beam_jit_audit':BEAM_JIT.audit_status(),
      'beam_jit_microbench':BEAM_CPU_BENCH,
      'beam_jit_sha256':sha256(WORK/'scope_beam_jit_v4.py'),
      'model_lock':model_fingerprint,'ltb_lock':MODEL_LOCK,
      'labels_in_inference':False,'no_dataset_id_in_inference':True,
      'method':'safe_scope','baseline':'original_svtrv2_full12',
      'important_limitation':'Coverage policy and E7 were developed after examining LTB and external benchmarks: this repeat is NOT independent test evidence.',
      'no_per_sample_regression_guarantee':True,
      'historical_e7_activated_row_parity':bool(E7_PARITY_PASS),
      'historical_e7_activated_rows':int(len(active_rows)),
      'external_dataset_status':external_status,
      'all_datasets_scored':len(external_scored)==len(plan),
      'all_inference_error_free':all(x.get('inference_failures',1)==0 for x in external_status),
      'general_mode':GENERAL_MODE,'general_sample_n':GENERAL_SAMPLE_N,
      'general_not_full_benchmark':GENERAL_MODE=='sample',
      'full_ltb_n':len(S)}
(OUT/'protocol.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False))
(EXTERNAL_OUT/'MULTIDATASET_PROTOCOL.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False))
FINAL_ZIP=MULTIDATA_PERSIST_ROOT/'SCOPE_V4_SPEED_OPTIMIZED_ALL_DATASETS_RESULTS.zip'
FINAL_ZIP=SUITE.pack_results(FINAL_ZIP,EXTERNAL_OUT,OUT,meta)
print('V4 BEAM PARITY AUDIT:',BEAM_JIT.audit_status())
if BEAM_JIT.audit_status()['audit_remaining']>0:
    print('CAUTION: fewer than 128 beam calls, limited runtime parity coverage')
print('FINAL RESULT ZIP:',FINAL_ZIP,'MB:',round(FINAL_ZIP.stat().st_size/1024**2,2))
if not meta['all_datasets_scored']:
    print('WARNING: SOME DATASETS FAILED; ZIP IS PARTIAL, SEE completion_status.csv')
elif not meta['all_inference_error_free']:
    print('WARNING: SOME IMAGES FAILED INFERENCE; see completion_status.csv')
if GENERAL_MODE=='sample':print('GENERAL: 2,000 deterministic sample, NOT full 400k')
try:
  from google.colab import files
  files.download(str(FINAL_ZIP))
except ImportError: print('Download:',FINAL_ZIP)


## Interpretation and boundaries

- Protected rows return the actual original SVTRv2-T transcript, **with no dense/orientation/cross-scale inference**. This is a structural preservation guarantee only for those inputs.
- Activated rows run original E7 FAST without changing the decoder. They can still make recognition worse on individual images.
- The policy is dataset-agnostic at inference, but **chosen on previously inspected test benchmarks**. Do not claim an independent validation from this rerun.
- Family macro scores have the usual sampling/normalization limitations; General uses a 2,000-image fixed sample, not its complete 400k benchmark.
- The remaining ≥56 and Dense low-evidence failures are open research problems; a universal non-regression guarantee requires a calibrated/validated guard beyond these geometric conditions.


## What constitutes success — and what is still *not established*

- **No test-time GT:** `scope_coverage_runtime.predict_image(image, recognizer)` takes neither labels nor dataset IDs. Its coverage signal is a prediction from crop views. No correctness oracle, and no guaranteed no-regression theorem.
- **Coverage/quality:** Report 26–35 / 36–55 / ≥56 on **all 4,789 LTB images**, plus accuracy, A-Avg, CER and paired wins/losses. Reporting only exact-line accuracy can hide catastrophic missing text.
- **Normal texts:** Report all Common, Union14M and OST subsets separately and macro/micro summaries. A 2,000 sample of General cannot stand in for full 400,000.
- **Runtime:** `safe_scope_total_ms` is measured synchronously and **includes** probing even on fallback. `scope_decoder_ms`, `coverage_probe_ms`, `paired_scale_forward_ms` are measured stages; check GPU model, batch sizes and P95. Numba speed tests are CPU microbenchmarks, not overall GPU speedups.
- **Reproduction:** Historical E7 parity is checked for every activated LTB row; CTC JIT has exact recurrence tests; the final ZIP includes the complete frozen source. If any mismatch appears, the notebook intentionally fails rather than declaring parity.
- **Novelty:** The research method is posterior-verified multi-view segment reconstruction. Early exit, crop-coverage gating and Numba are systems choices, not novel OCR representation learning.
- **Holdout required:** All these benchmarks were used for E0–E7 and SAFE v2 development. Before submission to a research venue, validate with a genuinely untouched long-text dataset and perform proper literature-matched baselines. No claim of arbitrary-long recovery, script-agnostic OCR or strict Pareto dominance is justified.


## V4 review protocol

1. Inspect all three LTB length buckets (26–35, 36–55, ≥56), exact accuracy, A-Avg, CER, S/D/I and paired changes against original SVTRv2-T.
2. Inspect per-dataset common/Union14M/OST micro/macro accuracy, regression counts and completed/excluded image counts.
3. Inspect actual `safe_scope_total_ms` and `scope_decoder_ms` distributions (mean, median, P95), **not just JIT microbenchmarks**. Compare against prior runs only as exploratory unless GPU, batch, measurement method and model runtime are matched.
4. Verify output parity on all 4,696 historically active LTB rows (or the actual activated count) and compare all predicted transcripts against the previous v3 archive before making quality-invariance claims.
5. Do not interpret the guard as pixel-space visual coverage; it is a transcript-length-disagreement probe. Do not claim a held-out generalization result on these reused development benchmarks.